# 06b Run tables for the rate-design batches (ratedesign, ratedesign2, batch2, batch3)

**Purpose.** Four ReEDS batches test credit designs and market conditions around the paper's
required-support result: `ratedesign` (66 runs), `ratedesign2` (93), `batch2` (89) and `batch3`
(151 registered, 149 returned). This notebook turns their extracted outputs into the derived run
tables that later notebooks read: the shadow price of the capacity mandate by year, the required
credit rate of the map anchors, the delivery outcome of every credit run against its target
trajectory, the ITC outlay, and the control-run outcomes that decide which cost samples are
informative for the flat-credit figure (SF15). A compact set of run sanity checks (solve status,
key sets, binding years, dropped load, credit landing, input fingerprints, market-condition echoes)
runs beside the tables, one registry per batch. The notebook replaces the four internal check
notebooks `rate_design_checks`, `rate_design2_checks`, `rate_design2_b2_checks` and
`rate_design2_b3_checks`; each output folder keeps that notebook's name and file names so the
provenance survives.

**Inputs.**
- ReEDS output extracts `results/extracts/{ratedesign, ratedesign2, batch2, batch3, smr100, step4}/*.parquet`
  and `results/extracts/key_inventory.parquet` (notebook 05). No h5 file is opened.
- `results/exports/01/mc_perdraw.npz` (notebook 01): the drawn cost samples, for the parameter echoes.
- Registration records under `data/registration/`: `ratedesign/` (u80 batch spec, u81 credit paths,
  u82 anchor spec), `ratedesign2/` (m01 map-anchor spec, m02 credit-run spec, m04 offers),
  `batch2/` (m50 run spec, m51 offers, m41b map-plus offers), `batch3/` (b3 run spec, b3 predictions).
- The fork's casefiles `cases_nuclearlearning_{smr100, step4, ratedesign, ratedesign2, batch2, batch3}.csv`,
  `cases.csv`, and ReEDS inputs under `inputs/` (financials, plant characteristics, mandate
  trajectories, incentives files, `emission_constraints/co2_cap.csv`).
- Optional cross-check: `results/exports/06a/step3_checks/duals_by_year.csv` (notebook 06a), when present.

**Outputs** (all under `results/exports/06b/`):
- `rate_design_checks/`: `duals_by_year.csv` (read by 07, 11, 13a/13b), `delivery_outcomes.csv`,
  `trajectory_gaps.csv`, `overbuild_by_case.csv`, `t09_required_itc_port_controls.csv`,
  `t09_required_itc_hz.csv`, `gx1_horizon.csv`, `hz_identity.csv`, `rate_recovery.csv`,
  `incentives_echo.csv`, `fingerprint_errors.csv`, `load_invariance.csv`, `checks_summary.csv`.
- `rate_design2_checks/`: `duals_by_year.csv`, `t09_required_itc_port_anchors.csv` (read by 10),
  `t09_required_itc_port_controls.csv`, `delivery_outcomes.csv`, `trajectory_gaps.csv`,
  `overbuild_by_case.csv`, `pair_identity.csv`, `rate_recovery.csv`, `incentives_echo.csv`,
  `fingerprint_errors.csv`, `load_invariance.csv`, `checks_summary.csv`.
- `rate_design2_b2_checks/`: `delivery_outcomes.csv`, `trajectory_gaps.csv`, `fingerprint_errors.csv`,
  `itc_payments.csv`, `checks_summary.csv` (read by 12), `control_outcomes.csv` (new; read by 12 for the
  SF15 informative flags), `pair_identity.csv`, `rate_recovery.csv`, `incentives_echo.csv`,
  `load_invariance.csv`.
- `rate_design2_b3_checks/`: `duals_by_year.csv`, `checks_summary.csv` (read by 11),
  `delivery_outcomes.csv`, `trajectory_gaps.csv`, `fingerprint_errors.csv`, `itc_payments.csv`
  (read by 12), `overbuild_by_case.csv`, `case_matrix.csv`, `design_echo.csv`, `switch_echo.csv`,
  `co2cap_binding.csv`, `rate_recovery.csv`, `incentives_echo.csv`, `load_invariance.csv`.

**Paper items.** None directly. The tables feed Table 1 and the t-tables (07), Fig 3, Fig 4 and
Table 2 (10, through the 36 map anchors' required rates), Fig 5 and ST4 (11, through the market-condition
shadow prices), SF15 (12) and SF24 (13a/13b).

**Runtime.** About 2 minutes on the reference machine (4 threads; extracts mode).

In [1]:
import sys
from pathlib import Path
_p = Path.cwd().resolve()
while not (_p / "nlpaper").is_dir():           # find z-ethan/ from wherever the kernel starts
    _p = _p.parent
sys.path.insert(0, str(_p))
import nlpaper
from nlpaper import paths, plotstyle as ps, h5io
nlpaper.setup(threads=4)

nlpaper setup: python 3.12.3 | numpy 1.26.4 | pandas 2.2.2 | matplotlib 3.8.4 | threads 4
determinism: seeds are explicit numpy Generators in each cell; results do not depend on PYTHONHASHSEED.


{'python': '3.12.3',
 'numpy': '1.26.4',
 'pandas': '2.2.2',
 'threads': 4,
 'threadpoolctl': True,
 'env_set': {'OMP_NUM_THREADS': '4',
  'MKL_NUM_THREADS': '4',
  'OPENBLAS_NUM_THREADS': '4',
  'NUMEXPR_NUM_THREADS': '4',
  'NUMBA_NUM_THREADS': '4'},
 'matplotlib': '3.8.4',
 'seed_note': None}

## 1. How to read this notebook

**The four batches and their run families.** Every run is one ReEDS solve (the sequential, or
myopic, solve: one linear program per model year, 2010 to 2050, 16 model years). Two run families
recur in every batch.

- A **mandate anchor** (map anchor) solves one cost sample with the SMR capacity mandate ON,
  learning OFF and no nuclear investment tax credit (ITC). A *cost sample* is one draw of the Monte
  Carlo cost engine (notebook 01): its overnight capital cost (OCC) path, construction duration and
  learning parameters. The *target trajectory* is the national SMR capacity path of one of six
  deployment schedules (eia, aj, iaea, mck, cop28, eo). The mandate forces national SMR capacity to
  at least the trajectory in every year. Its shadow price is the support the program needs.
- A **credit run** solves one cost sample with the mandate OFF and a nuclear ITC path fed through a
  per-run incentives file. It measures whether that credit path *delivers* the trajectory: whether
  the model builds the SMR capacity without being forced to. The fbB convention keeps learning OFF
  (the anchor's own cost files); the fbC convention turns learning ON with the sample's drawn
  learning parameters, so the in-run cost falls as the model builds.

| Batch (casefile) | Runs | Families |
|---|---|---|
| `ratedesign` (`rate_design_` files) | 66 | 48 credit runs (fb_ envelope, bd_ boundary-depth, hy_ two-tier hybrid; fbC); 12 mandate anchors at the 25th and 75th cost percentiles (`smr100_{schedule}_p25/p75`); 6 horizon reruns of the p50 anchors to 2055 (`hz_`) |
| `ratedesign2` | 93 | 36 map anchors in fresh cost samples (`ma_`); 57 credit runs (`ax_`) in the 30 solved anchor samples: exact-rate controls, capped-after-2035 pairs, flat 60% runs (fbB and fbC) |
| `batch2` | 89 | 48 statute runs (`bx_*_f50`: a flat 50% credit, the stacked statutory ceiling; `bx_*_g50`: the exact rate through 2038, then 50%); 21 learning-on exact-rate controls (`bx_*_u_lrn`); 20 map-margin runs (`bm_`); all fbC |
| `batch3` | 151 (149 returned) | 102 paired mandate anchors (the p05 to p95 anchors copied into one market condition each: low or high gas price, high demand, low or high renewable cost, limited transmission, a 95%-by-2050 CO2 cap); 48 flat-50% credit runs in three market conditions; 1 re-run of a batch-2 run |

Batch 3 has two registered runs with no output: `smr100_eo_p75_demhi` and `smr100_iaea_p75_demhi`
(the high-demand condition, 75th percentile). They are carried as "not returned" and every batch-3
count is over the 149 returned runs.

**The dual convention.** ReEDS reports the mandate constraint's shadow price twice:
`nuclear_cap_price` in 2004 dollars per MW-year of SMR capacity (the rental basis: the annual
payment that makes one more MW of SMR capacity break even) and `nuclear_cap_price_raw`, the raw
marginal in objective units. The two satisfy `raw = cost_scale x pvf_onm x converted`, where
`pvf_onm` is the operating-cost present-value factor of that solve year (14.7645 from 2026 in the
sequential solve) and `cost_scale` is 1. A run whose mandate never binds has no
`nuclear_cap_price` group: its shadow price is zero in every year. The duals table has one row per
case and model year: `mandate_MW` (the trajectory), `cap_MW` (national SMR capacity, all vintages),
`slack_MW = cap_MW - mandate_MW`, `dual_2004_MWyr`, `dual_raw` and `dual_2024_MWyr`
(`dual_2004_MWyr` / deflator(2024)). A year *binds* when `dual_2004_MWyr > 1`; a mandated year is
*clearly slack* when `slack_MW > 100`.

**Dollars.** Every export stays in the run's native dollars (2004 $ for ReEDS outputs, 2022 $/kW
for plant-characteristics inputs, 2024 $/kW for the u81/m04/m51 offer tables). Later notebooks
convert for display with `ps.to2025(from_year)`.

**Delivery tiers and outcomes** (the registered classifier, methods v2.3). A credit run's national
SMR capacity is compared with the trajectory in the *gate years*. For `ratedesign` and `ratedesign2`
the gate years are the offered years plus 2050; for `batch2` and `batch3` the flat-credit and
grant forms gate every model year 2031 to 2050 with a positive mandate, and the other forms the
offered years plus 2050 (methods v2.5). Tiers: *exact* (every gap inside 5 MW), *reproduces* (every
gated shortfall inside max(5% of the trajectory, 500 MW)), *partial* (the 2050 shortfall inside 15%),
*no*. Outcomes: *full-delivery* (exact or reproduces), *delayed-start-recover* (a gate-year shortfall
with 2050 inside the gate), *partial*, *under-delivery*, *no-build-lock-in* (national SMR capacity at
or below 5 MW in every year). A run *delivers* when its tier is exact or reproduces. Delivery is the
measured object of these batches, so an outcome is never a failed check.

**Sanity registries.** Each batch section records its checks in `checks_summary.csv`
(phase, check, status, detail) with the statuses PASS, FAIL and INFO. The check wording follows the
internal registries so the two can be compared row by row. Phase X holds checks added here
(dropped load, curtailment). Notebooks 11 and 12 read the registry as their gate R0: no FAIL.

In [2]:
# ---- common helpers: extracts, casefiles, dollars, registries ----------------------------
import json
import numpy as np
import pandas as pd

pd.set_option("display.width", 200)

OUT = paths.exports("06b")
def outdir(name):
    p = OUT / name
    p.mkdir(parents=True, exist_ok=True)
    return p

REG_DIR = paths.DATA / "registration"
INPUTS = paths.reeds_inputs()
FIN_DIR = INPUTS / "financials"
PC_DIR = INPUTS / "plant_characteristics"
NL_DIR = INPUTS / "nuclear_learning"

SCHEDULES = ["eia", "aj", "iaea", "mck", "cop28", "eo"]      # ordered by 2050 ambition
STEM2TOK = {"eia": "eia_aeo_high", "aj": "abou_jaoude", "iaea": "iaea_high",
            "mck": "mckinsey", "cop28": "cop28", "eo": "eo2025"}
NUC = ["nuclear", "nuclear-smr"]
PEN = 0.1                     # itc_tax_equity_penalty of every fed incentives file (monetized = 0.9 x rate)
MODEL_YEARS = [2031, 2032, 2033, 2034, 2035, 2038, 2041, 2044, 2047, 2050]
REF4 = "smr100_eia_p50_gaslo"  # the Step 4 run whose key set is the vintage reference
REF_LOAD = "smr100_eia_p50"    # the Step 3 load reference

# ---- extracts (notebook 05): one frame per (batch, key), split by case on first use -------
KI = pd.read_parquet(paths.require(paths.EXTRACTS / "key_inventory.parquet", "notebook 05_extract_reeds_outputs"))
KI = KI[~KI["pilot"]].set_index(["batch", "case"])
_GROUPS = {}

def load(batch, case, key):
    """One report table of one run, from the parquet extract (columns as in the h5 group)."""
    k = (batch, key)
    if k not in _GROUPS:
        df = h5io.load(batch, key)
        _GROUPS[k] = {c: g.drop(columns="case").reset_index(drop=True) for c, g in df.groupby("case", sort=False)}
    g = _GROUPS[k]
    if case not in g:
        raise KeyError(f"{key} not in {batch}/{case}")
    return g[case]

def keyset(batch, case):
    return set(str(KI.loc[(batch, case), "keys"]).split(";"))

def returned(batch, case):
    return (batch, case) in KI.index and bool(KI.loc[(batch, case), "returned"])

def dual_series(batch, case):
    """(converted, raw) shadow price by year; empty when the run carries no dual group (never bound)."""
    if "nuclear_cap_price" in keyset(batch, case):
        return (load(batch, case, "nuclear_cap_price").set_index("t")["Value"],
                load(batch, case, "nuclear_cap_price_raw").set_index("t")["Value"])
    return pd.Series(dtype=float), pd.Series(dtype=float)

# ---- dollars: deflator.csv holds Deflator(t) relative to 2004 --------------------------------
DEFL = pd.read_csv(paths.deflator_path())
DEFL.columns = ["t", "Deflator"]
DEFL = DEFL.set_index("t")["Deflator"]
D2022 = float(DEFL.loc[2022])                       # 2022$ -> 2004$
TO2024 = 1.0 / float(DEFL.loc[2024])                # 2004$ -> 2024$
D2224 = float(DEFL.loc[2022] / DEFL.loc[2024])      # 2022$/kW -> 2024$/kW (u81, m04, m51 units)

# ---- casefiles ------------------------------------------------------------------------------
def casefile(name, **kw):
    return pd.read_csv(paths.casefile(name), index_col=0, **kw)

def make_sw(casefiles, fallback):
    """sw(case, name): one switch value; a blank cell takes that casefile's Default Value; a switch
    row the case's casefile lacks takes the fallback casefile's Default Value (the source convention)."""
    def sw(case, name):
        for cf in casefiles:
            if case in cf.columns:
                break
        else:
            raise KeyError(case)
        if name not in cf.index:
            v = fallback.loc[name, "Default Value"]
            return "" if pd.isna(v) else str(v).strip()
        v = cf.loc[name, case]
        if pd.isna(v) or str(v).strip() == "":
            v = cf.loc[name, "Default Value"]
        return "" if pd.isna(v) else str(v).strip()
    return sw

def meta_of(sw, case):
    ts = sw(case, "GSw_NuclearCapMandateTechScen")
    return dict(sched=case.split("_")[1],
                scen=sw(case, "GSw_NuclearCapMandateScen"),
                mode=int(float(sw(case, "GSw_NuclearCapMandate"))),
                learning=int(float(sw(case, "GSw_NuclearLearning"))),
                techscen=ts, mandated_tech={"smr": "nuclear-smr", "large": "nuclear"}[ts],
                suffix=sw(case, "incentives_suffix"),
                foreign=sw(case, "GSw_NuclearLearning_ForeignScen"),
                pc={"nuclear": sw(case, "plantchar_nuclear"), "nuclear-smr": sw(case, "plantchar_nuclear_smr")},
                fin=sw(case, "financials_tech_suffix"),
                constr=sw(case, "construction_times_suffix"),
                endyear=int(float(sw(case, "endyear"))),
                yearset=[int(y) for y in sw(case, "yearset").split("_")],
                tcx=sw(case, "GSw_TCPhaseout_NuclearExempt"))

def sched_of(c):
    return c.split("_")[1]

def trajectory(scen):
    tr = pd.read_csv(NL_DIR / f"nuclear_cap_trajectory_{scen}.csv")
    tr.columns = [c.lstrip("*") for c in tr.columns]
    return tr.set_index(tr.columns[0])["MW"]

def plantchar(name):
    df = pd.read_csv(PC_DIR / f"{name}.csv")
    df.columns = [c.lstrip("*").lower() for c in df.columns]
    return df.set_index("t")

# ---- check registries: one list per batch ---------------------------------------------------
def make_record(registry, label):
    def record(phase, name, status, detail=""):
        assert status in ("PASS", "FAIL", "INFO")
        registry.append(dict(phase=phase, check=name, status=status, detail=str(detail)))
        print(f"[{status}] {label} {phase} :: {name}" + (f" -- {detail}" if detail else ""))
    return record

def write_registry(registry, folder):
    s = pd.DataFrame(registry, columns=["phase", "check", "status", "detail"])
    s.to_csv(folder / "checks_summary.csv", index=False)
    print(folder.name, s["status"].value_counts().to_dict())
    return s

# ---- the drawn cost samples (notebook 01) ----------------------------------------------------
Z = np.load(paths.require(paths.exports("01") / "mc_perdraw.npz", "notebook 01_monte_carlo_cost_engine"))
WCOLS = list(Z["world_columns"])
NPZ_YEARS = [int(y) for y in Z["years"]]
SWITCH_OF = {"lr_large": "GSw_NuclearLearning_LR_large", "lr_smr": "GSw_NuclearLearning_LR_smr",
             "boak_large": "GSw_NuclearLearning_BOAK_large", "boak_smr": "GSw_NuclearLearning_BOAK_smr",
             "s": "GSw_NuclearLearning_Spillover", "n_vendors": "GSw_NuclearLearning_Vendors",
             "conv_full": "GSw_NuclearLearning_Convention", "ces_rho": "GSw_NuclearLearning_CES_rho",
             "dur_lambda": "GSw_NuclearLearning_Dur_Lambda",
             "x_ls": "GSw_NuclearLearning_CrossTech_x_ls", "x_sl": "GSw_NuclearLearning_CrossTech_x_sl"}

# the Step 3 load reference and the Step 4 key-vintage reference
REF_LOAD_RT = load("smr100", REF_LOAD, "load_rt").set_index(["r", "t"])["Value"]
REF4_KEYS = keyset("step4", REF4)
DUAL_KEYS = {"nuclear_cap_price", "nuclear_cap_price_raw", "nuclear_cap_price_ub", "nuclear_cap_price_ub_raw"}
NONNEG = ["cap", "cap_nat", "load_rt", "stor_in", "stor_out", "curt_ann", "hours"]
print(f"2022$->2004$ {D2022:.6f}; 2004$->2024$ {TO2024:.4f}; 2022$->2024$ {D2224:.4f}; "
      f"key inventory {len(KI)} runs; Step 4 reference keys {len(REF4_KEYS)}")

2022$->2004$ 0.645694; 2004$->2024$ 1.6590; 2022$->2024$ 1.0712; key inventory 587 runs; Step 4 reference keys 209


### 1.1 Shared machinery: rate recovery, solve-health scans, the delivery classifier

Three pieces of logic recur in all four batches, so they are defined once.

- **Rate recovery.** A credit lands inside ReEDS's capital-cost financing multiplier. The ratio of
  the multiplier with the credit to the one without it gives the monetized credit `m` back
  (`reeds/financials.py`): `fin_mult / fin_mult_noITC = (1 - tax (1 - m/2) PVdep - m) / (1 - tax PVdep)`,
  so `m = (1 - ratio)(1 - tax PVdep) / (1 - tax PVdep / 2)`, with `PVdep` the present value of the
  depreciation schedule. The construction-financing multiplier cancels in the ratio, so the identity
  holds in learning-on runs too. GAMS loads the multipliers at 3 decimals; the tolerance is |dm| <= 0.005.
- **Solve-health scans** read the key inventory of notebook 05 for the key sets, the non-finite count
  and the minimum of each non-negative table; the other tests read the extracts.
- **The delivery classifier** (tier and outcome, section 1) takes the gap table of one run and the
  gate mask of its batch.

In [3]:
# ---- rate recovery: tax rate + PV of depreciation (ccmult cancels in the ratio) ------------
YEARS_FIN = np.arange(2010, 2056)

def yi_fin(t):
    return int(t - YEARS_FIN[0])

_sf = pd.read_csv(FIN_DIR / "financials_sys_ATB2024.csv").merge(pd.read_csv(FIN_DIR / "inflation_default.csv"), on="t", how="left")
_sf["d_nom"] = ((1 - _sf["debt_fraction"]) * (_sf["rroe_nom"] - 1)
                + _sf["debt_fraction"] * (_sf["interest_rate_nom"] - 1) * (1 - _sf["tax_rate"]) + 1)

def _on_years_fin(col):
    s = _sf.set_index("t")[col].reindex(range(1990, YEARS_FIN[-1] + 1)).ffill()
    return s.loc[YEARS_FIN].to_numpy(float)

TAX_FIN = _on_years_fin("tax_rate")
D_NOM_FIN = _on_years_fin("d_nom")
DEP_ = pd.read_csv(FIN_DIR / "depreciation_schedules_default.csv")
_PVDEP = {}

def pv_dep_nuclear(fin_suffix):
    if fin_suffix in _PVDEP:
        return _PVDEP[fin_suffix]
    ft = pd.read_csv(FIN_DIR / f"financials_tech_{fin_suffix}.csv")
    ft.columns = [c.lstrip("*") for c in ft.columns]
    ft = ft[ft["i"].isin(["Nuclear", "Nuclear-SMR"])]
    out = {}
    for iname, grp in ft.groupby("i"):
        grp = grp.set_index("t").reindex(YEARS_FIN).ffill().bfill()
        dep_frac = DEP_[str(int(grp["depreciation_sch"].iloc[0]))].to_numpy(float)
        out[iname.lower()] = pd.Series([np.sum(dep_frac / dn ** np.arange(1, len(dep_frac) + 1)) for dn in D_NOM_FIN],
                                       index=YEARS_FIN)
    _PVDEP[fin_suffix] = out
    return out

def recovered_m(batch, case, fin_suffix):
    fm = load(batch, case, "cost_cap_fin_mult")
    fno = load(batch, case, "cost_cap_fin_mult_noITC")
    fm = fm[fm["i"].isin(NUC)].set_index(["i", "r", "t"])["Value"]
    fno = fno[fno["i"].isin(NUC)].set_index(["i", "r", "t"])["Value"]
    j = pd.concat([fm, fno], axis=1, keys=["fm", "fno"]).dropna().reset_index()
    pv = pv_dep_nuclear(fin_suffix)
    pd_ = np.array([pv[i].loc[t] for i, t in zip(j["i"], j["t"])])
    tau = TAX_FIN[[yi_fin(t) for t in j["t"]]]
    ratio = j["fm"].to_numpy() / j["fno"].to_numpy()
    j["m_rec"] = (1.0 - ratio) * (1.0 - tau * pd_) / (1.0 - tau * pd_ / 2.0)
    return j

def rate_recovery_rows(batch, case, fin_suffix, m_exp, extra):
    """One row per (tech, year): median recovered m over regions, the region spread, m expected."""
    j = recovered_m(batch, case, fin_suffix)
    g = j.groupby(["i", "t"])["m_rec"].agg(["median", "min", "max"])
    g["spread_r"] = g["max"] - g["min"]
    rows = []
    for (i, t), r in g.iterrows():
        me = m_exp.get(t, 0.0) if i == "nuclear-smr" else 0.0
        rows.append(dict(case=case, **extra, i=i, t=t,
                         m_expected=round(me, 4) if pd.notna(me) else np.nan,
                         m_recovered=round(float(r["median"]), 4),
                         spread_r=round(float(r["spread_r"]), 5),
                         delta=round(float(r["median"]) - me, 4) if pd.notna(me) else np.nan))
    return rows

TOL_M = 0.005

def rate_landing_checks(record, rr, offer, names):
    """B4, B5, B8, B9 on a rate-recovery table (only runs with an offer); names = the four check names."""
    on = rr[(rr["i"] == "nuclear-smr") & np.array([r.t in offer[r.case] for r in rr.itertuples()])]
    w = on.loc[on["delta"].abs().idxmax()]
    record("B", names[0],
           "PASS" if on["delta"].abs().max() <= TOL_M else "FAIL",
           f"{len(on)} fed case-years; worst |dm| {on['delta'].abs().max():.4f} at ({w.case}, {w.t})")
    off = rr[[not (r.i == "nuclear-smr" and r.t in offer[r.case]) for r in rr.itertuples()]]
    record("B", names[1],
           "PASS" if off["m_recovered"].abs().max() <= TOL_M else "FAIL",
           f"{len(off)} points; max |m| {off['m_recovered'].abs().max():.4f}")
    sp = rr[rr["i"] == "nuclear-smr"]
    record("B", names[2],
           "PASS" if sp["spread_r"].max() <= TOL_M else "FAIL", f"max spread over regions {sp['spread_r'].max():.5f}")
    late = rr[(rr["t"] >= 2036) & (rr["m_expected"] > 0)]
    record("B", names[3],
           "PASS" if late["delta"].abs().max() <= TOL_M else "FAIL",
           f"{len(late)} credited case-years at t >= 2036; worst |dm| {late['delta'].abs().max():.4f}")

def incentives_rows(r, suffix, rates):
    """The fed incentives file of one credit run against its registered offer path."""
    inc = pd.read_csv(FIN_DIR / f"incentives_{suffix}.csv")
    nuc_rows = inc[inc["i"].str.lower().str.startswith("nuclear")]
    only_smr = set(nuc_rows["i"]) == {"Nuclear-SMR"}
    got = {int(t): float(f) for t, f in zip(nuc_rows["t_start_construction"], nuc_rows["itc_frac"])}
    conv_ok = bool((nuc_rows["safe_harbor"] == 0).all()
                   and (nuc_rows["t_max_online"] == nuc_rows["t_start_construction"]).all()
                   and (nuc_rows["itc_tax_equity_penalty"] == PEN).all()
                   and (nuc_rows["itc_energy_comm_bonus"] == 0.0).all()
                   and (nuc_rows["itc_percpt_domestic_bonus"] == 0.0).all())
    return got, dict(only_smr=only_smr, conv_ok=conv_ok, rate_ok=got == rates, below1=max(got.values()) < 1.0)

# ---- solve-health scans (phase A and D) -------------------------------------------------------
def solve_health(record, batch, cases, years_of, key_labels=None, key_exclude=frozenset()):
    """A4/A5 key sets (when key_labels is given), A6 residuals, A7 objective, A8 model years, A9 sequential solve."""
    ks = {c: keyset(batch, c) for c in cases}
    union = set.union(*ks.values())
    if key_labels is not None:
        bad = {}
        for c in cases:
            diff = ((union - ks[c]) | (ks[c] - union)) - set(key_exclude)
            if diff:
                bad[c] = sorted(diff)
        drift = sorted(((union - REF4_KEYS) | (REF4_KEYS - union)) - DUAL_KEYS)
        record("A", key_labels[0], "PASS" if not bad else "FAIL",
               f"key counts {sorted(set(len(k) for k in ks.values()))}" + (f"; unexpected differences: {bad}" if bad else ""))
        record("A", key_labels[1], "PASS" if not drift else "FAIL",
               f"drift vs {REF4}: {drift}" if drift else f"{len(REF4_KEYS)} keys, no drift")
    worst_z, worst_at = 0.0, None
    for c in cases:
        m = float(load(batch, c, "error_check").set_index("*")["Value"].abs().max())
        if m > worst_z:
            worst_z, worst_at = m, c
    record("A", "the solver residuals are small in every file",
           "PASS" if worst_z < 1e-2 else "FAIL", f"worst |residual| {worst_z:.2e} at {worst_at}")
    obj = {c: float(load(batch, c, "objfn_raw")["Value"].iloc[0]) for c in cases}
    record("A", "the objective value is a normal positive number in every file",
           "PASS" if all(np.isfinite(v) and v > 0 for v in obj.values()) else "FAIL",
           f"range {min(obj.values()):.4e} to {max(obj.values()):.4e}")
    bad_years = {}
    for c in cases:
        y = sorted(load(batch, c, "cap")["t"].unique())
        exp_y, ok_meta = years_of(c)
        if y != exp_y or not ok_meta:
            bad_years[c] = y[-3:]
    record("A", years_label[batch], "PASS" if not bad_years else "FAIL",
           "every file on its registered year set" + (f"; wrong in {bad_years}" if bad_years else ""))
    worst = dict(pvfc=0.0, pvfo=0.0, cs=0.0, zrep=0)
    pvf_all, cs_all = {}, {}
    for c in cases:
        pvfc = load(batch, c, "pvf_capital").set_index("t")["Value"]
        pvfo = load(batch, c, "pvf_onm").set_index("t")["Value"]
        pvf_all[c] = pvfo
        pvfo = pvfo[pvfo.index >= 2026]
        cs = float(load(batch, c, "cost_scale")["Value"].iloc[0])
        cs_all[c] = cs
        worst["pvfc"] = max(worst["pvfc"], float((pvfc - 1.0).abs().max()))
        worst["pvfo"] = max(worst["pvfo"], float((pvfo - pvfo.mean()).abs().max()))
        worst["cs"] = max(worst["cs"], abs(cs - 1.0))
        worst["zrep"] = max(worst["zrep"], abs(len(load(batch, c, "z_rep")) - len(years_of(c)[0])))
    ok = worst["pvfc"] < 1e-9 and worst["pvfo"] < 1e-3 and worst["cs"] < 1e-9 and worst["zrep"] == 0
    record("A", "every run used the sequential solve mode", "PASS" if ok else "FAIL",
           f"pvf_capital==1 (max dev {worst['pvfc']:.1e}); pvf_onm flat from 2026 (max dev {worst['pvfo']:.1e}); "
           "cost_scale==1; z_rep has one row per model year")
    return ks, union, pvf_all, cs_all

years_label = {
    "ratedesign": "the model years are correct in every file (16-year standard set; 18-year extended set with 2053 and 2055 in the six horizon runs)",
    "ratedesign2": "the model years are the standard 16-year set with end year 2050 in every file",
    "batch2": "the model years are the standard 16-year set with end year 2050 in every file",
    "batch3": "the model years are the standard 16-year set with end year 2050 in every file",
}

def data_scans(record, batch, cases, n_label, n_keys):
    """D2-D5: non-finite values and non-negative tables (key inventory); national totals."""
    ki = KI.loc[[(batch, c) for c in cases]]
    nan_bad = [(c, ki.loc[(batch, c), "nonfinite_keys"]) for c in cases if ki.loc[(batch, c), "n_nonfinite"] > 0]
    neg_bad = [(c, k, float(ki.loc[(batch, c), f"min_{k}"])) for c in cases for k in NONNEG
               if pd.notna(ki.loc[(batch, c), f"min_{k}"]) and float(ki.loc[(batch, c), f"min_{k}"]) < -1.0]
    nuc_neg = []
    for c in cases:
        for k in ["cap_new_ann", "gen_ann"]:
            df = load(batch, c, k)
            nuc = df[df["i"].isin(NUC)]
            if len(nuc) and float(nuc["Value"].min()) < -1.0:
                nuc_neg.append((c, k, float(nuc["Value"].min())))
    record("D", "no data value is NaN or infinite in any file",
           "PASS" if not nan_bad else "FAIL", nan_bad[:5] or f"all {n_keys} keys x {n_label} files scanned (key inventory)")
    record("D", "no negative values occur where values must not be negative",
           "PASS" if not neg_bad else "FAIL",
           neg_bad[:5] or f"checked {NONNEG} (storage charging and upgrades are legitimately negative elsewhere)")
    record("D", "the nuclear build and generation data is not negative",
           "PASS" if not nuc_neg else "FAIL", nuc_neg[:5] or f"cap_new_ann and gen_ann nuclear rows >= 0, all {n_label} cases")
    tot_bad = []
    for c in cases:
        for key, nat_key in [("cap", "cap_nat"), ("gen_ann", "gen_ann_nat"), ("cap_new_ann", "cap_new_ann_nat"), ("ret_ann", "ret_ann_nat")]:
            s = load(batch, c, key).groupby(["i", "t"])["Value"].sum()
            nat_df = load(batch, c, nat_key)
            n = nat_df.groupby([col for col in nat_df.columns if col in ("i", "t")])["Value"].sum()
            j = pd.concat([s, n], axis=1, keys=["sum", "nat"]).fillna(0.0)
            rel = float(((j["sum"] - j["nat"]).abs() / j["nat"].abs().clip(lower=1.0)).max())
            if rel > 1e-3:
                tot_bad.append((c, key, rel))
    record("D", "the national totals equal the sum of the regional data",
           "PASS" if not tot_bad else "FAIL", tot_bad[:5] or f"cap, gen_ann, cap_new_ann, ret_ann; all {n_label} cases within 1e-3")

def added_scans(record, batch, cases, n_label):
    """Phase X (added in the public pipeline): dropped load after 2025; curtailment (context)."""
    dl_bad, hist = [], []
    for c in cases:
        dl = load(batch, c, "dropped_load")
        modern = float(dl.loc[dl["t"] >= 2026, "Value"].sum())
        hist.append(float(dl.loc[dl["t"] < 2026, "Value"].sum()))
        if modern >= 1.0:
            dl_bad.append((c, modern))
    record("X", "no load is dropped after 2025 (the step3_checks E3 rule)", "PASS" if not dl_bad else "FAIL",
           dl_bad[:5] or f"{n_label} runs; the historical 2010-2023 artifact {np.mean(hist) / 1e6:.2f} TWh "
           f"(case spread {max(hist) - min(hist):.1f} MWh)")
    cu = pd.Series({c: float(load(batch, c, "curt_ann").query("t == 2050")["Value"].sum()) for c in cases})
    record("X", "national curtailment in 2050 (context)", "INFO",
           f"{cu.min() / 1e6:.1f} to {cu.max() / 1e6:.1f} TWh (median {cu.median() / 1e6:.1f}) over {n_label} runs")

def load_invariance(batch, cases, extra=None):
    rows = []
    for c in cases:
        a = load(batch, c, "load_rt").set_index(["r", "t"])["Value"]
        j = pd.concat([a, REF_LOAD_RT], axis=1, keys=["c", "ref"], join="inner")
        rel = float(((j["c"] - j["ref"]).abs() / j["ref"].abs().clip(lower=1.0)).max())
        rows.append(dict(case=c, **(extra(c) if extra else {}), vs=REF_LOAD, n_shared=len(j), max_rel_diff=rel))
    return pd.DataFrame(rows)

def smr_cap(batch, case):
    return load(batch, case, "cap").query("i == 'nuclear-smr'").groupby("t")["Value"].sum()

def fingerprint_rows(batch, case, meta, extra):
    """Capital-cost fingerprint: the run's cost_cap against its plantchar input (2022$/kW x 1000 x D2022)."""
    cc = load(batch, case, "cost_cap")
    cc = cc[cc["i"].isin(NUC)].set_index(["i", "t"])["Value"]
    rows = []
    for itech in NUC:
        pc_df = plantchar(meta["pc"][itech])
        for t, occ_kw in pc_df["capcost"].items():
            if (itech, t) not in cc.index:
                continue
            expect = occ_kw * 1000.0 * D2022
            got = float(cc.loc[(itech, t)])
            rows.append(dict(case=case, **extra, learning=meta["learning"], i=itech, t=t,
                             expect=expect, got=got, rel_err=abs(got - expect) / expect))
    return rows

### 1.2 Shared helper: the mandate duals table (marked for factoring into `nlpaper`)

`duals_rows` builds the canonical duals schema of section 1 for one mandate run, and
`dual_checks` applies the three consistency tests of the internal registries: the capacity meets
the floor in every mandated year (no shortfall above 5 MW), the capacity equals the floor where
the dual is positive (dual > $1/MW-yr implies |slack| <= 5 MW), and a positive dual occurs only
in binding years (slack > 100 MW implies a zero dual). Notebook 06a builds the same table for
smr100 and step4; the lead can move this cell into `nlpaper` unchanged.

In [4]:
# ==== SHARED HELPER (06a/06b): the canonical duals table ==================================
def duals_rows(case, traj, cap_by_t, dual, raw, years, extra=None):
    rows = []
    for t in years:
        req = float(traj.get(t, 0.0))
        have = float(cap_by_t.get(t, 0.0))
        d = float(dual.get(t, 0.0))
        rows.append(dict(case=case, **(extra or {}), t=t, mandate_MW=req, cap_MW=round(have, 1),
                         slack_MW=round(have - req, 1), dual_2004_MWyr=d,
                         dual_raw=float(raw.get(t, 0.0)), dual_2024_MWyr=d * TO2024))
    return rows

def dual_checks(record, duals, label):
    m = duals[duals["mandate_MW"] > 0]
    viol = m[m["slack_MW"] < -5]
    bind_bad = m[(m["dual_2004_MWyr"] > 1.0) & (m["slack_MW"].abs() > 5)]
    slack_bad = m[(m["slack_MW"] > 100) & (m["dual_2004_MWyr"] > 1e-3)]
    record("C", f"the capacity satisfies the mandate floor in every mandate run ({label})",
           "PASS" if viol.empty else "FAIL",
           list(viol[["case", "t", "slack_MW"]].itertuples(index=False))[:5] or "no violation larger than 5 MW in any mandated year")
    record("C", "the capacity equals the floor in each binding year (positive dual)",
           "PASS" if bind_bad.empty else "FAIL",
           list(bind_bad[["case", "t"]].itertuples(index=False))[:5] or "every year with dual > $1/MW-yr has |slack| <= 5 MW")
    record("C", "a positive dual occurs only in the binding years",
           "PASS" if slack_bad.empty else "FAIL",
           list(slack_bad[["case", "t"]].itertuples(index=False))[:5] or "every year with slack > 100 MW has a zero dual")

def overbuild_rows(duals):
    """The step3_checks C8 summary per mandate run (as step3_analysis derives it for the p25/p75 anchors)."""
    rows = []
    for c, g in duals.groupby("case", sort=False):
        sub = g[(g["mandate_MW"] > 0) & (g["t"] <= 2050)]
        b = sub["dual_2004_MWyr"] > 1.0
        rows.append(dict(case=c, n_mandated_years=len(sub), n_binding_years=int(b.sum()),
                         n_slack_years=int((sub["slack_MW"] > 100).sum()),
                         first_binding_year=(int(sub[b]["t"].min()) if b.any() else None),
                         max_overbuild_ratio=round(float((sub["cap_MW"] / sub["mandate_MW"]).max()), 2)))
    return pd.DataFrame(rows)
# ==== END SHARED HELPER ====================================================================

### 1.3 Shared helper: the t09 required-rate chain (marked for factoring into `nlpaper`)

The required credit rate of a mandate anchor turns the shadow price into the Equivalent ITC: the
investment tax credit rate that pays the same support as the mandate. This is the `step3_analysis`
t09 code path (cells ba104441, e2510eca, bfdebdc1, e3e7fc5c, c78b98c2, 5375aaf6), which the
internal checks ported verbatim; the arithmetic below is unchanged. For each mandated year the
status ladder is: *slack* (no binding dual: the required ITC is zero), *no mandate increase*, *no
new builds* (the dual prices retention, not a build), or *rate*. In a rate year, with `S` the raw
dual (2004 $/MW, present value), `cc` the national overnight cost, `fm` the region's financing
multiplier without ITC and `K = (1 - tax PVdep) / (1 - tax PVdep / 2)`, the monetized credit is
`m_r = S / (cc fm) x K` in each build region and the statutory rate `i_r = m_r / (1 - 0.10)`
(the tax-equity haircut). The headline rate `i_model_headline` is the maximum over the build
regions; `i_pis` multiplies by the construction-financing multiplier (the placed-in-service
companion). A rate above 1 is flagged `itc_insufficient`: no credit up to 100% delivers.

The discount rate does not enter t09 (it enters only the rental-transfer table t08, which this
notebook does not port). Notebook 07 runs the same chain on the smr100 cases; the lead can move
this cell into `nlpaper`.

In [5]:
# ==== SHARED HELPER (07/06b): the t09 required-rate chain ==================================
YEARS = np.arange(2010, 2051)
_sys_fin = pd.read_csv(FIN_DIR / "financials_sys_ATB2024.csv").merge(pd.read_csv(FIN_DIR / "inflation_default.csv"), on="t", how="left")
_sys_fin["d_nom"] = ((1 - _sys_fin["debt_fraction"]) * (_sys_fin["rroe_nom"] - 1)
                     + _sys_fin["debt_fraction"] * (_sys_fin["interest_rate_nom"] - 1) * (1 - _sys_fin["tax_rate"]) + 1)
_sys_fin["d_real"] = _sys_fin["d_nom"] / _sys_fin["inflation_rate"]

def _on_years(col):
    s = _sys_fin.set_index("t")[col].reindex(range(1990, YEARS[-1] + 1)).ffill()
    return s.loc[YEARS].to_numpy(float)

D_REAL, D_NOM, TAX, IB = _on_years("d_real"), _on_years("d_nom"), _on_years("tax_rate"), _on_years("interest_rate_nom")

def yi(t):
    return int(t - YEARS[0])

DEP = pd.read_csv(FIN_DIR / "depreciation_schedules_default.csv")
CS = pd.read_csv(FIN_DIR / "construction_schedules_mc.csv")
CC_EXPS = np.append([0.0], np.arange(0.5, 10.5, 1.0))   # header 'NA' row -> 0, rows t=0..9 -> 0.5..9.5
assert len(CS) == len(CC_EXPS), (len(CS), len(CC_EXPS))

def ccmult_reeds(sched_name, ib):
    x = pd.to_numeric(CS[str(sched_name)], errors="coerce").fillna(0.0).to_numpy()
    return 1.0 + float(np.sum(x * (ib ** CC_EXPS - 1.0)))

IN_NAME = {"nuclear": "Nuclear", "nuclear-smr": "Nuclear-SMR"}
_NF_CACHE = {}

def natfin(fin_suffix, tech):
    """(natbase, k, ccm) on YEARS for one financials_tech file and technology (step3 cell bfdebdc1)."""
    key = (fin_suffix, tech)
    if key in _NF_CACHE:
        return _NF_CACHE[key]
    ft = pd.read_csv(FIN_DIR / f"financials_tech_{fin_suffix}.csv")
    ft.columns = [c.lstrip("*") for c in ft.columns]
    grp = ft[ft["i"] == IN_NAME[tech]].set_index("t").reindex(YEARS).ffill().bfill()
    dep_frac = pd.to_numeric(DEP[str(int(grp["depreciation_sch"].iloc[0]))], errors="coerce").fillna(0.0).to_numpy(float)
    pv_dep = np.array([np.sum(dep_frac / dn ** np.arange(1, len(dep_frac) + 1)) for dn in D_NOM])
    eval_p = float(grp["eval_period"].iloc[0])
    risk = 1.0 + grp["finance_diff_real"].to_numpy(float) * ((1 - (1 / D_REAL) ** eval_p) / (D_REAL - 1.0))
    sys_pvf = (1 - (1 / D_REAL) ** (30 - 1)) / (D_REAL - 1.0) + 1
    tech_pvf = (1 - (1 / D_REAL) ** (eval_p - 1)) / (D_REAL - 1.0) + 1
    ccm = np.array([ccmult_reeds(grp.loc[t, "construction_sch"], IB[yi(t)]) for t in YEARS])
    natbase = pd.Series(ccm / (1 - TAX) * (1 - TAX * pv_dep) * risk * (sys_pvf / tech_pvf), index=YEARS)
    k = pd.Series((1 - TAX * pv_dep) / (1 - TAX * pv_dep / 2), index=YEARS)
    _NF_CACHE[key] = (natbase, k, pd.Series(ccm, index=YEARS))
    return _NF_CACHE[key]

# the statutory conversion uses the OBBBA haircut of a hypothetical nuclear credit (step3 cell e3e7fc5c)
_inc_ob = pd.read_csv(FIN_DIR / "incentives_obbba.csv")
_nuc_ob = _inc_ob[_inc_ob["i"].str.upper().str.startswith("NUCLEAR")]
PEN_T09 = float(_nuc_ob["itc_tax_equity_penalty"].iloc[0])
assert (_nuc_ob["itc_tax_equity_penalty"] == PEN_T09).all() and PEN_T09 == 0.1, PEN_T09
_inc_no = pd.read_csv(FIN_DIR / "incentives_obbba_nonuclearitc.csv")
_nuc_no = _inc_no[_inc_no["i"].str.upper().str.startswith("NUCLEAR")]
assert float(_nuc_no[[c for c in _inc_no.columns if c.startswith("itc_")]].abs().to_numpy().sum()) == 0.0, \
    "the no-nuclear-ITC incentives file carries a nuclear credit"

def t09_inputs(batch_of, cases, meta):
    """Per-case inputs of the chain from the extracts (step3 cell e2510eca, the t09 part)."""
    COSTCAP, FINMULT, FMDIFF, CAPNEW, PVF = {}, {}, {}, {}, {}
    for c in cases:
        b, tech = batch_of[c], meta[c]["mandated_tech"]
        PVF[c] = load(b, c, "pvf_onm").set_index("t")["Value"]
        cc = load(b, c, "cost_cap")
        COSTCAP[c] = cc[cc["i"] == tech].groupby("t")["Value"].mean()
        fmn = load(b, c, "cost_cap_fin_mult_noITC")
        FINMULT[c] = fmn[fmn["i"] == tech].set_index(["r", "t"])["Value"]
        fmi = load(b, c, "cost_cap_fin_mult")
        FMDIFF[c] = float((fmi[fmi["i"] == tech].set_index(["r", "t"])["Value"] - FINMULT[c]).abs().max())
        cn = load(b, c, "cap_new_ann")
        CAPNEW[c] = cn[cn["i"] == tech].set_index(["r", "t"])["Value"]
    worst_fm = max(FMDIFF.values())
    assert worst_fm < 1e-9, f"fin_mult != fin_mult_noITC for a mandated technology: {worst_fm} (the dual must be the total support)"
    # natfin self-check: natbase x a time-constant regional factor reproduces the h5 multiplier
    for c in cases:
        nb, _, _ = natfin(meta[c]["fin"], meta[c]["mandated_tech"])
        piv = FINMULT[c].unstack("t")
        yrs = [t for t in piv.columns if t >= 2026 and t in nb.index]
        ratio = piv[yrs].div(nb.reindex(yrs), axis=1)
        regfac = ratio.mean(axis=1)
        resid = (piv[yrs] - pd.DataFrame(np.outer(regfac, nb.reindex(yrs)), index=piv.index, columns=yrs)).abs() / piv[yrs]
        assert float(resid.max().max()) < 5e-3, (c, float(resid.max().max()))
        assert 0.85 < float(regfac.min()) and float(regfac.max()) < 1.35, c
    return COSTCAP, FINMULT, CAPNEW, PVF

def t09_table(duals, years_run, cases, meta, traj, inputs):
    """The t09 rows of `cases` (step3 cells c78b98c2 slice + 5375aaf6), unchanged arithmetic."""
    COSTCAP, FINMULT, CAPNEW, PVF = inputs
    D = duals[duals["case"].isin(cases)].copy()
    D["mandated"] = D["mandate_MW"] > 0
    D["binding"] = D["dual_2004_MWyr"] > 1.0
    TPREV = {t: (years_run[i - 1] if i > 0 else None) for i, t in enumerate(years_run)}
    rate_rows = []
    for c in cases:
        nb, K, CCM = natfin(meta[c]["fin"], meta[c]["mandated_tech"])
        d = D[D["case"] == c].set_index("t").sort_index()
        cc, fm, bl, tr = COSTCAP[c], FINMULT[c], CAPNEW[c], traj[c]
        for t in years_run:
            if not bool(d.loc[t, "mandated"]):
                continue
            dual_pos = bool(d.loc[t, "binding"])
            m_inc = float(tr.get(t, 0.0)) - float(tr.get(TPREV[t], 0.0) if TPREV[t] else 0.0)
            blt = bl.xs(t, level="t") if t in bl.index.get_level_values("t") else pd.Series(dtype=float)
            blt = blt[blt > 0]
            if not dual_pos:
                status = "slack: required ITC is zero"
            elif m_inc <= 0:
                status = "no mandate increase: no build receives a credit"
            elif not len(blt):
                status = "no new builds: dual prices retention, not a build"
            else:
                status = "rate"
            row = dict(case=c, t=t, status=status, S_t_2004_per_MW=None, n_build_regions=0,
                       ccmult=None, i_model_min=None, i_model_max=None, i_model_headline=None,
                       i_pis_min=None, i_pis_max=None, i_pis_headline=None, S_x_2p2GW_2024B=None,
                       itc_insufficient=False, pis_insufficient=False)
            if status == "rate":
                S = float(d.loc[t, "dual_raw"])                                   # cost_scale = 1
                S_alt = float(d.loc[t, "dual_2004_MWyr"]) * float(PVF[c].loc[t])
                assert abs(S - S_alt) <= 1e-4 * max(1.0, abs(S)), (c, t, S, S_alt)
                fmb = fm.xs(t, level="t").reindex(blt.index).dropna()
                m_r = S / (float(cc.loc[t]) * fmb) * float(K.loc[t])            # monetized, per region
                i_r = m_r / (1.0 - PEN_T09)                                       # statutory (haircut 10%)
                ccm_t = float(CCM.loc[t])
                i_pis = i_r * ccm_t                                               # placed-in-service companion
                row.update(S_t_2004_per_MW=round(S), n_build_regions=len(i_r), ccmult=round(ccm_t, 4),
                           i_model_min=round(float(i_r.min()), 3), i_model_max=round(float(i_r.max()), 3),
                           i_model_headline=round(float(i_r.max()), 3),
                           i_pis_min=round(float(i_pis.min()), 3), i_pis_max=round(float(i_pis.max()), 3),
                           i_pis_headline=round(float(i_pis.max()), 3),
                           S_x_2p2GW_2024B=round(S * 2200.0 * TO2024 / 1e9, 1),
                           itc_insufficient=bool(i_r.max() > 1.0), pis_insufficient=bool(i_pis.max() > 1.0))
            rate_rows.append(row)
    return pd.DataFrame(rate_rows)
# ==== END SHARED HELPER ====================================================================

### 1.4 Shared helper: the delivery classifier

`tier_of` and `outcome_of` are the registered classifier (section 1). The two batch generations
differ only in which rows are gated: `ratedesign`/`ratedesign2` gate the offered years plus 2050
and test the "delayed start" clause on the offered years alone; `batch2`/`batch3` gate by the v2.5
rule and use the gated rows for both. The batch-3 verdict gate GB0 asserted that the batch-3
classifier was the batch-2 code; here one function serves both batches, so GB0 holds by construction.

In [6]:
def tier_of(sub, gated):
    """exact / reproduces / partial / no, from one run's gap rows and its gate mask."""
    shortfall = (-sub["gap_MW"]).clip(lower=0.0)
    if float(shortfall.max()) <= 5.0 and float(sub["gap_MW"].abs().max()) <= 5.0:
        return "exact"
    gate = sub[gated]
    g_short = (-gate["gap_MW"]).clip(lower=0.0)
    tol1 = np.maximum(0.05 * gate["mandate_MW"], 500.0)
    if bool((g_short <= tol1).all()):
        return "reproduces"
    end = gate[gate["t"] == 2050]
    if len(end) and float(-end["gap_MW"].iloc[0]) <= 0.15 * float(end["mandate_MW"].iloc[0]):
        return "partial"
    return "no"

def outcome_of(sub, tier, cap_max, miss_rows, gated):
    """Outcome + (misses, max gated shortfall). miss_rows: the rows tested for a gate-year miss."""
    end = sub[sub["t"] == 2050]
    end_short = float(-end["gap_MW"].iloc[0])
    end_req = float(end["mandate_MW"].iloc[0])
    gate = sub[miss_rows]
    rate_miss = bool(((-gate["gap_MW"]).clip(lower=0.0) > np.maximum(0.05 * gate["mandate_MW"], 500.0)).any())
    end_ok = end_short <= max(0.05 * end_req, 500.0)
    if cap_max <= 5.0:
        outcome = "no-build-lock-in"
    elif tier in ("exact", "reproduces"):
        outcome = "full-delivery"
    elif rate_miss and end_ok:
        outcome = "delayed-start-recover"
    elif end_short <= 0.15 * end_req:
        outcome = "partial"
    else:
        outcome = "under-delivery"
    g = sub[gated]
    gtol = np.maximum(0.05 * g["mandate_MW"], 500.0)
    gshort = (-g["gap_MW"]).clip(lower=0.0)
    misses = [int(t) for t, s, tl in zip(g["t"], gshort, gtol) if s > tl]
    return outcome, misses, float(gshort.max()), end_req

def cross_check_06a(record, duals_mine, cases, label):
    """Optional: the control rows against notebook 06a's step3_checks duals, when 06a has run."""
    p = paths.RESULTS / "exports" / "06a" / "step3_checks" / "duals_by_year.csv"
    DC = ["mandate_MW", "cap_MW", "slack_MW", "dual_2004_MWyr", "dual_raw", "dual_2024_MWyr"]
    if not p.exists():
        record("C", f"the duals table reproduces the step3_checks rows of the {label}", "INFO",
               "notebook 06a export not found; cross-check skipped (run 06a first to enable it)")
        return
    frz = pd.read_csv(p)
    frz = frz[frz["case"].isin(cases) & (frz["t"] <= 2050)].sort_values(["case", "t"]).reset_index(drop=True)
    mine = duals_mine[duals_mine["case"].isin(cases) & (duals_mine["t"] <= 2050)].sort_values(["case", "t"]).reset_index(drop=True)
    dmax = float((frz[DC] - mine[DC]).abs().max().max()) if len(frz) == len(mine) else np.inf
    record("C", f"the duals table reproduces the step3_checks rows of the {label}",
           "PASS" if dmax < 1e-6 else "FAIL", f"max |difference| {dmax:.1e} over {len(frz)} rows")

## 2. `ratedesign`: 48 declining-credit runs, 12 p25/p75 anchors, 6 horizon runs

**What the batch tests.** The 48 credit runs (fbC: learning ON, mandate OFF) each feed one
declining credit path (`u81_run_schedules.csv`) to one cost sample: the *envelope* runs (fb_) sit
on or inside the required-rate envelope, the *boundary-depth* probes (bd_) test how far below it a
path can go, and the *hybrid* runs (hy_) pay two tiers. The 12 anchors (`smr100_{schedule}_p25/p75`)
extend the smr100 mandate anchors from three cost percentiles (5th, 50th, 95th) to five; their
shadow prices are the rows of `duals_by_year.csv` that notebooks 07, 11 and 13a/13b read. The 6
horizon runs (`hz_{schedule}_p50_ext`) rerun the p50 anchors to 2055 with the mandate flat after
2050: they test whether the 2047-to-2050 rise of the required rate is a terminal-year artifact (the
GX1 diagnostic, `gx1_horizon.csv`).

**Tables.** `duals_by_year.csv` (the 18 mandate runs), `overbuild_by_case.csv` (their binding and
slack summary), `trajectory_gaps.csv` and `delivery_outcomes.csv` (the 48 credit runs; the gate
years are the build years with an offer in u81 plus 2050), `t09_required_itc_port_controls.csv`
(the t09 chain on the six p50 controls: it must equal notebook 07's t09 rows),
`t09_required_itc_hz.csv` and `gx1_horizon.csv` (the horizon runs), and the sanity outputs
(`rate_recovery.csv`, `incentives_echo.csv`, `fingerprint_errors.csv`, `load_invariance.csv`,
`hz_identity.csv`).

In [7]:
# ---- ratedesign: case matrix, registration, metadata ------------------------------------------
F_RD = outdir("rate_design_checks")
REG_RD = []
rec = make_record(REG_RD, "rd")
B_RD = "ratedesign"
cases_rd = casefile("ratedesign")
cases_s100 = casefile("smr100")
CASES_RD = [c for c in cases_rd.columns if c != "Default Value"]
assert len(CASES_RD) == 66, f"the ratedesign casefile has 66 runs, found {len(CASES_RD)}"
CTRL_RD = [f"smr100_{ab}_p50" for ab in SCHEDULES]          # the six Step 3 p50 anchors (smr100 batch)

def rd_block(c):
    if c.startswith(("fb_", "bd_", "hy_")):
        return "itc"
    if c.startswith("smr100_"):
        return "anchor"
    if c.startswith("hz_"):
        return "horizon"
    raise ValueError(c)

BLOCK_RD = {c: rd_block(c) for c in CASES_RD}
ITC_RD = [c for c in CASES_RD if BLOCK_RD[c] == "itc"]
ANCH_RD = [c for c in CASES_RD if BLOCK_RD[c] == "anchor"]
HZ_RD = [c for c in CASES_RD if BLOCK_RD[c] == "horizon"]
assert [len(ITC_RD), len(ANCH_RD), len(HZ_RD)] == [48, 12, 6]
MAND_RD = ANCH_RD + HZ_RD
sw_rd = make_sw([cases_rd, cases_s100], cases_rd)
META_RD = {c: meta_of(sw_rd, c) for c in CASES_RD + CTRL_RD}
BATCH_OF_RD = {**{c: B_RD for c in CASES_RD}, **{c: "smr100" for c in CTRL_RD}}

u80 = pd.read_csv(REG_DIR / "ratedesign" / "u80_batch_spec.csv").set_index("run")
u81 = pd.read_csv(REG_DIR / "ratedesign" / "u81_run_schedules.csv")
u82 = pd.read_csv(REG_DIR / "ratedesign" / "u82_anchor_spec.csv").set_index("run")
assert set(u80.index) == set(ITC_RD) | set(HZ_RD), "u80 rows != casefile ITC + horizon columns"
assert set(u82.index) == set(ANCH_RD) and set(u81["run"]) == set(ITC_RD)
OFFER_RD = {r: g.set_index("year")["rate_on_world"].to_dict() for r, g in u81.groupby("run")}
RATES_RD = {r: {t: round(v, 3) for t, v in OFFER_RD[r].items()} for r in ITC_RD}
M_EXP_RD = {r: {t: f * (1.0 - PEN) for t, f in RATES_RD[r].items()} for r in ITC_RD}
YEARS_STD = META_RD[CTRL_RD[0]]["yearset"]
YEARS_EXT = META_RD[HZ_RD[0]]["yearset"]
assert len(YEARS_STD) == 16 and len(YEARS_EXT) == 18 and YEARS_EXT[:16] == YEARS_STD and YEARS_EXT[16:] == [2053, 2055]
print(f"ratedesign: {len(ITC_RD)} credit runs + {len(ANCH_RD)} anchors + {len(HZ_RD)} horizon runs; controls {CTRL_RD}")

ratedesign: 48 credit runs + 12 anchors + 6 horizon runs; controls ['smr100_eia_p50', 'smr100_aj_p50', 'smr100_iaea_p50', 'smr100_mck_p50', 'smr100_cop28_p50', 'smr100_eo_p50']


### 2.1 Solve health and design echo (phases A and B)

Phase A tests that every run returned and solved cleanly: one output per case, the reference runs
present, one key set across the batch (the dual keys excluded: ReEDS drops an all-zero dual group),
no key drift against the Step 4 vintage, small solver residuals, a positive objective, the
registered model years, and the sequential solve. Phase B tests that each run encodes its design
cell (the switch values the casefile gives it), that the learning switches of the credit runs equal
the drawn cost sample, that the fed incentives files equal the u81 paths, that the credit landed
in the financing multiplier at exactly the fed rate (section 1.1), that the mandate runs carry no
nuclear ITC, and that each run used the capital-cost path of its cost sample.

In [8]:
# --- A1/A2: every case returned; the references are present -----------------------------------
missing = [c for c in CASES_RD if not returned(B_RD, c)]
rec("A", "one output file exists for each of the 66 cases", "PASS" if not missing else "FAIL",
    f"missing: {missing}" if missing else "66 runs in the key inventory")
ref_missing = [c for c in CTRL_RD if not returned("smr100", c)] + ([] if returned("step4", REF4) else [REF4])
rec("A", "the reference files are available (6 Step 3 p50 anchors, 1 key-vintage reference)",
    "PASS" if not ref_missing else "FAIL", f"missing: {ref_missing}" if ref_missing else "smr100 p50 and step4 reference runs present")

KS_RD, UNION_RD, PVF_RD, CS_RD = solve_health(
    rec, B_RD, CASES_RD,
    years_of=lambda c: (META_RD[c]["yearset"], (META_RD[c]["endyear"] == 2055) == (c in HZ_RD)),
    key_labels=("all 66 files have the same data keys (dual keys excluded)",
                "the key set agrees with the Step 4 vintage (separate-delivery drift detector)"),
    key_exclude=DUAL_KEYS)

[PASS] rd A :: one output file exists for each of the 66 cases -- 66 runs in the key inventory
[PASS] rd A :: the reference files are available (6 Step 3 p50 anchors, 1 key-vintage reference) -- smr100 p50 and step4 reference runs present
[PASS] rd A :: all 66 files have the same data keys (dual keys excluded) -- key counts [207, 209]
[PASS] rd A :: the key set agrees with the Step 4 vintage (separate-delivery drift detector) -- 209 keys, no drift
[PASS] rd A :: the solver residuals are small in every file -- worst |residual| 7.76e-04 at hy_mck_cb50
[PASS] rd A :: the objective value is a normal positive number in every file -- range 1.3463e+12 to 1.7448e+12


[PASS] rd A :: the model years are correct in every file (16-year standard set; 18-year extended set with 2053 and 2055 in the six horizon runs) -- every file on its registered year set
[PASS] rd A :: every run used the sequential solve mode -- pvf_capital==1 (max dev 0.0e+00); pvf_onm flat from 2026 (max dev 1.9e-06); cost_scale==1; z_rep has one row per model year


In [9]:
# --- B1: design-matrix echo per block ----------------------------------------------------------
mx_bad = []
for c in CASES_RD:
    m_, ab = META_RD[c], sched_of(c)
    tok = STEM2TOK[ab]
    if BLOCK_RD[c] == "itc":
        w = int(u80.loc[c, "draw_index"])
        ok = (m_["mode"] == 0 and m_["learning"] == 1 and m_["suffix"] == f"obbba_rd_{c}"
              and m_["foreign"] == f"rd_{ab}_d{w}" and m_["pc"]["nuclear-smr"] == f"nuclear-smr_mc_rd_{ab}_d{w}"
              and m_["pc"]["nuclear"] == f"nuclear_mc_rd_{ab}_d{w}" and m_["fin"] == f"mc_rd_{ab}_d{w}"
              and m_["scen"] == f"{tok}_smr" and m_["endyear"] == 2050)
    elif BLOCK_RD[c] == "anchor":
        p = c.split("_")[2]
        ok = (m_["mode"] == 1 and m_["learning"] == 0 and m_["suffix"] == "obbba_nonuclearitc"
              and m_["pc"]["nuclear-smr"] == f"nuclear-smr_mc_smr100_{ab}_{p}" and m_["fin"] == f"mc_smr100_{ab}_{p}"
              and m_["scen"] == f"{tok}_smr" and m_["endyear"] == 2050 and m_["tcx"] == "0" and m_["techscen"] == "smr")
    else:
        ok = (m_["mode"] == 1 and m_["learning"] == 0 and m_["suffix"] == "obbba_nonuclearitc"
              and m_["pc"]["nuclear-smr"] == f"nuclear-smr_mc_smr100_{ab}_p50" and m_["fin"] == f"mc_smr100_{ab}_p50"
              and m_["scen"] == f"{tok}_smr_ext" and m_["endyear"] == 2055 and m_["yearset"] == YEARS_EXT and m_["tcx"] == "0")
    if not ok:
        mx_bad.append(c)
rec("B", "the case matrix encodes the intended block design (48 ITC-arm fbC-convention runs, 12 mandate anchors, "
    "6 extended-horizon reruns)", "PASS" if not mx_bad else "FAIL", mx_bad or "66 cases match the block rules and the naming conventions")

# --- B1b: credit-run learning switches == the drawn cost sample -----------------------------------
b1b_bad = []
for c in ITC_RD:
    ab, w = sched_of(c), int(u80.loc[c, "draw_index"])
    world = Z[f"worlds_{STEM2TOK[ab]}"][w]
    for col, name in SWITCH_OF.items():
        exp, got = float(world[WCOLS.index(col)]), float(sw_rd(c, name))
        if abs(got - exp) > max(1e-5 * abs(exp), 5e-6):
            b1b_bad.append((c, col, got, exp))
rec("B", "the ITC-arm learning switches equal the drawn world parameters at the u80 draw index",
    "PASS" if not b1b_bad else "FAIL", b1b_bad[:5] or f"48 runs x {len(SWITCH_OF)} switches match mc_perdraw.npz to 1e-5")

# --- B2: fed incentives files vs the u81 credit paths --------------------------------------------
echo_rows, b2_bad = [], []
for r in ITC_RD:
    got, flags = incentives_rows(r, f"obbba_rd_{r}", RATES_RD[r])
    if not all(flags.values()):
        b2_bad.append((r, flags))
    for t in sorted(set(got) | set(RATES_RD[r])):
        echo_rows.append(dict(run=r, block=BLOCK_RD[r], sched=sched_of(r), t=t,
                              itc_frac_file=got.get(t), itc_frac_u81=RATES_RD[r].get(t)))
pd.DataFrame(echo_rows).to_csv(F_RD / "incentives_echo.csv", index=False)
rec("B", "the 48 fed incentives files equal the u81 credit paths at exactly the build years (rate_on_world at 3 decimals)",
    "PASS" if not b2_bad else "FAIL",
    b2_bad[:3] or "48 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0; every rate below 1.0")

# --- rate recovery + B4/B5/B8/B9 -------------------------------------------------------------------
rr_rd = pd.DataFrame([row for c in ITC_RD for row in rate_recovery_rows(
    B_RD, c, META_RD[c]["fin"], M_EXP_RD[c], dict(block=BLOCK_RD[c], sched=sched_of(c)))])
rr_rd.to_csv(F_RD / "rate_recovery.csv", index=False)
rate_landing_checks(rec, rr_rd, OFFER_RD, (
    "the recovered monetized rate equals 0.9 x the fed rate at exactly the build years (48 learning-on runs; the learned "
    "construction multiplier cancels in the ratio)",
    "the recovered rate is zero outside the build years and for the large reactor in all 48 runs",
    "the recovered rate is uniform across regions",
    "the phaseout does not reduce the nuclear credit in any year (exemption held)"))

# --- B3: the 18 mandate runs carry no nuclear ITC --------------------------------------------------
def fmdiff(batch, c):
    fm = load(batch, c, "cost_cap_fin_mult"); fno = load(batch, c, "cost_cap_fin_mult_noITC")
    fm = fm[fm["i"].isin(NUC)].set_index([k for k in fm.columns if k != "Value"])["Value"]
    fno = fno[fno["i"].isin(NUC)].set_index([k for k in fno.columns if k != "Value"])["Value"]
    return float((fm - fno).abs().max())
FMD = {c: fmdiff(B_RD, c) for c in MAND_RD}
rec("B", "the 12 anchors and 6 horizon runs carry no nuclear ITC (fin_mult == fin_mult_noITC for both nuclear technologies)",
    "PASS" if max(FMD.values()) < 1e-9 else "FAIL", f"max |fin_mult - fin_mult_noITC| {max(FMD.values()):.1e} over 18 runs")

# --- B11/B12: ITC payment flows ------------------------------------------------------------------
SMR_NEW = {c: float(load(B_RD, c, "cap_new_ann").query("i == 'nuclear-smr' and t > 2030")["Value"].sum()) for c in CASES_RD}
pay_bad, no_build = [], []
for c in ITC_RD:
    sc = load(B_RD, c, "systemcost_techba")
    v = float(sc[sc["sys_costs"].str.contains("itc", case=False) & sc["i"].isin(NUC)]["Value"].abs().sum())
    if SMR_NEW[c] > 100.0 and v < 1e6:
        pay_bad.append((c, f"builds {SMR_NEW[c]:,.0f} MW but no nuclear ITC payments"))
    if SMR_NEW[c] <= 100.0:
        no_build.append(c)
rec("B", "the system cost data shows nuclear ITC payments wherever SMR builds occurred (ITC-arm runs)",
    "PASS" if not pay_bad else "FAIL",
    pay_bad or f"payments present in every case with builds; cases with ~no builds (lock-in candidates, read in phase C): {no_build or 'none'}")
itc_mag = {c: abs(float(load(B_RD, c, "tax_expenditure_itc")["Value"].sum())) for c in CASES_RD}
rec("B", "the other technologies keep their ITC in all 66 runs", "PASS" if min(itc_mag.values()) > 1e8 else "FAIL",
    f"total ITC tax expenditure magnitude {min(itc_mag.values())/1e9:.1f} to {max(itc_mag.values())/1e9:.1f} B$ per case")

[PASS] rd B :: the case matrix encodes the intended block design (48 ITC-arm fbC-convention runs, 12 mandate anchors, 6 extended-horizon reruns) -- 66 cases match the block rules and the naming conventions


[PASS] rd B :: the ITC-arm learning switches equal the drawn world parameters at the u80 draw index -- 48 runs x 11 switches match mc_perdraw.npz to 1e-5


[PASS] rd B :: the 48 fed incentives files equal the u81 credit paths at exactly the build years (rate_on_world at 3 decimals) -- 48 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0; every rate below 1.0


[PASS] rd B :: the recovered monetized rate equals 0.9 x the fed rate at exactly the build years (48 learning-on runs; the learned construction multiplier cancels in the ratio) -- 450 fed case-years; worst |dm| 0.0001 at (fb_aj_m100_cw, 2031)
[PASS] rd B :: the recovered rate is zero outside the build years and for the large reactor in all 48 runs -- 702 points; max |m| 0.0000
[PASS] rd B :: the recovered rate is uniform across regions -- max spread over regions 0.00105
[PASS] rd B :: the phaseout does not reduce the nuclear credit in any year (exemption held) -- 240 credited case-years at t >= 2036; worst |dm| 0.0001
[PASS] rd B :: the 12 anchors and 6 horizon runs carry no nuclear ITC (fin_mult == fin_mult_noITC for both nuclear technologies) -- max |fin_mult - fin_mult_noITC| 0.0e+00 over 18 runs


[PASS] rd B :: the system cost data shows nuclear ITC payments wherever SMR builds occurred (ITC-arm runs) -- payments present in every case with builds; cases with ~no builds (lock-in candidates, read in phase C): none
[PASS] rd B :: the other technologies keep their ITC in all 66 runs -- total ITC tax expenditure magnitude 123.6 to 1140.3 B$ per case


In [10]:
# --- B13: capital-cost fingerprints --------------------------------------------------------------
fp_rd = pd.DataFrame([row for c in CASES_RD for row in fingerprint_rows(B_RD, c, META_RD[c], dict(block=BLOCK_RD[c]))])
fp_rd.to_csv(F_RD / "fingerprint_errors.csv", index=False)
off_fp = fp_rd[fp_rd["learning"] == 0]
w_ = off_fp.loc[off_fp.rel_err.idxmax()]
rec("B", "the 18 learning-off runs use their plantchar cost path without change (anchors: own p25/p75 files; horizon: the "
    "p50 files, through 2055)", "PASS" if off_fp.rel_err.max() < 1e-4 else "FAIL",
    f"{len(off_fp)} (case, tech, year) points; max rel err {off_fp.rel_err.max():.2e} at ({w_.case}, {w_.i}, {w_.t})")
pre_fp = fp_rd[(fp_rd["learning"] == 1) & (fp_rd["t"] <= 2030)]
rec("B", "the 48 learning-on runs use the drawn cost path before the 2031 anchor",
    "PASS" if pre_fp.rel_err.max() < 1e-4 else "FAIL", f"{len(pre_fp)} points; max rel err {pre_fp.rel_err.max():.2e}")
dev = fp_rd[(fp_rd["learning"] == 1) & (fp_rd["t"] >= 2031) & (fp_rd["i"] == "nuclear-smr")].groupby("case")["rel_err"].max().round(3)
rec("B", "the learning-on runs write an in-run SMR cost path (max deviation from the drawn path, 2031-2050; context)", "INFO",
    f"median over runs {dev.median():.3f}; range {dev.min():.3f}-{dev.max():.3f}")

# B13c: the shipped plantchar files reproduce the offline cost samples the batch was placed on
map_bad, pre_dev = [], []
for r in ITC_RD:
    pc_df = plantchar(META_RD[r]["pc"]["nuclear-smr"])
    for t, occ_u81 in u81[u81["run"] == r].set_index("year")["occ_world_kW"].items():
        got = float(pc_df.loc[t, "capcost"]) * D2224
        if abs(got - occ_u81) / occ_u81 > 1e-3:
            map_bad.append((r, t, round(got, 1), occ_u81))
for c in ANCH_RD:
    ab, w = sched_of(c), int(u82.loc[c, "draw_index"])
    occ_npz = Z[f"occ_{STEM2TOK[ab]}_smr"][w]
    pc_df = plantchar(META_RD[c]["pc"]["nuclear-smr"])
    for j, t in enumerate(NPZ_YEARS):
        if t not in pc_df.index:
            continue
        rel = abs(float(pc_df.loc[t, "capcost"]) - occ_npz[j]) / occ_npz[j]
        if t >= 2031 and rel > 1e-3:
            map_bad.append((c, t))
        if t < 2031:
            pre_dev.append(rel)
rec("B", "the shipped SMR plantchar files reproduce the offline cost worlds on the drawn 2031-2050 map (ITC-arm: u81 "
    "occ_world_kW in 2024$; anchors: the mc_perdraw.npz row of the u82 draw in native 2022$)",
    "PASS" if not map_bad else "FAIL", map_bad[:5] or "48 ITC-arm build-year points and 12 anchors x 20 years within 1e-3")
rec("B", "before the 2031 anchor the plantchar files carry the pre-anchor path while the npz holds the flat BOAK value "
    "(outside the drawn map; context)", "INFO", f"12 anchors x 7 pre-2031 years; max rel difference {max(pre_dev):.2f}")

# --- B15: dual keys only where a mandate exists; B16: the extended mandate files ------------------
b15_bad = sorted(c for c in CASES_RD if bool(KS_RD[c] & DUAL_KEYS) != (c in MAND_RD))
rec("B", "a dual key exists in exactly the 18 mandate files and in none of the 48 ITC-arm files",
    "PASS" if not b15_bad else "FAIL", b15_bad or "nuclear_cap_price present in 12 anchors + 6 horizon runs, absent elsewhere")
TRAJ_RD = {c: trajectory(META_RD[c]["scen"]) for c in CASES_RD + CTRL_RD}
b16_bad = []
for c in HZ_RD:
    ext, std = TRAJ_RD[c], TRAJ_RD[f"smr100_{sched_of(c)}_p50"]
    if not (all(abs(float(ext.get(t, -1)) - float(std.loc[2050])) < 1e-6 for t in [2051, 2053, 2055])
            and all(abs(float(ext.get(t, -1)) - float(std.get(t, -2))) < 1e-6 for t in YEARS_STD if std.get(t, 0) > 0)):
        b16_bad.append(c)
rec("B", "the horizon runs' extended mandate files equal the standard file through 2050 and stay flat at the 2050 level "
    "through 2055", "PASS" if not b16_bad else "FAIL", b16_bad or "6 files; 2051-2055 == 2050 MW; 2031-2050 == the standard trajectory")

[PASS] rd B :: the 18 learning-off runs use their plantchar cost path without change (anchors: own p25/p75 files; horizon: the p50 files, through 2055) -- 1476 (case, tech, year) points; max rel err 2.64e-07 at (smr100_cop28_p25, nuclear-smr, 2046)
[PASS] rd B :: the 48 learning-on runs use the drawn cost path before the 2031 anchor -- 2016 points; max rel err 1.40e-07
[INFO] rd B :: the learning-on runs write an in-run SMR cost path (max deviation from the drawn path, 2031-2050; context) -- median over runs 0.435; range 0.165-0.879


[PASS] rd B :: the shipped SMR plantchar files reproduce the offline cost worlds on the drawn 2031-2050 map (ITC-arm: u81 occ_world_kW in 2024$; anchors: the mc_perdraw.npz row of the u82 draw in native 2022$) -- 48 ITC-arm build-year points and 12 anchors x 20 years within 1e-3
[INFO] rd B :: before the 2031 anchor the plantchar files carry the pre-anchor path while the npz holds the flat BOAK value (outside the drawn map; context) -- 12 anchors x 7 pre-2031 years; max rel difference 0.40
[PASS] rd B :: a dual key exists in exactly the 18 mandate files and in none of the 48 ITC-arm files -- nuclear_cap_price present in 12 anchors + 6 horizon runs, absent elsewhere


[PASS] rd B :: the horizon runs' extended mandate files equal the standard file through 2050 and stay flat at the 2050 level through 2055 -- 6 files; 2051-2055 == 2050 MW; 2031-2050 == the standard trajectory


### 2.2 Delivery outcomes and shadow prices (phase C)

The gap table compares national SMR capacity (all vintages) with the trajectory in every mandated
model year of each credit run; `status` is `rate` in a build year with an offer and `slack`
otherwise. The classifier of section 1.4 turns it into `delivery_outcomes.csv`, joined with the
u80 spec (gate, draw, expected outcome, margin). For the 18 mandate runs the duals table follows
the convention of section 1; the tests confirm the binding and slack pattern, the conversion
`raw = cost_scale x pvf_onm x converted` on each run's own `pvf_onm`, and a slack post-2050 mandate
in the horizon runs.

In [11]:
# --- national SMR capacity + gap table (48 credit runs) ---------------------------------------------
SMR_CAP_RD = {c: smr_cap(BATCH_OF_RD[c], c) for c in CASES_RD + CTRL_RD}
LARGE_NEW_RD = {c: load(B_RD, c, "cap_new_ann").query("i == 'nuclear' and t > 2030").groupby("t")["Value"].sum() for c in CASES_RD}
gap_rows = []
for c in ITC_RD:
    for t in YEARS_STD:
        req = float(TRAJ_RD[c].get(t, 0.0))
        if req <= 0:
            continue
        have = float(SMR_CAP_RD[c].get(t, 0.0))
        gap_rows.append(dict(case=c, block=BLOCK_RD[c], gate=u80.loc[c, "gate"], sched=sched_of(c), t=t,
                             status="rate" if t in OFFER_RD[c] else "slack", mandate_MW=req, cap_MW=round(have, 1),
                             gap_MW=round(have - req, 1), gap_pct=round(100 * (have - req) / req, 2)))
gaps_rd = pd.DataFrame(gap_rows)
gaps_rd.to_csv(F_RD / "trajectory_gaps.csv", index=False)

out_rows = []
for c in ITC_RD:
    sub = gaps_rd[gaps_rd["case"] == c]
    gated = (sub["status"] == "rate") | (sub["t"] == 2050)
    tier = tier_of(sub, gated)
    cap_max = float(SMR_CAP_RD[c].max()) if len(SMR_CAP_RD[c]) else 0.0
    outcome, misses, gmax, end_req = outcome_of(sub, tier, cap_max, sub["status"] == "rate", gated)
    builds = SMR_CAP_RD[c][SMR_CAP_RD[c] > 5.0]
    out_rows.append(dict(case=c, block=BLOCK_RD[c], gate=u80.loc[c, "gate"], sched=sched_of(c),
                         mult=u80.loc[c, "mult"], draw_index=int(u80.loc[c, "draw_index"]),
                         expected=u80.loc[c, "expected"], cap=u80.loc[c, "cap"],
                         margin_ratepts=u80.loc[c, "margin_ratepts"], target_ratepts=u80.loc[c, "target_ratepts"],
                         deficit_years_spec=u80.loc[c, "deficit_years"],
                         tier=tier, outcome=outcome, delivers=tier in ("exact", "reproduces"),
                         miss_years="|".join(str(t) for t in misses), max_gated_shortfall_MW=round(gmax, 1),
                         first_build_year=int(builds.index.min()) if len(builds) else 0,
                         smr_2050_MW=round(float(SMR_CAP_RD[c].get(2050, 0.0)), 1), traj_2050_MW=end_req,
                         delivered_frac_2050=round(float(SMR_CAP_RD[c].get(2050, 0.0)) / end_req, 4)))
outc_rd = pd.DataFrame(out_rows)
outc_rd.to_csv(F_RD / "delivery_outcomes.csv", index=False)
for blk in ["envelope", "boundary", "hybrid"]:
    sub = outc_rd[outc_rd["case"].map(lambda c: u80.loc[c, "block"]) == blk]
    rec("C", f"the delivery tiers and outcomes of the {blk} runs (a finding, not a pass condition)", "INFO",
        sub.groupby(["expected", "outcome"])["case"].count().to_dict())
rec("C", "the pre-registered outcome classification of all 48 ITC-arm runs", "INFO",
    outc_rd.groupby(["gate", "outcome"])["case"].count().to_dict())
print(outc_rd.groupby(["gate", "outcome"]).size().to_string())

[INFO] rd C :: the delivery tiers and outcomes of the envelope runs (a finding, not a pass condition) -- {('deliver', 'delayed-start-recover'): 1, ('deliver', 'full-delivery'): 14, ('deliver', 'under-delivery'): 3, ('shortfall', 'full-delivery'): 6, ('shortfall', 'under-delivery'): 3}
[INFO] rd C :: the delivery tiers and outcomes of the boundary runs (a finding, not a pass condition) -- {('shortfall', 'full-delivery'): 1, ('shortfall', 'under-delivery'): 6}
[INFO] rd C :: the delivery tiers and outcomes of the hybrid runs (a finding, not a pass condition) -- {('deliver', 'delayed-start-recover'): 1, ('deliver', 'full-delivery'): 4, ('deliver', 'under-delivery'): 1, ('shortfall', 'full-delivery'): 4, ('shortfall', 'under-delivery'): 4}
[INFO] rd C :: the pre-registered outcome classification of all 48 ITC-arm runs -- {('GE1', 'delayed-start-recover'): 1, ('GE1', 'full-delivery'): 14, ('GE1', 'under-delivery'): 3, ('GE2', 'full-delivery'): 7, ('GE2', 'under-delivery'): 9, ('GH1', 'delay

In [12]:
# --- C4: shadow prices of the 18 mandate runs (+ the 6 smr100 controls for the checks) ---------------
sub_lg = {c: round(float(LARGE_NEW_RD[c].sum()) / 1000.0, 2) for c in ITC_RD if float(LARGE_NEW_RD[c].sum()) > 0}
rec("C", "large-reactor substitution in the ITC-arm runs (a report, not a failure)", "INFO",
    sub_lg or "no new large builds after 2030 in any of the 48 runs")
DUAL_RD = {c: dual_series(BATCH_OF_RD[c], c) for c in MAND_RD + CTRL_RD}
duals_all_rd = pd.DataFrame([row for c in MAND_RD + CTRL_RD for row in duals_rows(
    c, TRAJ_RD[c], SMR_CAP_RD[c], DUAL_RD[c][0], DUAL_RD[c][1], META_RD[c]["yearset"])])
duals_rd = duals_all_rd[duals_all_rd["case"].isin(MAND_RD)].reset_index(drop=True)
duals_rd.to_csv(F_RD / "duals_by_year.csv", index=False)
dual_checks(rec, duals_all_rd, "12 anchors + 6 horizon")
cross_check_06a(rec, duals_all_rd, CTRL_RD, "six p50 controls")

worst_conv = 0.0
for c in MAND_RD:
    dual, raw = DUAL_RD[c]
    pvf = PVF_RD[c]
    implied = raw / dual / CS_RD[c]
    worst_conv = max(worst_conv, float(((implied - pvf.reindex(implied.index)).abs() / pvf.reindex(implied.index)).max()))
rec("C", "the dual conversion is correct on each mandate run's own pvf_onm (raw = cost_scale x pvf_onm x converted)",
    "PASS" if worst_conv < 1e-4 else "FAIL", f"max rel err {worst_conv:.2e} across 18 runs")
post = duals_rd[duals_rd["case"].isin(HZ_RD) & (duals_rd["t"] > 2050)]
rec("C", "the flat post-2050 mandate is slack in every horizon run (2053/2055: capacity above the floor, dual zero-suppressed)",
    "PASS" if (post["slack_MW"] >= -5).all() and (post["dual_2004_MWyr"] == 0).all() else "FAIL",
    f"{len(post)} rows; min slack {post['slack_MW'].min():.0f} MW; max dual {post['dual_2004_MWyr'].max():.1f}")
ovb_rd = overbuild_rows(duals_rd)
ovb_rd.to_csv(F_RD / "overbuild_by_case.csv", index=False)
assert len(duals_rd) == 12 * 16 + 6 * 18, f"the duals table has one row per mandate run and model year, found {len(duals_rd)}"
print(ovb_rd.to_string(index=False))

[INFO] rd C :: large-reactor substitution in the ITC-arm runs (a report, not a failure) -- no new large builds after 2030 in any of the 48 runs
[PASS] rd C :: the capacity satisfies the mandate floor in every mandate run (12 anchors + 6 horizon) -- no violation larger than 5 MW in any mandated year
[PASS] rd C :: the capacity equals the floor in each binding year (positive dual) -- every year with dual > $1/MW-yr has |slack| <= 5 MW
[PASS] rd C :: a positive dual occurs only in the binding years -- every year with slack > 100 MW has a zero dual


[PASS] rd C :: the duals table reproduces the step3_checks rows of the six p50 controls -- max |difference| 5.8e-11 over 96 rows
[PASS] rd C :: the dual conversion is correct on each mandate run's own pvf_onm (raw = cost_scale x pvf_onm x converted) -- max rel err 1.29e-07 across 18 runs
[PASS] rd C :: the flat post-2050 mandate is slack in every horizon run (2053/2055: capacity above the floor, dual zero-suppressed) -- 12 rows; min slack 0 MW; max dual 0.0


            case  n_mandated_years  n_binding_years  n_slack_years  first_binding_year  max_overbuild_ratio
  smr100_eia_p25                 5                5              0                2038                  1.0
  smr100_eia_p75                 5                5              0                2038                  1.0
   smr100_aj_p25                10               10              0                2031                  1.0
   smr100_aj_p75                10               10              0                2031                  1.0
 smr100_iaea_p25                10               10              0                2031                  1.0
 smr100_iaea_p75                10               10              0                2031                  1.0
  smr100_mck_p25                10               10              0                2031                  1.0
  smr100_mck_p75                10               10              0                2031                  1.0
smr100_cop28_p25            

### 2.3 Cross-case invariance and the horizon runs (phase D)

The 66 runs differ only in their nuclear inputs and, for the horizon block, the end year. The
exogenous load must therefore equal the Step 3 reference; no table may hold a non-finite value or
a negative value where none is allowed; the national tables must equal the regional sums; and no
SMR capacity may exist before 2031. Each horizon run is compared with its p50 anchor year by year
(`hz_identity.csv`): a sequential solve should reproduce the anchor before 2050, so any drift is
disclosed; the 2050 rows are the GX1 object.

In [13]:
li_rd = load_invariance(B_RD, CASES_RD)
li_rd.to_csv(F_RD / "load_invariance.csv", index=False)
rec("D", "the exogenous load is identical in all 66 runs and equal to the Step 3 reference load on the shared years",
    "PASS" if li_rd.max_rel_diff.max() < 1e-6 else "FAIL", f"max rel difference {li_rd.max_rel_diff.max():.2e} vs {REF_LOAD}")
data_scans(rec, B_RD, CASES_RD, 66, len(UNION_RD))

hz_rows = []
for c in HZ_RD:
    b = f"smr100_{sched_of(c)}_p50"
    sc_c = load(B_RD, c, "systemcost").groupby("t")["Value"].sum()
    sc_b = load("smr100", b, "systemcost").groupby("t")["Value"].sum()
    pvf_b = load("smr100", b, "pvf_onm").set_index("t")["Value"]
    for t in [y for y in YEARS_STD if y >= 2031]:
        d_c, d_b = float(DUAL_RD[c][0].get(t, 0.0)), float(DUAL_RD[b][0].get(t, 0.0))
        hz_rows.append(dict(case=c, anchor=b, t=t, terminal=(t == 2050), dual_hz=d_c, dual_anchor=d_b,
                            dual_rel_diff=(abs(d_c - d_b) / max(abs(d_b), 1.0)),
                            smr_cap_hz_MW=round(float(SMR_CAP_RD[c].get(t, 0.0)), 1),
                            smr_cap_anchor_MW=round(float(SMR_CAP_RD[b].get(t, 0.0)), 1),
                            syscost_rel_diff=abs(float(sc_c.get(t, 0.0)) - float(sc_b.get(t, 0.0))) / max(abs(float(sc_b.get(t, 0.0))), 1.0),
                            pvf_onm_hz=float(PVF_RD[c].loc[t]), pvf_onm_anchor=float(pvf_b.loc[t])))
hzid = pd.DataFrame(hz_rows)
hzid.to_csv(F_RD / "hz_identity.csv", index=False)
pre, term = hzid[~hzid["terminal"]], hzid[hzid["terminal"]]
rec("D", "each horizon run reproduces its p50 anchor on the pre-terminal years 2031-2047 (shadow price, SMR capacity, "
    "system cost) -- disclosed under GX1 either way", "INFO",
    f"max rel diff 2031-2047: dual {pre.dual_rel_diff.max():.2e}, system cost {pre.syscost_rel_diff.max():.2e}, "
    f"SMR capacity {(pre.smr_cap_hz_MW - pre.smr_cap_anchor_MW).abs().max():.1f} MW")
rec("D", "the 2050 shadow price of each horizon run against its p50 anchor (the GX1 object; 2004$/MW-yr)", "INFO",
    {r.case: (round(r.dual_hz), round(r.dual_anchor), round(r.dual_hz / r.dual_anchor - 1, 4)) for r in term.itertuples()})
pre_smr = {c: float(load(B_RD, c, "cap").query("i == 'nuclear-smr' and t < 2031")["Value"].sum()) for c in CASES_RD}
rec("D", "no SMR capacity exists before 2031", "PASS" if max(pre_smr.values()) == 0 else "FAIL",
    f"max pre-2031 SMR capacity {max(pre_smr.values()):.1f} MW")
added_scans(rec, B_RD, CASES_RD, 66)

[PASS] rd D :: the exogenous load is identical in all 66 runs and equal to the Step 3 reference load on the shared years -- max rel difference 1.18e-07 vs smr100_eia_p50


[PASS] rd D :: no data value is NaN or infinite in any file -- all 209 keys x 66 files scanned (key inventory)
[PASS] rd D :: no negative values occur where values must not be negative -- checked ['cap', 'cap_nat', 'load_rt', 'stor_in', 'stor_out', 'curt_ann', 'hours'] (storage charging and upgrades are legitimately negative elsewhere)
[PASS] rd D :: the nuclear build and generation data is not negative -- cap_new_ann and gen_ann nuclear rows >= 0, all 66 cases


[PASS] rd D :: the national totals equal the sum of the regional data -- cap, gen_ann, cap_new_ann, ret_ann; all 66 cases within 1e-3
[INFO] rd D :: each horizon run reproduces its p50 anchor on the pre-terminal years 2031-2047 (shadow price, SMR capacity, system cost) -- disclosed under GX1 either way -- max rel diff 2031-2047: dual 4.41e-05, system cost 9.27e-05, SMR capacity 0.0 MW
[INFO] rd D :: the 2050 shadow price of each horizon run against its p50 anchor (the GX1 object; 2004$/MW-yr) -- {'hz_eia_p50_ext': (186963, 186960, 0.0), 'hz_aj_p50_ext': (221874, 221874, -0.0), 'hz_iaea_p50_ext': (139186, 139187, -0.0), 'hz_mck_p50_ext': (161720, 161709, 0.0001), 'hz_cop28_p50_ext': (149888, 149888, 0.0), 'hz_eo_p50_ext': (189516, 189517, -0.0)}


[PASS] rd D :: no SMR capacity exists before 2031 -- max pre-2031 SMR capacity 0.0 MW
[PASS] rd X :: no load is dropped after 2025 (the step3_checks E3 rule) -- 66 runs; the historical 2010-2023 artifact 2.96 TWh (case spread 543.2 MWh)


[INFO] rd X :: national curtailment in 2050 (context) -- 1.8 to 33.6 TWh (median 7.9) over 66 runs


### 2.4 The t09 chain on the p50 controls and the horizon runs (phase E; GX1)

The t09 chain of section 1.3 runs twice: on the six p50 controls on the standard 16-year grid (the
rows must equal notebook 07's t09 rows for the same cases: the port-fidelity gate GP of the
internal checks), and on the six horizon runs on the extended 18-year grid. The horizon runs'
2053/2055 rows carry a zero dual, so the status ladder classifies them slack before any arithmetic
touches a post-2050 year. `gx1_horizon.csv` then compares the 2047-to-2050 rise of the required
rate in the extended run (`uptick_ext`) with the standard anchor (`uptick_std`): a shed fraction of
0.5 or more reads the rise as a terminal-year artifact.

In [14]:
T09_IN_RD = t09_inputs(BATCH_OF_RD, CTRL_RD + HZ_RD, META_RD)
t09_ctrl_rd = t09_table(duals_all_rd, list(YEARS_STD), CTRL_RD, META_RD, TRAJ_RD, T09_IN_RD)
t09_ctrl_rd.to_csv(F_RD / "t09_required_itc_port_controls.csv", index=False)
t09_hz = t09_table(duals_all_rd, list(YEARS_EXT), HZ_RD, META_RD, TRAJ_RD, T09_IN_RD)
t09_hz.to_csv(F_RD / "t09_required_itc_hz.csv", index=False)
post = t09_hz[t09_hz["t"] > 2050]
rec("E", "the horizon runs' 2053/2055 rows classify as slack (zero dual) and no rate row lies past 2050 -- the natfin year "
    "ceiling never bites", "PASS" if (post["status"] == "slack: required ITC is zero").all() and len(post) == 12
    and t09_hz[t09_hz["status"] == "rate"]["t"].max() <= 2050 else "FAIL",
    f"{len(post)} post-2050 rows: {post['status'].unique().tolist()}")

gx_rows = []
for ab in SCHEDULES:
    hz, ctrl = f"hz_{ab}_p50_ext", f"smr100_{ab}_p50"
    r_e, s_e = t09_hz[t09_hz["case"] == hz].set_index("t")["i_model_headline"], t09_hz[t09_hz["case"] == hz].set_index("t")["status"]
    r_s, s_s = t09_ctrl_rd[t09_ctrl_rd["case"] == ctrl].set_index("t")["i_model_headline"], t09_ctrl_rd[t09_ctrl_rd["case"] == ctrl].set_index("t")["status"]
    up_std = float(r_s.loc[2050] - r_s.loc[2047])
    up_ext = float(r_e.loc[2050] - r_e.loc[2047])
    shed = 1.0 - up_ext / up_std if up_std != 0 else np.nan
    pre_years = [t for t in YEARS_STD if 2031 <= t <= 2047 and s_s.get(t) == "rate"]
    gx_rows.append(dict(schedule=ab, hz_case=hz, anchor=ctrl,
                        rate2047_std=float(r_s.loc[2047]), rate2050_std=float(r_s.loc[2050]),
                        rate2047_ext=float(r_e.loc[2047]), rate2050_ext=float(r_e.loc[2050]),
                        uptick_std=round(up_std, 4), uptick_ext=round(up_ext, 4), shed_fraction=round(shed, 4),
                        verdict=("artifact: no-2050 basis" if shed >= 0.5 else "persists: with-2050 basis"),
                        max_abs_rate_diff_2031_2047=round(max(abs(float(r_e.loc[t]) - float(r_s.loc[t])) for t in pre_years), 4),
                        pvf_onm_2050_hz=float(PVF_RD[hz].loc[2050]),
                        pvf_onm_2050_anchor=float(load("smr100", ctrl, "pvf_onm").set_index("t")["Value"].loc[2050]),
                        status_2053=s_e.get(2053), status_2055=s_e.get(2055)))
gx1 = pd.DataFrame(gx_rows)
gx1.to_csv(F_RD / "gx1_horizon.csv", index=False)
rec("E", "GX1 metric per schedule: the 2047->2050 uptick in the extended run vs the standard anchor (shed >= 0.5 = artifact; "
    "the verdict notebook adjudicates)", "INFO", {r.schedule: (r.uptick_std, r.uptick_ext, r.shed_fraction) for r in gx1.itertuples()})
rec("E", "pre-terminal identity of the extended runs' rates (max |rate difference| 2031-2047)", "INFO",
    {r.schedule: r.max_abs_rate_diff_2031_2047 for r in gx1.itertuples()})
print(gx1[["schedule", "uptick_std", "uptick_ext", "shed_fraction", "verdict"]].to_string(index=False))
S_RD = write_registry(REG_RD, F_RD)

[PASS] rd E :: the horizon runs' 2053/2055 rows classify as slack (zero dual) and no rate row lies past 2050 -- the natfin year ceiling never bites -- 12 post-2050 rows: ['slack: required ITC is zero']
[INFO] rd E :: GX1 metric per schedule: the 2047->2050 uptick in the extended run vs the standard anchor (shed >= 0.5 = artifact; the verdict notebook adjudicates) -- {'eia': (0.054, 0.054, 0.0), 'aj': (0.059, 0.059, 0.0), 'iaea': (0.068, 0.068, 0.0), 'mck': (0.033, 0.033, 0.0), 'cop28': (0.079, 0.079, 0.0), 'eo': (0.045, 0.045, 0.0)}
[INFO] rd E :: pre-terminal identity of the extended runs' rates (max |rate difference| 2031-2047) -- {'eia': 0.0, 'aj': 0.0, 'iaea': 0.0, 'mck': 0.0, 'cop28': 0.0, 'eo': 0.0}
schedule  uptick_std  uptick_ext  shed_fraction                   verdict
     eia       0.054       0.054            0.0 persists: with-2050 basis
      aj       0.059       0.059            0.0 persists: with-2050 basis
    iaea       0.068       0.068            0.0 persists: with-

## 3. `ratedesign2`: 36 map anchors and 57 credit runs

**What the batch tests.** The 36 map anchors (`ma_{schedule}_d{draw}`, 6 per schedule) are mandate
anchors in fresh cost samples, chosen to widen the requirement map beyond the 30 solved anchors
(5 percentiles x 6 schedules). Their shadow prices go through the t09 chain of section 1.3:
`t09_required_itc_port_anchors.csv` is the exact required rate of each map anchor in each model year,
the input that notebook 10 joins to the 30 solved anchors to fit the requirement map (Fig 3, Fig 4,
Table 2). The 57 credit runs (`ax_*`) test offered credit paths in the 30 solved anchor samples:
the exact required rate (the control), the same rate capped after 2035, and a flat 60% credit,
under fbB (learning OFF, arms B, C, E) or fbC (learning ON, arms D, F).

**Tables.** `duals_by_year.csv` (36 map anchors), `t09_required_itc_port_anchors.csv`,
`t09_required_itc_port_controls.csv` (the chain on the 30 solved anchors; must equal notebook 07's
t09 rows), `overbuild_by_case.csv`, `trajectory_gaps.csv` and `delivery_outcomes.csv` (57 credit
runs, joined with the m02 spec), `pair_identity.csv` (each capped run against its control through
2035, where the two offers are equal), and the sanity outputs.

In [15]:
# ---- ratedesign2: case matrix, registration, metadata -----------------------------------------
F_RD2 = outdir("rate_design2_checks")
REG_RD2 = []
rec = make_record(REG_RD2, "rd2")
B_RD2 = "ratedesign2"
cases_rd2 = casefile("ratedesign2")
CASES_RD2 = [c for c in cases_rd2.columns if c != "Default Value"]
assert len(CASES_RD2) == 93, f"the ratedesign2 casefile has 93 runs, found {len(CASES_RD2)}"
PCTS5 = ["p05", "p25", "p50", "p75", "p95"]
ANCHORS30 = [f"smr100_{ab}_{p}" for ab in SCHEDULES for p in PCTS5]      # the 30 solved anchors
BATCH_OF_A30 = {c: ("smr100" if c.split("_")[2] in ("p05", "p50", "p95") else "ratedesign") for c in ANCHORS30}
m01 = pd.read_csv(REG_DIR / "ratedesign2" / "m01_map_anchor_spec.csv").set_index("run")
m02 = pd.read_csv(REG_DIR / "ratedesign2" / "m02_credit_run_spec.csv").set_index("run")
m04 = pd.read_csv(REG_DIR / "ratedesign2" / "m04_offers.csv")
MA_RD2 = [c for c in CASES_RD2 if c.startswith("ma_")]
AX_RD2 = [c for c in CASES_RD2 if c.startswith("ax_")]
assert [len(MA_RD2), len(AX_RD2)] == [36, 57]
assert set(m01.index) == set(MA_RD2) and set(m02.index) == set(AX_RD2) and set(m04["run"]) == set(AX_RD2)
FBB_RD2 = [c for c in AX_RD2 if m02.loc[c, "convention"] == "fbB"]
FBC_RD2 = [c for c in AX_RD2 if m02.loc[c, "convention"] == "fbC"]
assert [len(FBB_RD2), len(FBC_RD2)] == [43, 14]
BLOCK_RD2 = {c: ("anchor" if c.startswith("ma_") else "credit") for c in CASES_RD2}
sw_rd2 = make_sw([cases_rd2, cases_rd, cases_s100], cases_rd2)
META_RD2 = {c: meta_of(sw_rd2, c) for c in CASES_RD2 + ANCHORS30}
for c in AX_RD2:
    META_RD2[c].update(arm=m02.loc[c, "arm"], conv=m02.loc[c, "convention"], form=m02.loc[c, "form"],
                       cap=m02.loc[c, "cap"], world=m02.loc[c, "world"], gate=m02.loc[c, "gate"])
BATCH_OF_RD2 = {**{c: B_RD2 for c in CASES_RD2}, **BATCH_OF_A30}
OFFER_RD2 = {r: g.set_index("year")["offer_rate"].to_dict() for r, g in m04.groupby("run")}
RATES_RD2 = {r: {int(t): round(float(v), 3) for t, v in OFFER_RD2[r].items()} for r in AX_RD2}
M_EXP_RD2 = {r: {t: f * (1.0 - PEN) for t, f in RATES_RD2[r].items()} for r in AX_RD2}
W_PRIMARY = 2035
print(f"ratedesign2: {len(MA_RD2)} map anchors + {len(AX_RD2)} credit runs ({len(FBB_RD2)} fbB + {len(FBC_RD2)} fbC); "
      f"solved anchors {len(ANCHORS30)}")

ratedesign2: 36 map anchors + 57 credit runs (43 fbB + 14 fbC); solved anchors 30


### 3.1 Solve health and design echo (phases A and B)

The same tests as section 2.1, with the rd2 design rules: map anchors have the mandate ON,
learning OFF, the no-ITC baseline and their own `mc_ma_` cost files; fbB credit runs use the
anchor's `smr100` files with learning OFF; fbC credit runs use the `rd_` sample files with the
three learning channels ON. The map anchors' m01 rows must equal the drawn cost samples, and
the offered paths must carry the registered pair identities (control = capped run through 2035;
capped rate never above the cap after 2035; flat runs at 0.60 in every model year).

In [16]:
missing = [c for c in CASES_RD2 if not returned(B_RD2, c)]
rec("A", "one output file exists for each of the 93 cases", "PASS" if not missing else "FAIL",
    f"missing: {missing}" if missing else "93 runs in the key inventory")
ref_missing = [c for c in ANCHORS30 if not returned(BATCH_OF_A30[c], c)] + ([] if returned("step4", REF4) else [REF4])
rec("A", "the reference files are available (30 solved anchors, 1 key-vintage reference)",
    "PASS" if not ref_missing else "FAIL",
    f"missing: {ref_missing}" if ref_missing else "18 smr100 + 12 ratedesign anchors and the step4 reference present")
YEARS_STD2 = META_RD2[f"smr100_{SCHEDULES[0]}_p50"]["yearset"]
assert YEARS_STD2 == YEARS_STD
KS_RD2, UNION_RD2, PVF_RD2, CS_RD2 = solve_health(
    rec, B_RD2, CASES_RD2,
    years_of=lambda c: (YEARS_STD, META_RD2[c]["yearset"] == YEARS_STD and META_RD2[c]["endyear"] == 2050),
    key_labels=("all 93 files have the same data keys (dual keys excluded)",
                "the key set agrees with the Step 4 vintage (separate-delivery drift detector)"),
    key_exclude=DUAL_KEYS)
for c in ANCHORS30:
    PVF_RD2[c] = load(BATCH_OF_A30[c], c, "pvf_onm").set_index("t")["Value"]
    CS_RD2[c] = float(load(BATCH_OF_A30[c], c, "cost_scale")["Value"].iloc[0])

# --- B1: design-matrix echo per block ----------------------------------------------------------
mx_bad = []
for c in CASES_RD2:
    m_, ab = META_RD2[c], sched_of(c)
    tok = STEM2TOK[ab]
    common = m_["scen"] == f"{tok}_smr" and m_["endyear"] == 2050 and m_["techscen"] == "smr" and m_["yearset"] == YEARS_STD
    if BLOCK_RD2[c] == "anchor":
        w = int(m01.loc[c, "draw_index"])
        ok = (common and m_["mode"] == 1 and m_["learning"] == 0 and m_["suffix"] == "obbba_nonuclearitc" and m_["foreign"] == ""
              and m_["pc"]["nuclear-smr"] == f"nuclear-smr_mc_ma_{ab}_d{w}" and m_["pc"]["nuclear"] == f"nuclear_mc_ma_{ab}_d{w}"
              and m_["fin"] == f"mc_ma_{ab}_d{w}" and m_["constr"] == f"mc_ma_{ab}_d{w}" and m_["tcx"] == "0" and c == f"ma_{ab}_d{w}")
    elif m_["conv"] == "fbB":
        p = m02.loc[c, "pct"]
        ok = (common and m_["mode"] == 0 and m_["learning"] == 0 and m_["suffix"] == f"obbba_rd_{c}" and m_["foreign"] == ""
              and m_["pc"]["nuclear-smr"] == f"nuclear-smr_mc_smr100_{ab}_{p}" and m_["pc"]["nuclear"] == f"nuclear_mc_smr100_{ab}_{p}"
              and m_["fin"] == f"mc_smr100_{ab}_{p}" and m_["constr"] == f"mc_smr100_{ab}_{p}" and m_["tcx"] == "1"
              and m02.loc[c, "case"] == f"smr100_{ab}_{p}"
              and all(sw_rd2(c, k) == "" for k in ["GSw_NuclearLearning_OCC", "GSw_NuclearLearning_LR_smr"]))
    else:
        w = int(m02.loc[c, "draw_index"])
        ok = (common and m_["mode"] == 0 and m_["learning"] == 1 and m_["suffix"] == f"obbba_rd_{c}" and m_["foreign"] == f"rd_{ab}_d{w}"
              and m_["pc"]["nuclear-smr"] == f"nuclear-smr_mc_rd_{ab}_d{w}" and m_["pc"]["nuclear"] == f"nuclear_mc_rd_{ab}_d{w}"
              and m_["fin"] == f"mc_rd_{ab}_d{w}" and m_["constr"] == f"mc_rd_{ab}_d{w}" and m_["tcx"] == "1"
              and sw_rd2(c, "GSw_NuclearLearning_OCC") == "1" and sw_rd2(c, "GSw_NuclearLearning_Duration") == "1"
              and sw_rd2(c, "GSw_NuclearLearning_CrossTech") == "1")
    if not ok:
        mx_bad.append(c)
rec("B", "the case matrix encodes the intended block design (36 map anchors: mandate ON, learning OFF, no-ITC baseline, own "
    "files; 43 fbB credit runs: the anchor's files, learning OFF; 14 fbC credit runs: rd_ world files, learning ON)",
    "PASS" if not mx_bad else "FAIL", mx_bad or "93 cases match the block rules and the naming conventions")

b1b_bad = []
for c in FBC_RD2:
    ab, w = sched_of(c), int(m02.loc[c, "draw_index"])
    world = Z[f"worlds_{STEM2TOK[ab]}"][w]
    for col, name in SWITCH_OF.items():
        exp, got = float(world[WCOLS.index(col)]), float(sw_rd2(c, name))
        if abs(got - exp) > max(1e-5 * abs(exp), 5e-6):
            b1b_bad.append((c, col, got, exp))
rec("B", "the fbC learning switches equal the drawn world parameters at the m02 draw index",
    "PASS" if not b1b_bad else "FAIL", b1b_bad[:5] or f"14 runs x {len(SWITCH_OF)} switches match mc_perdraw.npz to 1e-5")
b1c_bad = []
for c in MA_RD2:
    ab, w = sched_of(c), int(m01.loc[c, "draw_index"])
    world = Z[f"worlds_{STEM2TOK[ab]}"][w]
    for col in WCOLS:
        v = float(world[WCOLS.index(col)])
        if abs(float(m01.loc[c, col]) - v) > 1e-9 * max(1.0, abs(v)):
            b1c_bad.append((c, col))
rec("B", "the 36 map anchors' m01 rows equal the drawn world parameters at their draw index",
    "PASS" if not b1c_bad else "FAIL", b1c_bad[:5] or f"36 anchors x {len(WCOLS)} parameters match to 1e-9")

# --- B2: fed incentives files vs m04; the pair identities -------------------------------------------
echo_rows, b2_bad = [], []
for r in AX_RD2:
    got, flags = incentives_rows(r, f"obbba_rd_{r}", RATES_RD2[r])
    if not all(flags.values()):
        b2_bad.append((r, flags))
    for t in sorted(set(got) | set(RATES_RD2[r])):
        echo_rows.append(dict(run=r, arm=META_RD2[r]["arm"], form=META_RD2[r]["form"], sched=sched_of(r), t=t,
                              itc_frac_file=got.get(t), itc_frac_m04=RATES_RD2[r].get(t)))
pd.DataFrame(echo_rows).to_csv(F_RD2 / "incentives_echo.csv", index=False)
rec("B", "the 57 fed incentives files equal the m04 credit paths at exactly the offered years (offer_rate at 3 decimals)",
    "PASS" if not b2_bad else "FAIL",
    b2_bad[:3] or "57 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0; every rate below 1.0")
pair_bad = []
for r in AX_RD2:
    f = META_RD2[r]["form"]
    if f == "capped_post2035":
        ctrl = [u for u in AX_RD2 if META_RD2[u]["world"] == META_RD2[r]["world"] and META_RD2[u]["conv"] == META_RD2[r]["conv"]
                and META_RD2[u]["form"] == "exact"]
        assert len(ctrl) == 1, (r, ctrl)
        a, b = RATES_RD2[ctrl[0]], RATES_RD2[r]
        cap = float(META_RD2[r]["cap"])
        if (sorted(a) != sorted(b) or any(a[t] != b[t] for t in a if t <= W_PRIMARY)
                or any(b[t] > cap + 1e-9 or b[t] > a[t] + 1e-9 for t in b if t > W_PRIMARY)):
            pair_bad.append(r)
    elif f == "flat60":
        if sorted(RATES_RD2[r]) != MODEL_YEARS or any(v != 0.60 for v in RATES_RD2[r].values()):
            pair_bad.append(r)
    elif f != "exact":
        pair_bad.append((r, f))
rec("B", "the offered paths carry the registered pair identities (control == capped run through 2035; capped rate never above "
    "the cap after 2035; flat-cap runs at 0.60 in every model year 2031-2050)",
    "PASS" if not pair_bad else "FAIL", pair_bad[:5] or "28 capped runs, 8 flat-cap runs, 21 controls")

rr_rd2 = pd.DataFrame([row for c in AX_RD2 for row in rate_recovery_rows(
    B_RD2, c, META_RD2[c]["fin"], M_EXP_RD2[c], dict(arm=META_RD2[c]["arm"], conv=META_RD2[c]["conv"], sched=sched_of(c)))])
rr_rd2.to_csv(F_RD2 / "rate_recovery.csv", index=False)
rate_landing_checks(rec, rr_rd2, OFFER_RD2, (
    "the recovered monetized rate equals 0.9 x the fed rate at exactly the offered years (57 credit runs; the construction "
    "multiplier cancels in the ratio)",
    "the recovered rate is zero outside the offered years and for the large reactor in all 57 runs",
    "the recovered rate is uniform across regions",
    "the phaseout does not reduce the nuclear credit in any year (exemption held)"))
FMD = {c: fmdiff(B_RD2, c) for c in MA_RD2}
rec("B", "the 36 map anchors carry no nuclear ITC (fin_mult == fin_mult_noITC for both nuclear technologies)",
    "PASS" if max(FMD.values()) < 1e-9 else "FAIL", f"max |fin_mult - fin_mult_noITC| {max(FMD.values()):.1e} over 36 runs")

SMR_NEW = {c: float(load(B_RD2, c, "cap_new_ann").query("i == 'nuclear-smr' and t > 2030")["Value"].sum()) for c in CASES_RD2}
pay_bad, no_build = [], []
for c in AX_RD2:
    sc = load(B_RD2, c, "systemcost_techba")
    v = float(sc[sc["sys_costs"].str.contains("itc", case=False) & sc["i"].isin(NUC)]["Value"].abs().sum())
    if SMR_NEW[c] > 100.0 and v < 1e6:
        pay_bad.append((c, f"builds {SMR_NEW[c]:,.0f} MW but no nuclear ITC payments"))
    if SMR_NEW[c] <= 100.0:
        no_build.append(c)
rec("B", "the system cost data shows nuclear ITC payments wherever SMR builds occurred (credit runs)",
    "PASS" if not pay_bad else "FAIL",
    pay_bad or f"payments present in every case with builds; cases with ~no builds (lock-in candidates, read in phase C): {no_build or 'none'}")
itc_mag = {c: abs(float(load(B_RD2, c, "tax_expenditure_itc")["Value"].sum())) for c in CASES_RD2}
rec("B", "the other technologies keep their ITC in all 93 runs", "PASS" if min(itc_mag.values()) > 1e8 else "FAIL",
    f"total ITC tax expenditure magnitude {min(itc_mag.values())/1e9:.1f} to {max(itc_mag.values())/1e9:.1f} B$ per case")

[PASS] rd2 A :: one output file exists for each of the 93 cases -- 93 runs in the key inventory
[PASS] rd2 A :: the reference files are available (30 solved anchors, 1 key-vintage reference) -- 18 smr100 + 12 ratedesign anchors and the step4 reference present
[PASS] rd2 A :: all 93 files have the same data keys (dual keys excluded) -- key counts [207, 209]
[PASS] rd2 A :: the key set agrees with the Step 4 vintage (separate-delivery drift detector) -- 209 keys, no drift


[PASS] rd2 A :: the solver residuals are small in every file -- worst |residual| 7.47e-04 at ax_eia_p75_u


[PASS] rd2 A :: the objective value is a normal positive number in every file -- range 1.3804e+12 to 1.9693e+12


[PASS] rd2 A :: the model years are the standard 16-year set with end year 2050 in every file -- every file on its registered year set


[PASS] rd2 A :: every run used the sequential solve mode -- pvf_capital==1 (max dev 0.0e+00); pvf_onm flat from 2026 (max dev 9.5e-07); cost_scale==1; z_rep has one row per model year
[PASS] rd2 B :: the case matrix encodes the intended block design (36 map anchors: mandate ON, learning OFF, no-ITC baseline, own files; 43 fbB credit runs: the anchor's files, learning OFF; 14 fbC credit runs: rd_ world files, learning ON) -- 93 cases match the block rules and the naming conventions
[PASS] rd2 B :: the fbC learning switches equal the drawn world parameters at the m02 draw index -- 14 runs x 11 switches match mc_perdraw.npz to 1e-5


[PASS] rd2 B :: the 36 map anchors' m01 rows equal the drawn world parameters at their draw index -- 36 anchors x 13 parameters match to 1e-9


[PASS] rd2 B :: the 57 fed incentives files equal the m04 credit paths at exactly the offered years (offer_rate at 3 decimals) -- 57 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0; every rate below 1.0
[PASS] rd2 B :: the offered paths carry the registered pair identities (control == capped run through 2035; capped rate never above the cap after 2035; flat-cap runs at 0.60 in every model year 2031-2050) -- 28 capped runs, 8 flat-cap runs, 21 controls


[PASS] rd2 B :: the recovered monetized rate equals 0.9 x the fed rate at exactly the offered years (57 credit runs; the construction multiplier cancels in the ratio) -- 525 fed case-years; worst |dm| 0.0001 at (ax_eo_p25_u, 2031)
[PASS] rd2 B :: the recovered rate is zero outside the offered years and for the large reactor in all 57 runs -- 843 points; max |m| 0.0000
[PASS] rd2 B :: the recovered rate is uniform across regions -- max spread over regions 0.00103
[PASS] rd2 B :: the phaseout does not reduce the nuclear credit in any year (exemption held) -- 285 credited case-years at t >= 2036; worst |dm| 0.0001


[PASS] rd2 B :: the 36 map anchors carry no nuclear ITC (fin_mult == fin_mult_noITC for both nuclear technologies) -- max |fin_mult - fin_mult_noITC| 0.0e+00 over 36 runs


[PASS] rd2 B :: the system cost data shows nuclear ITC payments wherever SMR builds occurred (credit runs) -- payments present in every case with builds; cases with ~no builds (lock-in candidates, read in phase C): ['ax_eia_p95_c60']
[PASS] rd2 B :: the other technologies keep their ITC in all 93 runs -- total ITC tax expenditure magnitude 123.6 to 1689.7 B$ per case


In [17]:
# --- B13: capital-cost fingerprints --------------------------------------------------------------
fp_rd2 = pd.DataFrame([row for c in CASES_RD2 for row in fingerprint_rows(B_RD2, c, META_RD2[c], dict(block=BLOCK_RD2[c]))])
fp_rd2.to_csv(F_RD2 / "fingerprint_errors.csv", index=False)
off_fp = fp_rd2[fp_rd2["learning"] == 0]
w_ = off_fp.loc[off_fp.rel_err.idxmax()]
rec("B", "the 79 learning-off runs use their plantchar cost path without change (map anchors: own mc_ma_ files; fbB credit runs: "
    "the anchor's smr100 files)", "PASS" if off_fp.rel_err.max() < 1e-4 else "FAIL",
    f"{len(off_fp)} (case, tech, year) points; max rel err {off_fp.rel_err.max():.2e} at ({w_.case}, {w_.i}, {w_.t})")
pre_fp = fp_rd2[(fp_rd2["learning"] == 1) & (fp_rd2["t"] <= 2030)]
rec("B", "the 14 learning-on runs use the drawn cost path before the 2031 anchor",
    "PASS" if pre_fp.rel_err.max() < 1e-4 else "FAIL", f"{len(pre_fp)} points; max rel err {pre_fp.rel_err.max():.2e}")
dev = fp_rd2[(fp_rd2["learning"] == 1) & (fp_rd2["t"] >= 2031) & (fp_rd2["i"] == "nuclear-smr")].groupby("case")["rel_err"].max().round(3)
rec("B", "the learning-on runs write an in-run SMR cost path (max deviation from the drawn path, 2031-2050; context, read by GW5)",
    "INFO", f"median over runs {dev.median():.3f}; range {dev.min():.3f}-{dev.max():.3f}; per run: {dev.to_dict()}")
map_bad, pre_dev = [], []
for r in AX_RD2:
    pc_df = plantchar(META_RD2[r]["pc"]["nuclear-smr"])
    for t, occ in m04[m04["run"] == r].set_index("year")["occ_world_kW"].items():
        got = float(pc_df.loc[t, "capcost"]) * D2224
        if abs(got - occ) / occ > 1e-3:
            map_bad.append((r, t, round(got, 1), occ))
for c in MA_RD2:
    ab, w = sched_of(c), int(m01.loc[c, "draw_index"])
    occ_npz = Z[f"occ_{STEM2TOK[ab]}_smr"][w]
    pc_df = plantchar(META_RD2[c]["pc"]["nuclear-smr"])
    for j, t in enumerate(NPZ_YEARS):
        if t not in pc_df.index:
            continue
        rel = abs(float(pc_df.loc[t, "capcost"]) - occ_npz[j]) / occ_npz[j]
        if t >= 2031 and rel > 1e-3:
            map_bad.append((c, t))
        if t < 2031:
            pre_dev.append(rel)
rec("B", "the shipped SMR plantchar files reproduce the offline cost worlds on the drawn 2031-2050 map (credit runs: m04 "
    "occ_world_kW in 2024$; map anchors: the mc_perdraw.npz row of the m01 draw in native 2022$)",
    "PASS" if not map_bad else "FAIL", map_bad[:5] or "57 credit runs' offered-year points and 36 anchors x 20 years within 1e-3")
rec("B", "before the 2031 anchor the plantchar files carry the pre-anchor path while the npz holds the flat BOAK value (outside "
    "the drawn map; context)", "INFO", f"36 anchors x 7 pre-2031 years; max rel difference {max(pre_dev):.2f}")
b15_bad = sorted(c for c in CASES_RD2 if bool(KS_RD2[c] & DUAL_KEYS) != (c in MA_RD2))
rec("B", "a dual key exists in exactly the 36 map-anchor files and in none of the 57 credit files",
    "PASS" if not b15_bad else "FAIL", b15_bad or "nuclear_cap_price present in the 36 map anchors, absent elsewhere")
TRAJ_RD2 = {c: trajectory(META_RD2[c]["scen"]) for c in CASES_RD2 + ANCHORS30}
b16_bad = [c for c in CASES_RD2 + ANCHORS30 if META_RD2[c]["scen"] != f"{STEM2TOK[sched_of(c)]}_smr"]
rec("B", "every run reads its schedule's standard mandate trajectory file (no _ext token)",
    "PASS" if not b16_bad else "FAIL", b16_bad or "93 runs + 30 solved anchors on the six standard files")

[PASS] rd2 B :: the 79 learning-off runs use their plantchar cost path without change (map anchors: own mc_ma_ files; fbB credit runs: the anchor's smr100 files) -- 6478 (case, tech, year) points; max rel err 3.73e-07 at (ma_eo_d2354, nuclear-smr, 2048)
[PASS] rd2 B :: the 14 learning-on runs use the drawn cost path before the 2031 anchor -- 588 points; max rel err 1.05e-07
[INFO] rd2 B :: the learning-on runs write an in-run SMR cost path (max deviation from the drawn path, 2031-2050; context, read by GW5) -- median over runs 0.168; range 0.067-0.248; per run: {'ax_aj_p50_c60_lrn': 0.067, 'ax_aj_p50_u_lrn': 0.078, 'ax_aj_p95_c60': 0.155, 'ax_aj_p95_u': 0.183, 'ax_cop28_p75_c60': 0.182, 'ax_cop28_p75_u': 0.182, 'ax_eia_p95_c60': 0.181, 'ax_eia_p95_u': 0.248, 'ax_eo_p75_c60': 0.202, 'ax_eo_p75_u': 0.202, 'ax_iaea_p95_c60': 0.077, 'ax_iaea_p95_u': 0.077, 'ax_mck_p95_c60': 0.087, 'ax_mck_p95_u': 0.087}


[PASS] rd2 B :: the shipped SMR plantchar files reproduce the offline cost worlds on the drawn 2031-2050 map (credit runs: m04 occ_world_kW in 2024$; map anchors: the mc_perdraw.npz row of the m01 draw in native 2022$) -- 57 credit runs' offered-year points and 36 anchors x 20 years within 1e-3
[INFO] rd2 B :: before the 2031 anchor the plantchar files carry the pre-anchor path while the npz holds the flat BOAK value (outside the drawn map; context) -- 36 anchors x 7 pre-2031 years; max rel difference 0.44
[PASS] rd2 B :: a dual key exists in exactly the 36 map-anchor files and in none of the 57 credit files -- nuclear_cap_price present in the 36 map anchors, absent elsewhere


[PASS] rd2 B :: every run reads its schedule's standard mandate trajectory file (no _ext token) -- 93 runs + 30 solved anchors on the six standard files


### 3.2 Delivery outcomes and shadow prices (phase C)

The gate years of a credit run are its offered years in m04 (the rate-status years for the exact
and capped forms; every model year 2031 to 2050 for the flat form) plus 2050. `prediction_met`
records whether the registered prediction (deliver or shortfall) matched the classifier. The duals
table covers the 36 map anchors; the 30 solved anchors are rebuilt beside them from the smr100 and
ratedesign extracts, because the t09 chain needs their rows too.

In [18]:
SMR_CAP_RD2 = {c: smr_cap(BATCH_OF_RD2[c], c) for c in CASES_RD2 + ANCHORS30}
LARGE_NEW_RD2 = {c: load(B_RD2, c, "cap_new_ann").query("i == 'nuclear' and t > 2030").groupby("t")["Value"].sum() for c in CASES_RD2}
gap_rows = []
for c in AX_RD2:
    for t in YEARS_STD:
        req = float(TRAJ_RD2[c].get(t, 0.0))
        if req <= 0:
            continue
        have = float(SMR_CAP_RD2[c].get(t, 0.0))
        gap_rows.append(dict(case=c, arm=META_RD2[c]["arm"], gate=META_RD2[c]["gate"], sched=sched_of(c), t=t,
                             status="rate" if t in OFFER_RD2[c] else "slack", mandate_MW=req, cap_MW=round(have, 1),
                             gap_MW=round(have - req, 1), gap_pct=round(100 * (have - req) / req, 2)))
gaps_rd2 = pd.DataFrame(gap_rows)
gaps_rd2.to_csv(F_RD2 / "trajectory_gaps.csv", index=False)
out_rows = []
for c in AX_RD2:
    sub = gaps_rd2[gaps_rd2["case"] == c]
    gated = (sub["status"] == "rate") | (sub["t"] == 2050)
    tier = tier_of(sub, gated)
    cap_max = float(SMR_CAP_RD2[c].max()) if len(SMR_CAP_RD2[c]) else 0.0
    outcome, misses, gmax, end_req = outcome_of(sub, tier, cap_max, sub["status"] == "rate", gated)
    builds = SMR_CAP_RD2[c][SMR_CAP_RD2[c] > 5.0]
    s = m02.loc[c]
    delivers = tier in ("exact", "reproduces")
    out_rows.append(dict(case=c, arm=s["arm"], convention=s["convention"], gate=s["gate"], sched=sched_of(c),
                         world=s["world"], pct=s["pct"], draw_index=int(s["draw_index"]), cap=s["cap"], form=s["form"],
                         excess=s["excess"], max_post_rate=s["max_post_rate"], eligible=bool(s["eligible"]),
                         prediction=s["prediction"], tier=tier, outcome=outcome, delivers=delivers,
                         prediction_met=bool(delivers == (s["prediction"] == "deliver")),
                         miss_years="|".join(str(t) for t in misses), max_gated_shortfall_MW=round(gmax, 1),
                         first_build_year=int(builds.index.min()) if len(builds) else 0,
                         smr_2050_MW=round(float(SMR_CAP_RD2[c].get(2050, 0.0)), 1), traj_2050_MW=end_req,
                         delivered_frac_2050=round(float(SMR_CAP_RD2[c].get(2050, 0.0)) / end_req, 4)))
outc_rd2 = pd.DataFrame(out_rows)
outc_rd2.to_csv(F_RD2 / "delivery_outcomes.csv", index=False)
for arm in ["B", "C", "D", "E", "F"]:
    sub = outc_rd2[outc_rd2["arm"] == arm]
    rec("C", f"the delivery tiers and outcomes of the arm-{arm} runs (a finding, not a pass condition)", "INFO",
        sub.groupby(["prediction", "outcome"])["case"].count().to_dict())
rec("C", "the pre-registered outcome classification of all 57 credit runs (prediction met?)", "INFO",
    outc_rd2.groupby(["gate", "prediction_met"])["case"].count().to_dict())
sub_lg = {c: round(float(LARGE_NEW_RD2[c].sum()) / 1000.0, 2) for c in AX_RD2 if float(LARGE_NEW_RD2[c].sum()) > 0}
rec("C", "large-reactor substitution in the credit runs (a report, not a failure)", "INFO",
    sub_lg or "no new large builds after 2030 in any of the 57 runs")

DUAL_RD2 = {c: dual_series(BATCH_OF_RD2[c], c) for c in MA_RD2 + ANCHORS30}
duals_all_rd2 = pd.DataFrame([row for c in MA_RD2 + ANCHORS30 for row in duals_rows(
    c, TRAJ_RD2[c], SMR_CAP_RD2[c], DUAL_RD2[c][0], DUAL_RD2[c][1], META_RD2[c]["yearset"])])
duals_rd2 = duals_all_rd2[duals_all_rd2["case"].isin(MA_RD2)].reset_index(drop=True)
duals_rd2.to_csv(F_RD2 / "duals_by_year.csv", index=False)
dual_checks(rec, duals_all_rd2, "36 map anchors + 30 solved anchors")
# the 12 p25/p75 anchors must equal section 2's rows (same runs, same code); the 18 smr100 anchors are 06a's
DC = ["mandate_MW", "cap_MW", "slack_MW", "dual_2004_MWyr", "dual_raw", "dual_2024_MWyr"]
_a = duals_all_rd2[duals_all_rd2["case"].isin(ANCH_RD)].sort_values(["case", "t"]).reset_index(drop=True)
_b = duals_rd[duals_rd["case"].isin(ANCH_RD)].sort_values(["case", "t"]).reset_index(drop=True)
assert len(_a) == len(_b) == 12 * 16 and float((_a[DC] - _b[DC]).abs().max().max()) == 0.0, "the p25/p75 anchor duals differ between sections"
cross_check_06a(rec, duals_all_rd2, [c for c in ANCHORS30 if BATCH_OF_A30[c] == "smr100"], "18 smr100 solved anchors")
worst_conv = 0.0
for c in MA_RD2 + ANCHORS30:
    dual, raw = DUAL_RD2[c]
    pvf = PVF_RD2[c]
    implied = raw / dual / CS_RD2[c]
    worst_conv = max(worst_conv, float(((implied - pvf.reindex(implied.index)).abs() / pvf.reindex(implied.index)).max()))
rec("C", "the dual conversion is correct on each mandate run's own pvf_onm (raw = cost_scale x pvf_onm x converted)",
    "PASS" if worst_conv < 1e-4 else "FAIL", f"max rel err {worst_conv:.2e} across 66 runs")
ovb_rd2 = overbuild_rows(duals_rd2)
ovb_rd2.to_csv(F_RD2 / "overbuild_by_case.csv", index=False)
assert len(duals_rd2) == 36 * 16, f"36 map anchors x 16 model years, found {len(duals_rd2)}"

[INFO] rd2 C :: the delivery tiers and outcomes of the arm-B runs (a finding, not a pass condition) -- {('deliver', 'full-delivery'): 15, ('shortfall', 'delayed-start-recover'): 1, ('shortfall', 'full-delivery'): 3, ('shortfall', 'partial'): 1, ('shortfall', 'under-delivery'): 6}
[INFO] rd2 C :: the delivery tiers and outcomes of the arm-C runs (a finding, not a pass condition) -- {('deliver', 'full-delivery'): 1, ('shortfall', 'delayed-start-recover'): 1, ('shortfall', 'full-delivery'): 2, ('shortfall', 'under-delivery'): 5}
[INFO] rd2 C :: the delivery tiers and outcomes of the arm-D runs (a finding, not a pass condition) -- {('deliver', 'full-delivery'): 6, ('shortfall', 'full-delivery'): 1, ('shortfall', 'no-build-lock-in'): 1, ('shortfall', 'under-delivery'): 4}
[INFO] rd2 C :: the delivery tiers and outcomes of the arm-E runs (a finding, not a pass condition) -- {('shortfall', 'delayed-start-recover'): 2, ('shortfall', 'under-delivery'): 6}
[INFO] rd2 C :: the delivery tiers and 

[PASS] rd2 C :: the capacity satisfies the mandate floor in every mandate run (36 map anchors + 30 solved anchors) -- no violation larger than 5 MW in any mandated year
[PASS] rd2 C :: the capacity equals the floor in each binding year (positive dual) -- every year with dual > $1/MW-yr has |slack| <= 5 MW
[PASS] rd2 C :: a positive dual occurs only in the binding years -- every year with slack > 100 MW has a zero dual
[PASS] rd2 C :: the duals table reproduces the step3_checks rows of the 18 smr100 solved anchors -- max |difference| 5.8e-11 over 288 rows
[PASS] rd2 C :: the dual conversion is correct on each mandate run's own pvf_onm (raw = cost_scale x pvf_onm x converted) -- max rel err 1.29e-07 across 66 runs


### 3.3 Cross-case invariance (phase D) and the t09 chain (phase E)

Phase D repeats the section 2.3 tests on the 93 runs and compares each capped run with its
control on the years through 2035 (`pair_identity.csv`), where the two offered paths are equal.
Phase E runs the t09 chain on the 30 solved anchors (`t09_required_itc_port_controls.csv`, the
port-fidelity reference against notebook 07) and on the 36 map anchors
(`t09_required_itc_port_anchors.csv`, notebook 10's input). A rate above 1 in a map anchor is
*censored*: the anchor needs more than a 100% credit, so its rate is a lower bound.

In [19]:
li_rd2 = load_invariance(B_RD2, CASES_RD2)
li_rd2.to_csv(F_RD2 / "load_invariance.csv", index=False)
rec("D", "the exogenous load is identical in all 93 runs and equal to the Step 3 reference load",
    "PASS" if li_rd2.max_rel_diff.max() < 1e-6 else "FAIL", f"max rel difference {li_rd2.max_rel_diff.max():.2e} vs {REF_LOAD}")
data_scans(rec, B_RD2, CASES_RD2, 93, len(UNION_RD2))
pair_rows = []
for c in AX_RD2:
    if META_RD2[c]["form"] != "capped_post2035":
        continue
    w, conv = META_RD2[c]["world"], META_RD2[c]["conv"]
    u = [x for x in AX_RD2 if META_RD2[x]["world"] == w and META_RD2[x]["conv"] == conv and META_RD2[x]["form"] == "exact"][0]
    sc_c = load(B_RD2, c, "systemcost").groupby("t")["Value"].sum()
    sc_u = load(B_RD2, u, "systemcost").groupby("t")["Value"].sum()
    for t in [y for y in YEARS_STD if 2031 <= y <= W_PRIMARY]:
        cap_c, cap_u = float(SMR_CAP_RD2[c].get(t, 0.0)), float(SMR_CAP_RD2[u].get(t, 0.0))
        pair_rows.append(dict(capped=c, control=u, world=w, convention=conv, t=t,
                              smr_cap_capped_MW=round(cap_c, 1), smr_cap_control_MW=round(cap_u, 1),
                              smr_cap_diff_MW=round(cap_c - cap_u, 1),
                              syscost_rel_diff=abs(float(sc_c.get(t, 0.0)) - float(sc_u.get(t, 0.0))) / max(abs(float(sc_u.get(t, 0.0))), 1.0)))
pid_rd2 = pd.DataFrame(pair_rows)
pid_rd2.to_csv(F_RD2 / "pair_identity.csv", index=False)
rec("D", "each capped run reproduces its control on the years through 2035, where the two offered paths are identical (SMR "
    "capacity, system cost) -- disclosed either way", "INFO",
    f"{pid_rd2['capped'].nunique()} pairs; max |SMR capacity difference| {pid_rd2.smr_cap_diff_MW.abs().max():.1f} MW; "
    f"max system-cost rel diff {pid_rd2.syscost_rel_diff.max():.2e}")
pre_smr = {c: float(load(B_RD2, c, "cap").query("i == 'nuclear-smr' and t < 2031")["Value"].sum()) for c in CASES_RD2}
rec("D", "no SMR capacity exists before 2031", "PASS" if max(pre_smr.values()) == 0 else "FAIL",
    f"max pre-2031 SMR capacity {max(pre_smr.values()):.1f} MW")

# --- phase E: the t09 chain on the 30 solved anchors and the 36 map anchors ----------------------
T09_IN_RD2 = t09_inputs(BATCH_OF_RD2, ANCHORS30 + MA_RD2, META_RD2)
t09_a30 = t09_table(duals_all_rd2, list(YEARS_STD), ANCHORS30, META_RD2, TRAJ_RD2, T09_IN_RD2)
t09_a30.to_csv(F_RD2 / "t09_required_itc_port_controls.csv", index=False)
t09_ma = t09_table(duals_all_rd2, list(YEARS_STD), MA_RD2, META_RD2, TRAJ_RD2, T09_IN_RD2)
t09_ma.to_csv(F_RD2 / "t09_required_itc_port_anchors.csv", index=False)
_c6 = t09_a30[t09_a30["case"].isin(CTRL_RD)].reset_index(drop=True)
pd.testing.assert_frame_equal(_c6, t09_ctrl_rd.reset_index(drop=True), check_dtype=False)   # same runs, same chain
n_rate = t09_ma[t09_ma["status"] == "rate"].groupby("case").size().reindex(MA_RD2).fillna(0)
rec("E", "every map anchor has at least one rate-status year and every rate row passed the dual-consistency assert inside the "
    "ported cell (S == dual x pvf_onm)", "PASS" if n_rate.min() >= 1 else "FAIL",
    f"rate rows per anchor: min {int(n_rate.min())}, max {int(n_rate.max())}; ladder {t09_ma.groupby('status').size().to_dict()}")
cens = t09_ma[t09_ma["itc_insufficient"].fillna(False).astype(bool)]
rec("E", "censored (itc_insufficient: rate >= 1.0, a lower bound) cells among the 36 map anchors", "INFO",
    f"{len(cens)} censored cells: " + "; ".join(f"{r.case} {int(r.t)} ({r.i_model_headline:.3f})" for r in cens.itertuples()))
_reg = pd.DataFrame(REG_RD2)
_parts = {"C4 duals": "capacity equals the floor", "C4b conversion": "dual conversion is correct",
          "B13c fingerprint": "shipped SMR plantchar files reproduce", "B1 design echo": "case matrix encodes"}
_stat = {k: _reg[_reg["check"].str.contains(v, regex=False)]["status"].tolist() for k, v in _parts.items()}
rec("E", "GA1': the 36 map anchors are audited (C4 dual audit, C4b conversion, B13c fingerprint, B1 design echo); the port "
    "fidelity half (GP') is the equality of t09_required_itc_port_controls.csv with notebook 07's t09 rows",
    "PASS" if all(s and all(x == "PASS" for x in s) for s in _stat.values()) else "FAIL", _stat)
added_scans(rec, B_RD2, CASES_RD2, 93)
print(t09_ma[t09_ma["status"] == "rate"].pivot_table(index="t", columns="case", values="i_model_headline").round(2).iloc[:, :6].to_string())
S_RD2 = write_registry(REG_RD2, F_RD2)

[PASS] rd2 D :: the exogenous load is identical in all 93 runs and equal to the Step 3 reference load -- max rel difference 0.00e+00 vs smr100_eia_p50


[PASS] rd2 D :: no data value is NaN or infinite in any file -- all 209 keys x 93 files scanned (key inventory)
[PASS] rd2 D :: no negative values occur where values must not be negative -- checked ['cap', 'cap_nat', 'load_rt', 'stor_in', 'stor_out', 'curt_ann', 'hours'] (storage charging and upgrades are legitimately negative elsewhere)
[PASS] rd2 D :: the nuclear build and generation data is not negative -- cap_new_ann and gen_ann nuclear rows >= 0, all 93 cases


[PASS] rd2 D :: the national totals equal the sum of the regional data -- cap, gen_ann, cap_new_ann, ret_ann; all 93 cases within 1e-3
[INFO] rd2 D :: each capped run reproduces its control on the years through 2035, where the two offered paths are identical (SMR capacity, system cost) -- disclosed either way -- 26 pairs; max |SMR capacity difference| 0.0 MW; max system-cost rel diff 0.00e+00


[PASS] rd2 D :: no SMR capacity exists before 2031 -- max pre-2031 SMR capacity 0.0 MW


[PASS] rd2 E :: every map anchor has at least one rate-status year and every rate row passed the dual-consistency assert inside the ported cell (S == dual x pvf_onm) -- rate rows per anchor: min 5, max 10; ladder {'rate': 326, 'slack: required ITC is zero': 4}
[INFO] rd2 E :: censored (itc_insufficient: rate >= 1.0, a lower bound) cells among the 36 map anchors -- 4 censored cells: ma_cop28_d8921 2050 (1.011); ma_eo_d9545 2050 (1.001); ma_eo_d243 2047 (1.029); ma_eo_d243 2050 (1.058)
[PASS] rd2 E :: GA1': the 36 map anchors are audited (C4 dual audit, C4b conversion, B13c fingerprint, B1 design echo); the port fidelity half (GP') is the equality of t09_required_itc_port_controls.csv with notebook 07's t09 rows -- {'C4 duals': ['PASS'], 'C4b conversion': ['PASS'], 'B13c fingerprint': ['PASS'], 'B1 design echo': ['PASS']}


[PASS] rd2 X :: no load is dropped after 2025 (the step3_checks E3 rule) -- 93 runs; the historical 2010-2023 artifact 2.96 TWh (case spread 0.0 MWh)


[INFO] rd2 X :: national curtailment in 2050 (context) -- 1.8 to 33.8 TWh (median 20.1) over 93 runs
case  ma_aj_d2349  ma_aj_d3363  ma_aj_d4243  ma_aj_d479  ma_aj_d8157  ma_aj_d9927
t                                                                                
2031         0.81         0.46         0.47        0.68         0.36         0.73
2032         0.62         0.20         0.24        0.46         0.19         0.55
2033         0.86         0.46         0.50        0.73         0.46         0.82
2034         0.84         0.44         0.50        0.66         0.45         0.80
2035         0.76         0.41         0.49        0.63         0.44         0.74
2038         0.78         0.37         0.53        0.65         0.44         0.80
2041         0.68         0.20         0.45        0.53         0.34         0.80
2044         0.68         0.19         0.49        0.54         0.37         0.80
2047         0.67         0.18         0.50        0.53         0.38         0.

## 4. `batch2`: the statutory credit, its learning-on controls, and the map-margin runs

**What the batch tests.** Every run is fbC (mandate OFF, learning ON with the sample's drawn
parameters). Block S asks what today's credit builds: in the 24 solved anchor samples (the 5th,
25th, 50th and 75th cost percentiles of each schedule) a flat 50% credit in every model year
(`bx_*_f50`; 50% is the stacked statutory ceiling: 30% base plus the energy-community and
domestic-content bonuses) and a grant form that pays the exact required rate through 2038, then 50%
(`bx_*_g50`). Block K holds the learning-on controls (`bx_*_u_lrn`): the exact required rate in
every rate-status year, with learning ON. Three S samples reuse a ratedesign2 control
(`ax_aj_p50_u_lrn`, `ax_cop28_p75_u`, `ax_eo_p75_u`, block K6 here). Block R1 (`bm_*_e1/_e2`) offers
the requirement map's rate plus 10 or 20 points in ten fresh samples.

**Why the controls matter (SF15).** A statute run is read against its learning-on control. If the
control, which offers the exact required rate, does not deliver, the sample is *uninformative*
for the statute reading: the learning feedback, not the credit, decides the outcome. Notebook 12
recomputes this "informative" flag from `control_outcomes.csv` (one row per S sample) with the
registered adjudication rule (a K control counts as delivering if it delivers, or if its largest
gated shortfall is below one 300 MW unit, or, for the eia schedule, if it is a delayed start that
recovers; a K6 control counts if it delivers). This notebook applies the same rule in the column
`control_counts_as_delivering` so the two can be compared.

**Tables.** `delivery_outcomes.csv` (89 runs + the 3 K6 controls, joined with the m50 spec),
`trajectory_gaps.csv` (with the v2.5 `gated` flag and `in_window` = year <= 2038),
`itc_payments.csv` (the nuclear ITC line of the system cost by year, 2004 $), `fingerprint_errors.csv`
(the in-run SMR cost path: learning feedback), `control_outcomes.csv`, `pair_identity.csv` (each
grants50 run against its control through 2038) and the sanity outputs.

In [20]:
# ---- batch2: case matrix, registration, metadata -------------------------------------------------
F_B2 = outdir("rate_design2_b2_checks")
REG_B2 = []
rec = make_record(REG_B2, "b2")
B_B2 = "batch2"
cases_b2 = casefile("batch2")
CASES_SPEC_B2 = [c for c in cases_b2.columns if c != "Default Value"]
assert len(CASES_SPEC_B2) == 89, f"the batch2 casefile has 89 runs, found {len(CASES_SPEC_B2)}"
MISSING_B2 = sorted(c for c in CASES_SPEC_B2 if not returned(B_B2, c))
assert MISSING_B2 == [], f"every batch-2 run returned (bm_cop28_d9396_e2 came back with batch 3): {MISSING_B2}"
CASES_B2 = [c for c in CASES_SPEC_B2 if c not in MISSING_B2]
CTRL_B6 = ["ax_aj_p50_u_lrn", "ax_cop28_p75_u", "ax_eo_p75_u"]    # ratedesign2 learning-on controls (references)
m50 = pd.read_csv(REG_DIR / "batch2" / "m50_b2_run_spec.csv").set_index("run")
m51 = pd.read_csv(REG_DIR / "batch2" / "m51_b2_offers.csv")
m41b = pd.read_csv(REG_DIR / "batch2" / "m41b_b21_offers.csv")
assert set(m50.index) == set(CASES_SPEC_B2) and set(m51["run"]) == set(CASES_SPEC_B2)
S_B2 = [c for c in CASES_B2 if m50.loc[c, "block"] == "S"]
K_B2 = [c for c in CASES_B2 if m50.loc[c, "block"] == "K"]
R_B2 = [c for c in CASES_B2 if m50.loc[c, "block"] == "R1"]
assert [len(S_B2), len(K_B2), len(R_B2)] == [48, 21, 20] and (m50["convention"] == "fbC").all()
BLOCK_B2 = {c: m50.loc[c, "block"] for c in CASES_B2}
sw_b2 = make_sw([cases_b2, cases_rd2, cases_rd, cases_s100], cases_b2)
META_B2 = {c: meta_of(sw_b2, c) for c in CASES_B2 + CTRL_B6 + [REF_LOAD]}
for c in CASES_B2:
    s = m50.loc[c]
    META_B2[c].update(form=s["form"], world=s["world"], world_case=s["world_case"], pct=s["pct"],
                      draw_index=int(s["draw_index"]), gate=s["gate"], window=int(s["window"]),
                      control_run=s["control_run"], prediction=s["prediction"])
BATCH_OF_B2 = {**{c: B_B2 for c in CASES_B2}, **{c: B_RD2 for c in CTRL_B6}}
OFFER_B2 = {r: g.set_index("year")["offer_rate"].to_dict() for r, g in m51.groupby("run")}
STATUS_B2 = {r: g.set_index("year")["status"].to_dict() for r, g in m51.groupby("run")}
for r, g in m04[m04["run"].isin(CTRL_B6)].groupby("run"):
    OFFER_B2[r] = g.set_index("year")["offer_rate"].to_dict()
RATES_B2 = {r: {int(t): round(float(v), 3) for t, v in OFFER_B2[r].items()} for r in CASES_B2 + CTRL_B6}
M_EXP_B2 = {r: {t: f * (1.0 - PEN) for t, f in RATES_B2[r].items()} for r in CASES_B2 + CTRL_B6}
W_S = 2038          # the block-S window, inclusive (methods v2.5)
POST_S = [t for t in MODEL_YEARS if t > W_S]
CAP_S = 0.50
print(f"batch2: {len(CASES_B2)} returned of {len(CASES_SPEC_B2)} = {len(S_B2)} S + {len(K_B2)} K + {len(R_B2)} R1; "
      f"ratedesign2 controls read as references: {CTRL_B6}")

batch2: 89 returned of 89 = 48 S + 21 K + 20 R1; ratedesign2 controls read as references: ['ax_aj_p50_u_lrn', 'ax_cop28_p75_u', 'ax_eo_p75_u']


### 4.1 Solve health and design echo (phases A and B)

As in section 2.1, with the batch-2 rules: every run is fbC with the three learning channels ON, its
own incentives file and the `rd_` sample files; no run carries a dual key (no mandate anywhere).
Test B2b checks the registered form rules: flat50 = 0.50 on the ten model years; grants50 = the
exact rate through 2038 (equal to the control's offer and to the t09 rate of the solved anchor,
section 3.3), then 0.50 = flat50; exact = the t09 rate-status years; map_plus = the m41b path,
at most 0.95, with two 2031 cells clipped at 0.950.

In [21]:
missing = [c for c in CASES_B2 if not returned(B_B2, c)]
rec("A", "one output file exists for each of the 89 returned cases", "PASS" if not missing else "FAIL",
    f"missing: {missing}" if missing else "89 runs in the key inventory")
rec("A", "registered runs with no output file (carried as 'not returned'; nothing is inferred)", "INFO",
    f"{MISSING_B2 or 'none'}: bm_cop28_d9396_e2 was absent from the 2026-09-14 delivery and returned with batch 3 on "
    "2026-09-18 (archived under the batch2_ stem)")
ref_missing = [c for c in CTRL_B6 if not returned(B_RD2, c)] + ([] if returned("smr100", REF_LOAD) else [REF_LOAD]) \
    + ([] if returned("step4", REF4) else [REF4])
rec("A", "the reference files are available (3 batch-6 learning-on controls, the Step 3 load reference, the Step 4 key-vintage "
    "reference)", "PASS" if not ref_missing else "FAIL",
    f"missing: {ref_missing}" if ref_missing else "ratedesign2 ax_* x3, smr100_eia_p50, step4 reference present")
KS_B2, UNION_B2, PVF_B2, CS_B2 = solve_health(
    rec, B_B2, CASES_B2,
    years_of=lambda c: (YEARS_STD, META_B2[c]["yearset"] == YEARS_STD and META_B2[c]["endyear"] == 2050),
    key_labels=("all 89 files have the same data keys",
                "the key set agrees with the Step 4 vintage (separate-delivery drift detector; dual keys excluded)"),
    key_exclude=frozenset())

mx_bad = []
for c in CASES_B2:
    m_, ab = META_B2[c], sched_of(c)
    tok, w = STEM2TOK[ab], m_["draw_index"]
    if BLOCK_B2[c] == "S":
        name_ok = c == f"bx_{ab}_{m_['pct']}_" + {"flat50": "f50", "grants50": "g50"}[m_["form"]] and m_["world"] == f"{ab}_{m_['pct']}"
    elif BLOCK_B2[c] == "K":
        name_ok = c == f"bx_{ab}_{m_['pct']}_u_lrn" and m_["form"] == "exact" and m_["world"] == f"{ab}_{m_['pct']}"
    else:
        name_ok = c == f"bm_{ab}_d{w}_" + {"map_plus_1e6": "e1", "map_plus_2e6": "e2"}[m_["form"]] and m_["world"] == f"{ab}_d{w}"
    ok = (m_["scen"] == f"{tok}_smr" and m_["endyear"] == 2050 and m_["techscen"] == "smr" and m_["yearset"] == YEARS_STD and name_ok
          and m_["mode"] == 0 and m_["learning"] == 1 and m_["suffix"] == f"obbba_rd_{c}" and m_["foreign"] == f"rd_{ab}_d{w}"
          and m_["world_case"] == f"rd_{ab}_d{w}" and m_["pc"]["nuclear-smr"] == f"nuclear-smr_mc_rd_{ab}_d{w}"
          and m_["pc"]["nuclear"] == f"nuclear_mc_rd_{ab}_d{w}" and m_["fin"] == f"mc_rd_{ab}_d{w}" and m_["constr"] == f"mc_rd_{ab}_d{w}"
          and m_["tcx"] == "1" and sw_b2(c, "GSw_NuclearLearning_OCC") == "1" and sw_b2(c, "GSw_NuclearLearning_Duration") == "1"
          and sw_b2(c, "GSw_NuclearLearning_CrossTech") == "1")
    if not ok:
        mx_bad.append(c)
rec("B", "the case matrix encodes the intended block design (89 fbC runs: mandate OFF, learning ON with the three channels, "
    "per-run incentives file, rd_ world files + foreign file, phaseout exemption; block naming)",
    "PASS" if not mx_bad else "FAIL", mx_bad or "89 cases match the fbC rules and the block naming conventions")
b1b_bad = []
for c in CASES_B2:
    ab, w = sched_of(c), META_B2[c]["draw_index"]
    world = Z[f"worlds_{STEM2TOK[ab]}"][w]
    for col, name in SWITCH_OF.items():
        exp, got = float(world[WCOLS.index(col)]), float(sw_b2(c, name))
        if abs(got - exp) > max(1e-5 * abs(exp), 5e-6):
            b1b_bad.append((c, col, got, exp))
rec("B", "the learning switches equal the drawn world parameters at the m50 draw index",
    "PASS" if not b1b_bad else "FAIL", b1b_bad[:5] or f"89 runs x {len(SWITCH_OF)} switches match mc_perdraw.npz to 1e-5")

echo_rows, b2_bad = [], []
for r in CASES_B2:
    got, flags = incentives_rows(r, f"obbba_rd_{r}", RATES_B2[r])
    if not all(flags.values()):
        b2_bad.append((r, flags))
    for t in sorted(set(got) | set(RATES_B2[r])):
        echo_rows.append(dict(run=r, block=BLOCK_B2[r], form=META_B2[r]["form"], sched=sched_of(r), t=t,
                              status=STATUS_B2[r].get(t, ""), itc_frac_file=got.get(t), itc_frac_m51=RATES_B2[r].get(t)))
pd.DataFrame(echo_rows).to_csv(F_B2 / "incentives_echo.csv", index=False)
rec("B", "the 89 fed incentives files equal the m51 credit paths at exactly the offered years (offer_rate at 3 decimals)",
    "PASS" if not b2_bad else "FAIL",
    b2_bad[:3] or "89 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0; every rate below 1.0")

# B2b: the registered form rules; the exact paths come from the t09 chain on the 30 solved anchors (section 3.3)
t09r = t09_a30[t09_a30["status"] == "rate"].set_index(["case", "t"])["i_model_headline"]
def exact_path(world_case):
    return {int(t): round(float(v), 3) for (c, t), v in t09r.items() if c == world_case}
form_bad = []
CLIPPED = {"bm_cop28_d8291_e2": [2031], "bm_eo_d2024_e2": [2031]}
for r in CASES_B2:
    f, b = META_B2[r]["form"], RATES_B2[r]
    if f == "flat50":
        ok = sorted(b) == MODEL_YEARS and all(v == CAP_S for v in b.values())
    elif f == "grants50":
        ctrl, f50 = RATES_B2[META_B2[r]["control_run"]], RATES_B2[r[:-4] + "_f50"]
        ex = exact_path(f"smr100_{META_B2[r]['world']}")
        ok = (sorted(t for t in b if t <= W_S) == sorted(t for t in ex if t <= W_S)
              and all(b[t] == ex[t] == ctrl[t] for t in b if t <= W_S)
              and sorted(t for t in b if t > W_S) == POST_S and all(b[t] == CAP_S == f50[t] for t in b if t > W_S))
    elif f == "exact":
        ok = b == exact_path(f"smr100_{META_B2[r]['world']}")
    else:
        mb = m41b[m41b["run"] == r].set_index("year")["offer_rate"].round(3).to_dict()
        ok = (b == {int(t): float(v) for t, v in mb.items()} and max(b.values()) <= 0.95
              and all(b[t] == 0.95 for t in CLIPPED.get(r, [])) and all(b[t] < 0.95 for t in b if t not in CLIPPED.get(r, [])))
    if not ok:
        form_bad.append((r, f))
rec("B", "the offered paths carry the registered form rules and pair identities (flat50 = 0.50 on the ten model years; grants50 = "
    "the frozen exact rate = the control through 2038, then 0.50 = flat50; exact = the frozen t09 rate-status years; map_plus = "
    "m41b, <= 0.95, the two clipped 2031 cells at 0.950)", "PASS" if not form_bad else "FAIL",
    form_bad[:5] or f"{len(S_B2)//2} flat50, {len(S_B2)//2} grants50, {len(K_B2)} exact controls, {len(R_B2)} map_plus runs (clipped: {CLIPPED})")

rr_b2 = pd.DataFrame([row for c in CASES_B2 for row in rate_recovery_rows(
    B_B2, c, META_B2[c]["fin"], M_EXP_B2[c], dict(block=BLOCK_B2[c], form=META_B2[c]["form"], sched=sched_of(c)))])
rr_b2.to_csv(F_B2 / "rate_recovery.csv", index=False)
rate_landing_checks(rec, rr_b2, OFFER_B2, (
    "the recovered monetized rate equals 0.9 x the fed rate at exactly the offered years (89 runs; the construction multiplier "
    "cancels in the ratio)",
    "the recovered rate is zero outside the offered years and for the large reactor in all 89 runs",
    "the recovered rate is uniform across regions",
    "the phaseout does not reduce the nuclear credit in any year (exemption held)"))

[PASS] b2 A :: one output file exists for each of the 89 returned cases -- 89 runs in the key inventory
[INFO] b2 A :: registered runs with no output file (carried as 'not returned'; nothing is inferred) -- none: bm_cop28_d9396_e2 was absent from the 2026-09-14 delivery and returned with batch 3 on 2026-09-18 (archived under the batch2_ stem)
[PASS] b2 A :: the reference files are available (3 batch-6 learning-on controls, the Step 3 load reference, the Step 4 key-vintage reference) -- ratedesign2 ax_* x3, smr100_eia_p50, step4 reference present
[PASS] b2 A :: all 89 files have the same data keys -- key counts [207]
[PASS] b2 A :: the key set agrees with the Step 4 vintage (separate-delivery drift detector; dual keys excluded) -- 209 keys, no drift
[PASS] b2 A :: the solver residuals are small in every file -- worst |residual| 7.53e-04 at bx_aj_p75_g50


[PASS] b2 A :: the objective value is a normal positive number in every file -- range 1.3178e+12 to 1.6380e+12


[PASS] b2 A :: the model years are the standard 16-year set with end year 2050 in every file -- every file on its registered year set


[PASS] b2 A :: every run used the sequential solve mode -- pvf_capital==1 (max dev 0.0e+00); pvf_onm flat from 2026 (max dev 9.5e-07); cost_scale==1; z_rep has one row per model year
[PASS] b2 B :: the case matrix encodes the intended block design (89 fbC runs: mandate OFF, learning ON with the three channels, per-run incentives file, rd_ world files + foreign file, phaseout exemption; block naming) -- 89 cases match the fbC rules and the block naming conventions


[PASS] b2 B :: the learning switches equal the drawn world parameters at the m50 draw index -- 89 runs x 11 switches match mc_perdraw.npz to 1e-5


[PASS] b2 B :: the 89 fed incentives files equal the m51 credit paths at exactly the offered years (offer_rate at 3 decimals) -- 89 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0; every rate below 1.0
[PASS] b2 B :: the offered paths carry the registered form rules and pair identities (flat50 = 0.50 on the ten model years; grants50 = the frozen exact rate = the control through 2038, then 0.50 = flat50; exact = the frozen t09 rate-status years; map_plus = m41b, <= 0.95, the two clipped 2031 cells at 0.950) -- 24 flat50, 24 grants50, 21 exact controls, 20 map_plus runs (clipped: {'bm_cop28_d8291_e2': [2031], 'bm_eo_d2024_e2': [2031]})


[PASS] b2 B :: the recovered monetized rate equals 0.9 x the fed rate at exactly the offered years (89 runs; the construction multiplier cancels in the ratio) -- 829 fed case-years; worst |dm| 0.0001 at (bx_eia_p05_f50, 2035)
[PASS] b2 B :: the recovered rate is zero outside the offered years and for the large reactor in all 89 runs -- 1307 points; max |m| 0.0000
[PASS] b2 B :: the recovered rate is uniform across regions -- max spread over regions 0.00123
[PASS] b2 B :: the phaseout does not reduce the nuclear credit in any year (exemption held) -- 444 credited case-years at t >= 2036; worst |dm| 0.0001


### 4.2 ITC outlay and cost fingerprints

`itc_payments.csv` holds, per run and year, the sum of the nuclear rows of the system-cost streams
whose name contains "itc" (2004 $; negative = a payment by the government). Notebook 12 turns it
into the outlay table (m66). The fingerprint table compares each run's in-run SMR capital cost with
its drawn input path: equal before the 2031 anchor, and below it afterwards where learning acts.

In [22]:
SMR_NEW = {c: float(load(B_B2, c, "cap_new_ann").query("i == 'nuclear-smr' and t > 2030")["Value"].sum()) for c in CASES_B2}
pay_bad, no_build, ITC_PAY = [], [], {}
for c in CASES_B2:
    sc = load(B_B2, c, "systemcost_techba")
    it = sc[sc["sys_costs"].str.contains("itc", case=False) & sc["i"].isin(NUC)]
    ITC_PAY[c] = it.groupby("t")["Value"].sum()
    if SMR_NEW[c] > 100.0 and float(it["Value"].abs().sum()) < 1e6:
        pay_bad.append((c, f"builds {SMR_NEW[c]:,.0f} MW but no nuclear ITC payments"))
    if SMR_NEW[c] <= 100.0:
        no_build.append(c)
rec("B", "the system cost data shows nuclear ITC payments wherever SMR builds occurred", "PASS" if not pay_bad else "FAIL",
    pay_bad or f"payments present in every case with builds; cases with ~no builds (lock-in candidates, read in phase C): {no_build or 'none'}")
itc_pay_b2 = pd.concat([s.rename("itc_pay_2004").reset_index().assign(case=c) for c, s in ITC_PAY.items()],
                       ignore_index=True)[["case", "t", "itc_pay_2004"]]
itc_pay_b2.to_csv(F_B2 / "itc_payments.csv", index=False)
itc_mag = {c: abs(float(load(B_B2, c, "tax_expenditure_itc")["Value"].sum())) for c in CASES_B2}
rec("B", "the other technologies keep their ITC in all 89 runs", "PASS" if min(itc_mag.values()) > 1e8 else "FAIL",
    f"total ITC tax expenditure magnitude {min(itc_mag.values())/1e9:.1f} to {max(itc_mag.values())/1e9:.1f} B$ per case")

fp_b2 = pd.DataFrame([row for c in CASES_B2 for row in fingerprint_rows(B_B2, c, META_B2[c], dict(block=BLOCK_B2[c]))])
fp_b2.to_csv(F_B2 / "fingerprint_errors.csv", index=False)
pre_fp = fp_b2[fp_b2["t"] <= 2030]
rec("B", "the 89 learning-on runs use the drawn cost path before the 2031 anchor",
    "PASS" if pre_fp.rel_err.max() < 1e-4 else "FAIL", f"{len(pre_fp)} points; max rel err {pre_fp.rel_err.max():.2e}")
on_fp = fp_b2[(fp_b2["t"] >= 2031) & (fp_b2["i"] == "nuclear-smr")]
dev = on_fp.groupby("case")["rel_err"].max().round(3)
rec("B", "the learning-on runs write an in-run SMR cost path (max deviation from the drawn path, 2031-2050; context, read by the "
    "verdict notebook as the learning feedback)", "INFO",
    f"median over runs {dev.median():.3f}; range {dev.min():.3f}-{dev.max():.3f}; by block: {on_fp.groupby('block')['rel_err'].max().round(3).to_dict()}")
map_bad, pre_dev = [], []
for r in CASES_B2:
    pc_df = plantchar(META_B2[r]["pc"]["nuclear-smr"])
    for t, occ in m51[m51["run"] == r].set_index("year")["occ_world_kW"].items():
        got = float(pc_df.loc[t, "capcost"]) * D2224
        if abs(got - occ) / occ > 1e-3:
            map_bad.append((r, t, round(got, 1), round(occ, 1)))
for c in R_B2:
    ab, w = sched_of(c), META_B2[c]["draw_index"]
    occ_npz = Z[f"occ_{STEM2TOK[ab]}_smr"][w]
    pc_df = plantchar(META_B2[c]["pc"]["nuclear-smr"])
    for j, t in enumerate(NPZ_YEARS):
        if t not in pc_df.index:
            continue
        rel = abs(float(pc_df.loc[t, "capcost"]) - occ_npz[j]) / occ_npz[j]
        if t >= 2031 and rel > 1e-3:
            map_bad.append((c, t))
        if t < 2031:
            pre_dev.append(rel)
rec("B", "the shipped SMR plantchar files reproduce the offline cost worlds (all 89 runs: m51 occ_world_kW in 2024$ at the offered "
    "years; the 20 R1 runs: the mc_perdraw.npz row of the m50 draw in native 2022$, 2031-2050)", "PASS" if not map_bad else "FAIL",
    map_bad[:5] or f"{len(m51[m51['run'].isin(CASES_B2)])} offered-year points and 10 fresh worlds x 20 years within 1e-3")
rec("B", "before the 2031 anchor the R1 plantchar files carry the pre-anchor path while the npz holds the flat BOAK value (outside "
    "the drawn map; context)", "INFO", f"10 fresh worlds x 7 pre-2031 years; max rel difference {max(pre_dev):.2f}")
b15_bad = sorted(c for c in CASES_B2 if KS_B2[c] & DUAL_KEYS)
rec("B", "no batch-2 file carries a dual key (mandate OFF everywhere)", "PASS" if not b15_bad else "FAIL",
    b15_bad or "nuclear_cap_price absent from all 89 files")
TRAJ_B2 = {c: trajectory(META_B2[c]["scen"]) for c in CASES_B2 + CTRL_B6}
b16_bad = [c for c in CASES_B2 + CTRL_B6 if META_B2[c]["scen"] != f"{STEM2TOK[sched_of(c)]}_smr"]
rec("B", "every run reads its schedule's standard mandate trajectory file (no _ext token)",
    "PASS" if not b16_bad else "FAIL", b16_bad or "89 runs + 3 reference controls on the six standard files")

[PASS] b2 B :: the system cost data shows nuclear ITC payments wherever SMR builds occurred -- payments present in every case with builds; cases with ~no builds (lock-in candidates, read in phase C): ['bx_iaea_p50_f50', 'bx_cop28_p50_f50', 'bx_eia_p75_f50', 'bx_aj_p75_f50', 'bx_iaea_p75_f50', 'bx_mck_p75_f50', 'bx_cop28_p75_f50', 'bx_eo_p75_f50']
[PASS] b2 B :: the other technologies keep their ITC in all 89 runs -- total ITC tax expenditure magnitude 124.8 to 1647.0 B$ per case


[PASS] b2 B :: the 89 learning-on runs use the drawn cost path before the 2031 anchor -- 3738 points; max rel err 1.39e-07
[INFO] b2 B :: the learning-on runs write an in-run SMR cost path (max deviation from the drawn path, 2031-2050; context, read by the verdict notebook as the learning feedback) -- median over runs 0.304; range 0.077-1.989; by block: {'K': 0.491, 'R1': 0.477, 'S': 1.989}


[PASS] b2 B :: the shipped SMR plantchar files reproduce the offline cost worlds (all 89 runs: m51 occ_world_kW in 2024$ at the offered years; the 20 R1 runs: the mc_perdraw.npz row of the m50 draw in native 2022$, 2031-2050) -- 829 offered-year points and 10 fresh worlds x 20 years within 1e-3
[INFO] b2 B :: before the 2031 anchor the R1 plantchar files carry the pre-anchor path while the npz holds the flat BOAK value (outside the drawn map; context) -- 10 fresh worlds x 7 pre-2031 years; max rel difference 0.38
[PASS] b2 B :: no batch-2 file carries a dual key (mandate OFF everywhere) -- nuclear_cap_price absent from all 89 files


[PASS] b2 B :: every run reads its schedule's standard mandate trajectory file (no _ext token) -- 89 runs + 3 reference controls on the six standard files


### 4.3 Delivery outcomes and the control table (phase C)

The gap table carries the v2.5 gate flag (`gated`) and the window flag (`in_window`, year <= 2038).
Two outcome columns are specific to batch 2: `first_shortfall_year` (the first gated year whose
shortfall exceeds the tier tolerance; 0 = none) and `in_window_shortfall` (such a year at or before
2038). The three K6 controls are classified with the same code; their outcomes must equal section 3.

In [23]:
SMR_CAP_B2 = {c: smr_cap(BATCH_OF_B2[c], c) for c in CASES_B2 + CTRL_B6}
LARGE_NEW_B2 = {c: load(B_B2, c, "cap_new_ann").query("i == 'nuclear' and t > 2030").groupby("t")["Value"].sum() for c in CASES_B2}
FORM_OF = {**{c: META_B2[c]["form"] for c in CASES_B2}, **{c: "exact" for c in CTRL_B6}}
BLOCK_OF = {**BLOCK_B2, **{c: "K6" for c in CTRL_B6}}

def is_gated_v25(c, t, req):
    if FORM_OF[c] in ("flat50", "grants50"):
        return (t in MODEL_YEARS and req > 0) or t == 2050
    return (t in OFFER_B2[c]) or t == 2050

gap_rows = []
for c in CASES_B2 + CTRL_B6:
    for t in YEARS_STD:
        req = float(TRAJ_B2[c].get(t, 0.0))
        if req <= 0:
            continue
        have = float(SMR_CAP_B2[c].get(t, 0.0))
        status = STATUS_B2[c].get(t, "unoffered") if c in STATUS_B2 else ("rate" if t in OFFER_B2[c] else "slack")
        gap_rows.append(dict(case=c, block=BLOCK_OF[c], form=FORM_OF[c], sched=sched_of(c), t=t, status=status,
                             gated=is_gated_v25(c, t, req), in_window=t <= W_S, mandate_MW=req, cap_MW=round(have, 1),
                             gap_MW=round(have - req, 1), gap_pct=round(100 * (have - req) / req, 2)))
gaps_b2 = pd.DataFrame(gap_rows)
gaps_b2.to_csv(F_B2 / "trajectory_gaps.csv", index=False)

out_rows = []
for c in CASES_B2 + CTRL_B6:
    sub = gaps_b2[gaps_b2["case"] == c]
    tier = tier_of(sub, sub["gated"])
    cap_max = float(SMR_CAP_B2[c].max()) if len(SMR_CAP_B2[c]) else 0.0
    outcome, misses, gmax, end_req = outcome_of(sub, tier, cap_max, sub["gated"], sub["gated"])
    builds = SMR_CAP_B2[c][SMR_CAP_B2[c] > 5.0]
    delivers = tier in ("exact", "reproduces")
    if c in CTRL_B6:
        w6 = c.split("_")[1] + "_" + c.split("_")[2]
        s6 = m50[(m50["world"] == w6) & (m50["form"] == "flat50")].iloc[0]
        s = dict(block="K6", gate="GC-lrn", world=w6, pct=c.split("_")[2], draw_index=int(s6["draw_index"]), cap=np.nan,
                 form="exact", excess=s6["excess"], max_post_rate=s6["max_post_rate"], max_in_rate=s6["max_in_rate"],
                 excess_in=s6["excess_in"], eligible=True, prediction="deliver", window=W_S, control_run=c, margin_pts=np.nan)
    else:
        s = m50.loc[c].to_dict()
        s["block"] = BLOCK_B2[c]
    out_rows.append(dict(case=c, block=s["block"], convention="fbC", gate=s["gate"], sched=sched_of(c), world=s["world"],
                         pct=s["pct"], draw_index=int(s["draw_index"]), cap=s["cap"], form=s["form"], window=int(s["window"]),
                         excess=s["excess"], max_post_rate=s["max_post_rate"], max_in_rate=s["max_in_rate"],
                         excess_in=s["excess_in"], margin_pts=s["margin_pts"], eligible=bool(s["eligible"]),
                         prediction=s["prediction"], control_run=s["control_run"], tier=tier, outcome=outcome,
                         delivers=delivers, prediction_met=bool(delivers == (s["prediction"] == "deliver")),
                         miss_years="|".join(str(t) for t in misses), first_shortfall_year=(min(misses) if misses else 0),
                         in_window_shortfall=bool(any(t <= W_S for t in misses)), max_gated_shortfall_MW=round(gmax, 1),
                         first_build_year=int(builds.index.min()) if len(builds) else 0,
                         smr_2038_MW=round(float(SMR_CAP_B2[c].get(W_S, 0.0)), 1), traj_2038_MW=float(TRAJ_B2[c].get(W_S, 0.0)),
                         smr_2050_MW=round(float(SMR_CAP_B2[c].get(2050, 0.0)), 1), traj_2050_MW=end_req,
                         delivered_frac_2050=round(float(SMR_CAP_B2[c].get(2050, 0.0)) / end_req, 4)))
outc_b2 = pd.DataFrame(out_rows)
outc_b2.to_csv(F_B2 / "delivery_outcomes.csv", index=False)
assert len(outc_b2) == 92, "89 batch-2 runs + the 3 ratedesign2 controls"
for blk, lab in [("S", "block-S statute runs"), ("K", "block-K learning-on controls"), ("R1", "block-R1 map-margin runs"),
                 ("K6", "the three batch-6 controls, re-classified here as references")]:
    rec("C", f"the delivery tiers and outcomes of the {lab} (a finding, not a pass condition)", "INFO",
        outc_b2[outc_b2["block"] == blk].groupby(["form", "prediction", "outcome"])["case"].count().to_dict())
b6 = outc_rd2.set_index("case")
same = all(b6.loc[c, "outcome"] == outc_b2.set_index("case").loc[c, "outcome"] for c in CTRL_B6)
rec("C", "the three batch-6 controls classify to the same outcome as in rate_design2_checks (same code, the exact form's gate "
    "years are unchanged)", "PASS" if same else "FAIL", {c: b6.loc[c, "outcome"] for c in CTRL_B6})
rec("C", "the pre-registered outcome classification of all 89 returned runs (prediction met?)", "INFO",
    outc_b2[outc_b2["block"] != "K6"].groupby(["gate", "prediction_met"])["case"].count().to_dict())
sub_lg = {c: round(float(LARGE_NEW_B2[c].sum()) / 1000.0, 2) for c in CASES_B2 if float(LARGE_NEW_B2[c].sum()) > 0}
rec("C", "large-reactor substitution in the batch-2 runs (a report, not a failure)", "INFO",
    sub_lg or "no new large builds after 2030 in any of the 89 runs")
print(outc_b2.groupby(["block", "form", "outcome"]).size().to_string())

[INFO] b2 C :: the delivery tiers and outcomes of the block-S statute runs (a finding, not a pass condition) -- {('flat50', 'deliver', 'delayed-start-recover'): 5, ('flat50', 'deliver', 'full-delivery'): 2, ('flat50', 'deliver', 'under-delivery'): 3, ('flat50', 'shortfall', 'no-build-lock-in'): 8, ('flat50', 'shortfall', 'under-delivery'): 6, ('grants50', 'deliver', 'delayed-start-recover'): 2, ('grants50', 'deliver', 'full-delivery'): 8, ('grants50', 'shortfall', 'full-delivery'): 7, ('grants50', 'shortfall', 'partial'): 1, ('grants50', 'shortfall', 'under-delivery'): 6}
[INFO] b2 C :: the delivery tiers and outcomes of the block-K learning-on controls (a finding, not a pass condition) -- {('exact', 'deliver', 'delayed-start-recover'): 1, ('exact', 'deliver', 'full-delivery'): 18, ('exact', 'deliver', 'under-delivery'): 2}
[INFO] b2 C :: the delivery tiers and outcomes of the block-R1 map-margin runs (a finding, not a pass condition) -- {('map_plus_1e6', 'deliver', 'delayed-start-reco

In [24]:
# --- the control run of every block-S sample (SF15 informative flags; notebook 12 recomputes) -----------
UNIT_MW = 300.0             # one SMR unit (the lumpiness clause of the adjudication)
O2 = outc_b2.set_index("case")

def control_counts(c):
    r = O2.loc[c]
    if r["block"] == "K6":
        return bool(r["delivers"])                 # the batch-6 controls count by tier
    if bool(r["delivers"]):
        return True
    if r["prediction"] == "deliver":
        if float(r["max_gated_shortfall_MW"]) < UNIT_MW:
            return True
        if r["sched"] == "eia" and r["outcome"] == "delayed-start-recover":
            return True
    return False

ctrl_rows = []
for ab in SCHEDULES:
    for p in ["p05", "p25", "p50", "p75"]:
        w = f"{ab}_{p}"
        cr = m50[(m50["world"] == w) & (m50["form"] == "flat50")]["control_run"].iloc[0]
        r = O2.loc[cr]
        ctrl_rows.append(dict(world=w, schedule=ab, pct=p, flat50_run=f"bx_{w}_f50", grants50_run=f"bx_{w}_g50",
                              control_run=cr, control_batch=BATCH_OF_B2[cr], control_block=r["block"], tier=r["tier"],
                              outcome=r["outcome"], delivers=bool(r["delivers"]), prediction=r["prediction"],
                              max_gated_shortfall_MW=r["max_gated_shortfall_MW"], delivered_frac_2050=r["delivered_frac_2050"],
                              control_counts_as_delivering=control_counts(cr)))
ctrl_b2 = pd.DataFrame(ctrl_rows)
ctrl_b2["informative"] = ctrl_b2["control_counts_as_delivering"]
ctrl_b2.to_csv(F_B2 / "control_outcomes.csv", index=False)
assert len(ctrl_b2) == 24 and ctrl_b2["control_run"].isin(K_B2 + CTRL_B6).all()
print(f"controls that count as delivering: {int(ctrl_b2['informative'].sum())} of 24; uninformative samples: "
      f"{sorted(ctrl_b2.loc[~ctrl_b2['informative'], 'world'])}")

controls that count as delivering: 22 of 24; uninformative samples: ['eia_p05', 'eia_p25']


In [25]:
li_b2 = load_invariance(B_B2, CASES_B2)
li_b2.to_csv(F_B2 / "load_invariance.csv", index=False)
rec("D", "the exogenous load is identical in all 89 runs and equal to the Step 3 reference load",
    "PASS" if li_b2.max_rel_diff.max() < 1e-6 else "FAIL", f"max rel difference {li_b2.max_rel_diff.max():.2e} vs {REF_LOAD}")
data_scans(rec, B_B2, CASES_B2, 89, len(UNION_B2))
pair_rows = []
for c in S_B2:
    if META_B2[c]["form"] != "grants50":
        continue
    u = META_B2[c]["control_run"]
    sc_c = load(B_B2, c, "systemcost").groupby("t")["Value"].sum()
    sc_u = load(BATCH_OF_B2[u], u, "systemcost").groupby("t")["Value"].sum()
    for t in [y for y in YEARS_STD if 2031 <= y <= W_S]:
        cap_c, cap_u = float(SMR_CAP_B2[c].get(t, 0.0)), float(SMR_CAP_B2[u].get(t, 0.0))
        pair_rows.append(dict(grants50=c, control=u, world=META_B2[c]["world"], t=t,
                              smr_cap_grants50_MW=round(cap_c, 1), smr_cap_control_MW=round(cap_u, 1),
                              smr_cap_diff_MW=round(cap_c - cap_u, 1),
                              syscost_rel_diff=abs(float(sc_c.get(t, 0.0)) - float(sc_u.get(t, 0.0))) / max(abs(float(sc_u.get(t, 0.0))), 1.0)))
pid_b2 = pd.DataFrame(pair_rows)
pid_b2.to_csv(F_B2 / "pair_identity.csv", index=False)
rec("D", "each grants50 run reproduces its learning-on control on the years through 2038, where the two offered paths are identical "
    "(SMR capacity, system cost) -- disclosed either way", "INFO",
    f"{pid_b2['grants50'].nunique()} pairs (3 controls from batch 6); max |SMR capacity difference| "
    f"{pid_b2.smr_cap_diff_MW.abs().max():.1f} MW; max system-cost rel diff {pid_b2.syscost_rel_diff.max():.2e}")
pre_smr = {c: float(load(B_B2, c, "cap").query("i == 'nuclear-smr' and t < 2031")["Value"].sum()) for c in CASES_B2}
rec("D", "no SMR capacity exists before 2031", "PASS" if max(pre_smr.values()) == 0 else "FAIL",
    f"max pre-2031 SMR capacity {max(pre_smr.values()):.1f} MW")
added_scans(rec, B_B2, CASES_B2, 89)
S_B2REG = write_registry(REG_B2, F_B2)

[PASS] b2 D :: the exogenous load is identical in all 89 runs and equal to the Step 3 reference load -- max rel difference 0.00e+00 vs smr100_eia_p50


[PASS] b2 D :: no data value is NaN or infinite in any file -- all 207 keys x 89 files scanned (key inventory)
[PASS] b2 D :: no negative values occur where values must not be negative -- checked ['cap', 'cap_nat', 'load_rt', 'stor_in', 'stor_out', 'curt_ann', 'hours'] (storage charging and upgrades are legitimately negative elsewhere)
[PASS] b2 D :: the nuclear build and generation data is not negative -- cap_new_ann and gen_ann nuclear rows >= 0, all 89 cases


[PASS] b2 D :: the national totals equal the sum of the regional data -- cap, gen_ann, cap_new_ann, ret_ann; all 89 cases within 1e-3


[INFO] b2 D :: each grants50 run reproduces its learning-on control on the years through 2038, where the two offered paths are identical (SMR capacity, system cost) -- disclosed either way -- 24 pairs (3 controls from batch 6); max |SMR capacity difference| 0.0 MW; max system-cost rel diff 0.00e+00


[PASS] b2 D :: no SMR capacity exists before 2031 -- max pre-2031 SMR capacity 0.0 MW


[PASS] b2 X :: no load is dropped after 2025 (the step3_checks E3 rule) -- 89 runs; the historical 2010-2023 artifact 2.96 TWh (case spread 0.0 MWh)


[INFO] b2 X :: national curtailment in 2050 (context) -- 1.6 to 34.4 TWh (median 7.9) over 89 runs
rate_design2_b2_checks {'PASS': 30, 'INFO': 11}


## 5. `batch3`: paired market-condition anchors and the flat credit under three market conditions

**What the batch tests.** Each batch-3 run copies a base-condition run and changes one market
condition (the *market world*): `gaslo` / `gashi` (the AEO 2026 high / low oil-and-gas-supply
scenario: a low / high gas price), `demhi` (high electricity demand), `relo` / `rehi` (advanced /
conservative ATB renewable and storage costs), `translim` (constrained transmission) and `co2cap`
(a national CO2 cap falling to 5% of the 2005 level by 2050). Block A holds the *paired samples*:
the mandate anchors of the base condition copied into one world each (A1: the p25 and p75 anchors
in the six Step 4 worlds, 72 runs; A2: all five anchors in the CO2-cap world, 30 runs). Their
shadow prices (`duals_by_year.csv`) are what notebook 11 reads to transfer the required rate
across market conditions (Fig 5, ST4). Block B holds the 48 flat-50% credit runs of batch 2 in the
high-gas, CO2-cap and high-renewable-cost worlds; notebook 12 draws them in SF15. Block R is the
re-run of `bm_cop28_d9396_e2`.

**The two unreturned runs.** `smr100_eo_p75_demhi` and `smr100_iaea_p75_demhi` have no output.
The notebook asserts that exactly these two are missing, carries them as "not returned" and runs
every test on the 149 returned files; the high-demand condition rests on 10 of its 12 anchors.

**World checks (phase W).** Each world's switch must have its intended effect against the
run's own base-condition source run, and must not leak into the other worlds: the gas price moves
in the right direction in the gas worlds, the load rises in `demhi`, the renewable capital costs
move in `relo`/`rehi`, transmission build falls in `translim`, and in `co2cap` national process
CO2 stays at or below the cap, the CO2 price is positive exactly where the cap binds, and the
distributed-PV path rises (`co2cap_binding.csv`, `switch_echo.csv`).

In [26]:
# ---- batch3: case matrix, registration, metadata ------------------------------------------------
F_B3 = outdir("rate_design2_b3_checks")
REG_B3 = []
rec = make_record(REG_B3, "b3")
B_B3 = "batch3"
CF = {name: casefile(name, dtype=str) for name in ("batch3", "batch2", "ratedesign", "smr100", "step4")}
cases_b3 = CF["batch3"]
CF_MASTER = pd.read_csv(paths.REPO / "cases.csv", index_col=0, dtype=str)   # the fork's switch table
CASES_SPEC_B3 = [c for c in cases_b3.columns if c != "Default Value"]
assert len(CASES_SPEC_B3) == 151, f"the batch3 casefile has 151 runs, found {len(CASES_SPEC_B3)}"
MISSING_B3 = sorted(c for c in CASES_SPEC_B3 if not returned(B_B3, c))
assert MISSING_B3 == ["smr100_eo_p75_demhi", "smr100_iaea_p75_demhi"], \
    f"exactly the two high-demand p75 anchors of eo and iaea never returned; found {MISSING_B3}"
CASES_B3 = [c for c in CASES_SPEC_B3 if c not in MISSING_B3]
assert len(CASES_B3) == 149
WORLDS = ["gaslo", "gashi", "demhi", "relo", "rehi", "translim", "co2cap"]
WORLD_LABEL = {"gaslo": "low gas price (high oil and gas supply)", "gashi": "high gas price (low supply)",
               "demhi": "high demand", "relo": "low RE and storage cost", "rehi": "high RE and storage cost",
               "translim": "constrained transmission", "co2cap": "95%-by-2050 national CO2 cap", "base": "base market world"}
SENS = {   # the registered override set of each world (methods v2.6); asserted against the step4 casefile below
    "gaslo": {"ngscen": "AEO_2026_HOG"},
    "gashi": {"ngscen": "AEO_2026_LOG"},
    "demhi": {"GSw_LoadProfiles": "EER2025_100by2050"},
    "relo": {"plantchar_upv": "upv_ATB_2024_advanced", "plantchar_onswind": "ons-wind_ATB_2024_advanced",
             "plantchar_ofswind": "ofs-wind_ATB_2024_advanced", "plantchar_battery": "battery_ATB_2024_advanced",
             "plantchar_csp": "csp_ATB_2024_advanced", "plantchar_geo": "geo_ATB_2024_advanced",
             "plantchar_hydro": "hydro_ATB_2019_low", "geodiscov": "TI", "distpvscen": "stscen2023_lowre"},
    "rehi": {"plantchar_upv": "upv_ATB_2024_conservative", "plantchar_onswind": "ons-wind_ATB_2024_conservative",
             "plantchar_ofswind": "ofs-wind_ATB_2024_conservative", "plantchar_battery": "battery_ATB_2024_conservative",
             "plantchar_csp": "csp_ATB_2024_conservative", "plantchar_geo": "geo_ATB_2024_conservative",
             "distpvscen": "stscen2023_highre"},
    "translim": {"GSw_TransRestrict": "transreg", "GSw_TransInvMaxLongTerm": "1.07"},
    "co2cap": {"GSw_AnnualCap": "1", "GSw_AnnualCapScen": "start2027_95pct2050", "distpvscen": "stscen2023_mid_case_95_by_2050"},
    "base": {},
}
s4 = CF["step4"]
_ov = {k for ov in SENS.values() for k in ov}
for w in WORLDS[:-1]:
    for col in [c for c in s4.columns if c.endswith("_" + w)][:3]:
        got = {row: s4.loc[row, col] for row in s4.index if isinstance(s4.loc[row, col], str) and s4.loc[row, col] != "" and row in _ov}
        assert got == SENS[w], (col, got, SENS[w])

spec = pd.read_csv(REG_DIR / "batch3" / "b3_run_spec.csv").set_index("run")
pred = pd.read_csv(REG_DIR / "batch3" / "b3_predictions.csv").set_index("run")
assert set(spec.index) == set(CASES_SPEC_B3)
assert spec["block"].value_counts().to_dict() == {"A1": 72, "A2": 30, "B2": 24, "B1": 18, "B3": 6, "R": 1}
assert set(pred.index) == set(spec[spec["block"].isin(["B1", "B2", "B3"])].index)
BLOCK_B3 = {c: spec.loc[c, "block"] for c in CASES_B3}
WORLD = {c: spec.loc[c, "world"] for c in CASES_B3}
SRC = {c: spec.loc[c, "source_column"] for c in CASES_B3}
SRC_CF = {c: spec.loc[c, "source_casefile"].replace("cases_nuclearlearning_", "").replace(".csv", "") for c in CASES_B3}
SRC_BATCH = {c: ("smr100" if SRC_CF[c] == "smr100" else "ratedesign" if SRC_CF[c] == "ratedesign" else "batch2") for c in CASES_B3}
A_B3 = [c for c in CASES_B3 if BLOCK_B3[c] in ("A1", "A2")]
Bb_B3 = [c for c in CASES_B3 if BLOCK_B3[c] in ("B1", "B2", "B3")]
R_B3 = [c for c in CASES_B3 if BLOCK_B3[c] == "R"]
assert [len(A_B3), len(Bb_B3), len(R_B3)] == [100, 48, 1]

def pct_of(c):
    return c.split("_")[2]

def resolved(cf, col):
    # the full switch set of one column: its value where given, else that casefile's Default Value,
    # else (a row the casefile lacks) the fork's cases.csv default
    out = {}
    for row in cases_b3.index:
        v = cf.loc[row, col] if row in cf.index else None
        if not isinstance(v, str) or v.strip() == "":
            v = cf.loc[row, "Default Value"] if row in cf.index else None
        if not isinstance(v, str) or v.strip() == "":
            v = CF_MASTER.loc[row, "Default Value"] if row in CF_MASTER.index else ""
        out[row] = "" if not isinstance(v, str) else v.strip()
    return out

def sw_b3(case, name):
    v = cases_b3.loc[name, case] if name in cases_b3.index else None
    if not isinstance(v, str) or v.strip() == "":
        v = cases_b3.loc[name, "Default Value"]
    return "" if not isinstance(v, str) else v.strip()

META_B3 = {}
for c in CASES_B3:
    META_B3[c] = meta_of(sw_b3, c)
    META_B3[c].update(block=BLOCK_B3[c], world=WORLD[c], pct=pct_of(c), src=SRC[c],
                      cap=sw_b3(c, "GSw_AnnualCap"), capscen=sw_b3(c, "GSw_AnnualCapScen"), ngscen=sw_b3(c, "ngscen"),
                      loadprof=sw_b3(c, "GSw_LoadProfiles"), distpv=sw_b3(c, "distpvscen"))
    if BLOCK_B3[c] in ("B1", "B2", "B3"):
        s = m50.loc[SRC[c]]
        META_B3[c].update(form=s["form"], b2_world=s["world"], world_case=s["world_case"], draw_index=int(s["draw_index"]),
                          control_run=s["control_run"], b2_prediction=s["prediction"], prediction=pred.loc[c, "prediction"])
    elif BLOCK_B3[c] == "R":
        s = m50.loc[SRC[c]]
        META_B3[c].update(form=s["form"], b2_world=s["world"], world_case=s["world_case"], draw_index=int(s["draw_index"]))
    else:
        META_B3[c].update(form="mandate", draw_index=int(spec.loc[c, "draw_index"]))
OFFER_B3 = {c: m51[m51["run"] == SRC[c]].set_index("year")["offer_rate"].to_dict() for c in Bb_B3}
RATES_B3 = {c: {int(t): round(float(v), 3) for t, v in OFFER_B3[c].items()} for c in Bb_B3}
M_EXP_B3 = {c: {t: f * (1.0 - PEN) for t, f in RATES_B3[c].items()} for c in Bb_B3}

def load_src(c, key):
    return load(SRC_BATCH[c], SRC[c], key)

print(f"batch3: {len(CASES_B3)} returned of {len(CASES_SPEC_B3)} = {len(A_B3)} A + {len(Bb_B3)} B + {len(R_B3)} R; "
      f"not returned: {MISSING_B3}")

batch3: 149 returned of 151 = 100 A + 48 B + 1 R; not returned: ['smr100_eo_p75_demhi', 'smr100_iaea_p75_demhi']


In [27]:
# --- phase A ------------------------------------------------------------------------------------
missing = [c for c in CASES_B3 if not returned(B_B3, c)]
rec("A", "one output file exists for each of the 149 returned cases", "PASS" if not missing else "FAIL",
    f"missing: {missing}" if missing else "149 runs in the key inventory (file stem batch3_, not the registered sensitivity_analysis2_)")
rec("A", "registered runs with no output file in the delivery (carried as 'not returned'; nothing is inferred)", "INFO",
    f"{MISSING_B3} (2 of 151: the p75 anchors of eo and iaea in the high-demand world; the demhi world shift rests on 10 of 12 anchors)")
src_missing = sorted({(SRC_BATCH[c], SRC[c]) for c in CASES_B3 if not returned(SRC_BATCH[c], SRC[c])})
ref_missing = ([] if returned("step4", REF4) else [REF4]) + ([] if returned("smr100", REF_LOAD) else [REF_LOAD])
rec("A", "the source files are available (18 test1_ + 12 rate_design_ map anchors, the 24 batch-2 flat50 sources, the batch-2 "
    "re-run's own column, the Step 4 key-vintage reference)", "PASS" if not (src_missing or ref_missing) else "FAIL",
    f"missing: {src_missing + ref_missing}" if (src_missing or ref_missing) else
    f"{len({(SRC_BATCH[c], SRC[c]) for c in CASES_B3})} distinct source runs present")
CAP_KEYS = {"co2_price"}
KS_B3 = {c: keyset(B_B3, c) for c in CASES_B3}
UNION_B3 = set.union(*KS_B3.values())
bad = {c: sorted(d) for c in CASES_B3 if (d := ((UNION_B3 - KS_B3[c]) | (KS_B3[c] - UNION_B3)) - DUAL_KEYS - CAP_KEYS)}
cap_key_bad = [c for c in CASES_B3 if (("co2_price" in KS_B3[c]) != (WORLD[c] == "co2cap"))]
a_dual = [c for c in A_B3 if not ({"nuclear_cap_price", "nuclear_cap_price_raw"} <= KS_B3[c])]
b_dual = [c for c in Bb_B3 + R_B3 if KS_B3[c] & DUAL_KEYS]
drift = sorted(((UNION_B3 - REF4_KEYS) | (REF4_KEYS - UNION_B3)) - DUAL_KEYS - CAP_KEYS)
rec("A", "all 149 files have the same data keys up to the dual keys (block A carries nuclear_cap_price, blocks B/R do not) and "
    "the cap key (co2_price exists exactly in the co2cap runs)", "PASS" if not (bad or a_dual or b_dual or cap_key_bad) else "FAIL",
    f"key counts {sorted(set(len(k) for k in KS_B3.values()))}" + (f"; unexpected differences: {bad}" if bad else "")
    + (f"; anchors without a dual key: {a_dual}" if a_dual else "") + (f"; credit runs with a dual key: {b_dual}" if b_dual else "")
    + (f"; co2_price against the world: {cap_key_bad}" if cap_key_bad else ""))
rec("A", "the key set agrees with the Step 4 vintage (separate-delivery drift detector; dual and cap keys excluded)",
    "PASS" if not drift else "FAIL", f"drift vs {REF4}: {drift}" if drift else f"{len(REF4_KEYS)} keys, no drift")
_, _, PVF_B3, CS_B3 = solve_health(
    rec, B_B3, CASES_B3,
    years_of=lambda c: (YEARS_STD, META_B3[c]["yearset"] == YEARS_STD and META_B3[c]["endyear"] == 2050))

[PASS] b3 A :: one output file exists for each of the 149 returned cases -- 149 runs in the key inventory (file stem batch3_, not the registered sensitivity_analysis2_)
[INFO] b3 A :: registered runs with no output file in the delivery (carried as 'not returned'; nothing is inferred) -- ['smr100_eo_p75_demhi', 'smr100_iaea_p75_demhi'] (2 of 151: the p75 anchors of eo and iaea in the high-demand world; the demhi world shift rests on 10 of 12 anchors)
[PASS] b3 A :: the source files are available (18 test1_ + 12 rate_design_ map anchors, the 24 batch-2 flat50 sources, the batch-2 re-run's own column, the Step 4 key-vintage reference) -- 55 distinct source runs present
[PASS] b3 A :: all 149 files have the same data keys up to the dual keys (block A carries nuclear_cap_price, blocks B/R do not) and the cap key (co2_price exists exactly in the co2cap runs) -- key counts [207, 208, 209, 210]
[PASS] b3 A :: the key set agrees with the Step 4 vintage (separate-delivery drift detector; dual an

[PASS] b3 A :: the solver residuals are small in every file -- worst |residual| 6.00e-03 at smr100_aj_p50_co2cap
[PASS] b3 A :: the objective value is a normal positive number in every file -- range 1.3268e+12 to 2.0775e+12


[PASS] b3 A :: the model years are the standard 16-year set with end year 2050 in every file -- every file on its registered year set


[PASS] b3 A :: every run used the sequential solve mode -- pvf_capital==1 (max dev 0.0e+00); pvf_onm flat from 2026 (max dev 9.5e-07); cost_scale==1; z_rep has one row per model year


In [28]:
# --- B1: the QA-R6 rule: resolved(batch3 column) - resolved(source column) == the world's override set ---
de_rows, b1_bad = [], []
for c in CASES_B3:
    r3, rs = resolved(cases_b3, c), resolved(CF[SRC_CF[c]], SRC[c])
    diff = {k: (rs[k], r3[k]) for k in r3 if r3[k] != rs[k]}
    if {k: v[1] for k, v in diff.items()} != SENS[WORLD[c]]:
        b1_bad.append((c, diff, SENS[WORLD[c]]))
    for k, (a, b) in diff.items():
        de_rows.append(dict(run=c, block=BLOCK_B3[c], world=WORLD[c], source=SRC[c], switch=k, source_value=a, batch3_value=b))
pd.DataFrame(de_rows).to_csv(F_B3 / "design_echo.csv", index=False)
rec("B", "each returned column's resolved switch set differs from its source column by exactly its world's override set (QA-R6: "
    "gas scenario / load profile / RE plant files + distpv / transmission / CO2 cap + cap scenario + distpv; the re-run by nothing)",
    "PASS" if not b1_bad else "FAIL", b1_bad[:3] or f"149 columns; override rows {sorted({r['switch'] for r in de_rows})}")

mx_rows, mx_bad = [], []
for c in CASES_B3:
    m_, ab, p = META_B3[c], sched_of(c), pct_of(c)
    tok = STEM2TOK[ab]
    common = m_["scen"] == f"{tok}_smr" and m_["endyear"] == 2050 and m_["techscen"] == "smr" and m_["yearset"] == YEARS_STD
    if BLOCK_B3[c] in ("A1", "A2"):
        ok = (common and c == f"smr100_{ab}_{p}_{WORLD[c]}" and SRC[c] == f"smr100_{ab}_{p}" and m_["mode"] == 1 and m_["learning"] == 0
              and m_["suffix"] == "obbba_nonuclearitc" and m_["foreign"] == "mid" and m_["pc"]["nuclear-smr"] == f"nuclear-smr_mc_smr100_{ab}_{p}"
              and m_["fin"] == f"mc_smr100_{ab}_{p}" and m_["constr"] == f"mc_smr100_{ab}_{p}" and m_["tcx"] == "0")
    else:
        w = m_["draw_index"]
        src_ok = ((c == f"bx_{ab}_{p}_f50_{WORLD[c]}" and SRC[c] == f"bx_{ab}_{p}_f50" and m_["form"] == "flat50") if BLOCK_B3[c] != "R"
                  else (c == SRC[c] == "bm_cop28_d9396_e2" and m_["form"] == "map_plus_2e6" and WORLD[c] == "base"))
        ok = (common and src_ok and m_["mode"] == 0 and m_["learning"] == 1 and m_["suffix"] == f"obbba_rd_{SRC[c]}"
              and m_["foreign"] == f"rd_{ab}_d{w}" and m_["world_case"] == f"rd_{ab}_d{w}"
              and m_["pc"]["nuclear-smr"] == f"nuclear-smr_mc_rd_{ab}_d{w}" and m_["pc"]["nuclear"] == f"nuclear_mc_rd_{ab}_d{w}"
              and m_["fin"] == f"mc_rd_{ab}_d{w}" and m_["constr"] == f"mc_rd_{ab}_d{w}" and m_["tcx"] == "1"
              and sw_b3(c, "GSw_NuclearLearning_OCC") == "1" and sw_b3(c, "GSw_NuclearLearning_Duration") == "1"
              and sw_b3(c, "GSw_NuclearLearning_CrossTech") == "1")
    if not ok:
        mx_bad.append(c)
    mx_rows.append(dict(case=c, block=BLOCK_B3[c], world=WORLD[c], form=m_["form"], sched=ab, pct=p, mandate=m_["mode"],
                        learning=m_["learning"], incentives=m_["suffix"], foreign=m_["foreign"], scen=m_["scen"], tcx=m_["tcx"],
                        cap=m_["cap"], capscen=m_["capscen"], ngscen=m_["ngscen"], loadprof=m_["loadprof"], distpv=m_["distpv"]))
pd.DataFrame(mx_rows).to_csv(F_B3 / "case_matrix.csv", index=False)
rec("B", "the block conventions hold (A: mandate ON, learning OFF, no nuclear ITC, the anchor's own cost files, the paired-anchor "
    "naming; B/R: fbC with the source run's incentives file, rd_ world files and foreign file, the phaseout exemption, the flat50 / "
    "map_plus form)", "PASS" if not mx_bad else "FAIL", mx_bad or "149 cases match the block rules and naming")
b1c_bad = []
for c in Bb_B3 + R_B3:
    ab, w = sched_of(c), META_B3[c]["draw_index"]
    world = Z[f"worlds_{STEM2TOK[ab]}"][w]
    for col, name in SWITCH_OF.items():
        exp, got = float(world[WCOLS.index(col)]), float(sw_b3(c, name))
        if abs(got - exp) > max(1e-5 * abs(exp), 5e-6):
            b1c_bad.append((c, col, got, exp))
rec("B", "the block-B and block-R learning switches equal the drawn world parameters at the m50 draw index",
    "PASS" if not b1c_bad else "FAIL", b1c_bad[:5] or f"49 runs x {len(SWITCH_OF)} switches match mc_perdraw.npz to 1e-5")

echo_rows, b2_bad = [], []
for r in Bb_B3:
    got, flags = incentives_rows(r, META_B3[r]["suffix"], RATES_B3[r])
    flat_ok = sorted(got) == MODEL_YEARS and all(v == CAP_S for v in got.values())
    flags = dict(only_smr=flags["only_smr"], conv_ok=flags["conv_ok"], rate_ok=flags["rate_ok"], flat_ok=flat_ok)
    if not all(flags.values()):
        b2_bad.append((r, flags))
    for t in sorted(set(got) | set(RATES_B3[r])):
        echo_rows.append(dict(run=r, block=BLOCK_B3[r], world=WORLD[r], sched=sched_of(r), t=t,
                              itc_frac_file=got.get(t), itc_frac_m51=RATES_B3[r].get(t)))
pd.DataFrame(echo_rows).to_csv(F_B3 / "incentives_echo.csv", index=False)
rec("B", "the 48 block-B fed incentives files equal the m51 flat50 paths (0.50 on the ten model years; the batch-2 file of the "
    "source run reused unchanged)", "PASS" if not b2_bad else "FAIL",
    b2_bad[:3] or "48 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0; 0.50 x 10 years")

class _NaNExp(dict):
    def get(self, k, default=None):
        return np.nan
rr_b3 = pd.DataFrame([row for c in Bb_B3 + R_B3 for row in rate_recovery_rows(
    B_B3, c, META_B3[c]["fin"], M_EXP_B3[c] if c in M_EXP_B3 else _NaNExp(),
    dict(block=BLOCK_B3[c], world=WORLD[c], sched=sched_of(c)))])
rr_b3.to_csv(F_B3 / "rate_recovery.csv", index=False)
a_itc = [(c, d) for c in A_B3 if (d := fmdiff(B_B3, c)) > 1e-9]
rec("B", "the block-A anchors carry no nuclear credit (fin_mult == fin_mult_noITC for nuclear and nuclear-smr, all 100)",
    "PASS" if not a_itc else "FAIL", a_itc[:5] or "no nuclear ITC in any anchor (obbba_nonuclearitc)")
rrb = rr_b3[rr_b3["block"] != "R"]
rate_landing_checks(rec, rrb, OFFER_B3, (
    "the recovered monetized rate equals 0.9 x 0.50 at exactly the ten model years (48 block-B runs; the construction multiplier "
    "cancels in the ratio)",
    "the recovered rate is zero outside the offered years and for the large reactor in all 48 block-B runs",
    "the recovered rate is uniform across regions",
    "the phaseout does not reduce the nuclear credit in any year (exemption held)"))
rrr = rr_b3[(rr_b3["block"] == "R") & (rr_b3["i"] == "nuclear-smr")].set_index("t")["m_recovered"]
m51r = m51[m51["run"] == "bm_cop28_d9396_e2"].set_index("year")["offer_rate"]
d_r = max(abs(float(rrr.get(t, 0.0)) - 0.9 * float(m51r.loc[t])) for t in m51r.index)
rec("B", "the re-run bm_cop28_d9396_e2 carries its batch-2 map_plus_2e6 path (0.9 x the m51 offer at the offered years)",
    "PASS" if d_r <= TOL_M else "FAIL", f"worst |dm| {d_r:.4f} over {len(m51r)} offered years")

[PASS] b3 B :: each returned column's resolved switch set differs from its source column by exactly its world's override set (QA-R6: gas scenario / load profile / RE plant files + distpv / transmission / CO2 cap + cap scenario + distpv; the re-run by nothing) -- 149 columns; override rows ['GSw_AnnualCap', 'GSw_AnnualCapScen', 'GSw_LoadProfiles', 'GSw_TransInvMaxLongTerm', 'GSw_TransRestrict', 'distpvscen', 'geodiscov', 'ngscen', 'plantchar_battery', 'plantchar_csp', 'plantchar_geo', 'plantchar_hydro', 'plantchar_ofswind', 'plantchar_onswind', 'plantchar_upv']
[PASS] b3 B :: the block conventions hold (A: mandate ON, learning OFF, no nuclear ITC, the anchor's own cost files, the paired-anchor naming; B/R: fbC with the source run's incentives file, rd_ world files and foreign file, the phaseout exemption, the flat50 / map_plus form) -- 149 cases match the block rules and naming


[PASS] b3 B :: the block-B and block-R learning switches equal the drawn world parameters at the m50 draw index -- 49 runs x 11 switches match mc_perdraw.npz to 1e-5


[PASS] b3 B :: the 48 block-B fed incentives files equal the m51 flat50 paths (0.50 on the ten model years; the batch-2 file of the source run reused unchanged) -- 48 files; Nuclear-SMR only; safe_harbor 0; online == start; penalty 0.1; bonuses 0; 0.50 x 10 years


[PASS] b3 B :: the block-A anchors carry no nuclear credit (fin_mult == fin_mult_noITC for nuclear and nuclear-smr, all 100) -- no nuclear ITC in any anchor (obbba_nonuclearitc)
[PASS] b3 B :: the recovered monetized rate equals 0.9 x 0.50 at exactly the ten model years (48 block-B runs; the construction multiplier cancels in the ratio) -- 480 fed case-years; worst |dm| 0.0001 at (bx_eia_p05_f50_gashi, 2033)
[PASS] b3 B :: the recovered rate is zero outside the offered years and for the large reactor in all 48 block-B runs -- 672 points; max |m| 0.0000
[PASS] b3 B :: the recovered rate is uniform across regions -- max spread over regions 0.00094
[PASS] b3 B :: the phaseout does not reduce the nuclear credit in any year (exemption held) -- 240 credited case-years at t >= 2036; worst |dm| 0.0001
[PASS] b3 B :: the re-run bm_cop28_d9396_e2 carries its batch-2 map_plus_2e6 path (0.9 x the m51 offer at the offered years) -- worst |dm| 0.0001 over 10 offered years


In [29]:
# --- B11/B12: ITC payments; B13: cost fingerprints; B16 ------------------------------------------
SMR_NEW = {c: float(load(B_B3, c, "cap_new_ann").query("i == 'nuclear-smr' and t > 2030")["Value"].sum()) for c in CASES_B3}
pay_bad, no_build, a_pay, ITC_PAY = [], [], [], {}
for c in CASES_B3:
    sc = load(B_B3, c, "systemcost_techba")
    it = sc[sc["sys_costs"].str.contains("itc", case=False) & sc["i"].isin(NUC)]
    ITC_PAY[c] = it.groupby("t")["Value"].sum()
    v = float(it["Value"].abs().sum())
    if c in Bb_B3 + R_B3:
        if SMR_NEW[c] > 100.0 and v < 1e6:
            pay_bad.append((c, f"builds {SMR_NEW[c]:,.0f} MW but no nuclear ITC payments"))
        if SMR_NEW[c] <= 100.0:
            no_build.append(c)
    elif v > 1e3:
        a_pay.append((c, v))
rec("B", "the system cost data shows nuclear ITC payments wherever SMR builds occurred (blocks B/R) and none in block A",
    "PASS" if not (pay_bad or a_pay) else "FAIL",
    (pay_bad + a_pay)[:5] or f"payments present in every credit run with builds; credit runs with ~no builds (lock-in candidates, "
    f"read in phase C): {no_build or 'none'}; no nuclear ITC line in any anchor")
itc_pay_b3 = pd.concat([s.rename("itc_pay_2004").reset_index().assign(case=c) for c, s in ITC_PAY.items()],
                       ignore_index=True)[["case", "t", "itc_pay_2004"]]
itc_pay_b3.to_csv(F_B3 / "itc_payments.csv", index=False)
itc_mag = {c: abs(float(load(B_B3, c, "tax_expenditure_itc")["Value"].sum())) for c in CASES_B3}
rec("B", "the other technologies keep their ITC in all 149 runs", "PASS" if min(itc_mag.values()) > 1e8 else "FAIL",
    f"total ITC tax expenditure magnitude {min(itc_mag.values())/1e9:.1f} to {max(itc_mag.values())/1e9:.1f} B$ per case")

fp_b3 = pd.DataFrame([row for c in CASES_B3 for row in fingerprint_rows(B_B3, c, META_B3[c], dict(block=BLOCK_B3[c], world=WORLD[c]))])
fp_b3.to_csv(F_B3 / "fingerprint_errors.csv", index=False)
a_fp = fp_b3[fp_b3["block"].isin(["A1", "A2"])]
rec("B", "the 100 block-A anchors use their world's cost path in every year (learning OFF; the anchor's own plantchar file)",
    "PASS" if a_fp.rel_err.max() < 1e-4 else "FAIL", f"{len(a_fp)} points; max rel err {a_fp.rel_err.max():.2e}")
pre_fp = fp_b3[fp_b3["block"].isin(["B1", "B2", "B3", "R"]) & (fp_b3["t"] <= 2030)]
rec("B", "the 49 learning-on runs use the drawn cost path before the 2031 anchor",
    "PASS" if pre_fp.rel_err.max() < 1e-4 else "FAIL", f"{len(pre_fp)} points; max rel err {pre_fp.rel_err.max():.2e}")
on_fp = fp_b3[fp_b3["block"].isin(["B1", "B2", "B3"]) & (fp_b3["t"] >= 2031) & (fp_b3["i"] == "nuclear-smr")]
dev = on_fp.groupby("case")["rel_err"].max().round(3)
rec("B", "the learning-on runs write an in-run SMR cost path (max deviation from the drawn path, 2031-2050; context, read by the "
    "block-B verdict notebook as the learning feedback)", "INFO",
    f"median over runs {dev.median():.3f}; range {dev.min():.3f}-{dev.max():.3f}; by world: {on_fp.groupby('world')['rel_err'].max().round(3).to_dict()}")
pair_bad = []
for c in A_B3:
    a = load(B_B3, c, "cost_cap"); b = load_src(c, "cost_cap")
    a = a[a["i"] == "nuclear-smr"].groupby("t")["Value"].mean()
    b = b[b["i"] == "nuclear-smr"].groupby("t")["Value"].mean()
    d = float((a - b.reindex(a.index)).abs().max() / b.abs().max())
    if d > 1e-9:
        pair_bad.append((c, d))
rec("B", "each block-A anchor's SMR cost path equals its base-world source anchor's (the paired difference is the market world alone)",
    "PASS" if not pair_bad else "FAIL", pair_bad[:5] or "100 anchors: cost_cap nuclear-smr identical to the source run")
TRAJ_B3 = {c: trajectory(META_B3[c]["scen"]) for c in CASES_B3}
b16_bad = [c for c in CASES_B3 if META_B3[c]["scen"] != f"{STEM2TOK[sched_of(c)]}_smr"]
rec("B", "every run reads its schedule's standard mandate trajectory file (no _ext token)",
    "PASS" if not b16_bad else "FAIL", b16_bad or "149 runs on the six standard files")

[PASS] b3 B :: the system cost data shows nuclear ITC payments wherever SMR builds occurred (blocks B/R) and none in block A -- payments present in every credit run with builds; credit runs with ~no builds (lock-in candidates, read in phase C): ['bx_eia_p75_f50_co2cap', 'bx_aj_p75_f50_co2cap', 'bx_iaea_p50_f50_co2cap', 'bx_iaea_p75_f50_co2cap', 'bx_mck_p75_f50_co2cap', 'bx_cop28_p50_f50_co2cap', 'bx_cop28_p75_f50_co2cap', 'bx_eo_p75_f50_co2cap']; no nuclear ITC line in any anchor


[PASS] b3 B :: the other technologies keep their ITC in all 149 runs -- total ITC tax expenditure magnitude 110.0 to 1006.5 B$ per case


[PASS] b3 B :: the 100 block-A anchors use their world's cost path in every year (learning OFF; the anchor's own plantchar file) -- 8200 points; max rel err 4.24e-07
[PASS] b3 B :: the 49 learning-on runs use the drawn cost path before the 2031 anchor -- 2058 points; max rel err 1.27e-07
[INFO] b3 B :: the learning-on runs write an in-run SMR cost path (max deviation from the drawn path, 2031-2050; context, read by the block-B verdict notebook as the learning feedback) -- median over runs 0.441; range 0.056-2.040; by world: {'co2cap': 2.04, 'gashi': 1.39, 'rehi': 0.553}


[PASS] b3 B :: each block-A anchor's SMR cost path equals its base-world source anchor's (the paired difference is the market world alone) -- 100 anchors: cost_cap nuclear-smr identical to the source run


[PASS] b3 B :: every run reads its schedule's standard mandate trajectory file (no _ext token) -- 149 runs on the six standard files


In [30]:
# --- phase W: world fingerprints against each run's own base-condition source run --------------------
echo_rows = []

def nat_series(c, key, src=False):
    df = load_src(c, key) if src else load(B_B3, c, key)
    return df.groupby("t")["Value"].sum()

RE_FAMS = ["upv", "wind-ons", "wind-ofs", "battery", "csp", "geo"]

def re_cost(c, src=False):
    cc = load_src(c, "cost_cap") if src else load(B_B3, c, "cost_cap")
    cc = cc.assign(fam=cc["i"].str.extract(r"^([a-z\-]+)", expand=False))
    cc = cc[cc["fam"].isin(RE_FAMS) & (cc["t"] >= 2026)]
    return cc.groupby(["fam", "t"])["Value"].mean()

BY_WORLD = {w: [c for c in CASES_B3 if WORLD[c] == w] for w in WORLDS}
gas_bad = []
for w in ("gaslo", "gashi"):
    for c in BY_WORLD[w]:
        g = load(B_B3, c, "repgasprice_nat").set_index("t")["Value"]
        gb = load_src(c, "repgasprice_nat").set_index("t")["Value"]
        yrs = [t for t in g.index if t >= 2026 and t in gb.index]
        d = g.reindex(yrs) - gb.reindex(yrs)
        frac_dir = float((d < 0).mean()) if w == "gaslo" else float((d > 0).mean())
        echo_rows.append(dict(case=c, world=w, metric="gas_price_mean_rel", value=float((g.reindex(yrs) / gb.reindex(yrs)).mean())))
        if frac_dir < 0.9:
            gas_bad.append((c, round(frac_dir, 2)))
rec("W", "the gas price sits below the source in gaslo and above it in gashi", "PASS" if not gas_bad else "FAIL",
    gas_bad[:5] or f"correct direction in >=90% of model years, {len(BY_WORLD['gaslo']) + len(BY_WORLD['gashi'])} gas-world runs")
dem_bad = []
for c in BY_WORLD["demhi"]:
    lr, lb = nat_series(c, "load_rt"), nat_series(c, "load_rt", src=True)
    yrs = [t for t in lr.index if t >= 2031]
    echo_rows.append(dict(case=c, world="demhi", metric="load_mean_rel", value=float((lr.reindex(yrs) / lb.reindex(yrs)).mean())))
    if not ((lr.reindex(yrs) >= lb.reindex(yrs) - 1.0).all() and float((lr.reindex(yrs) / lb.reindex(yrs)).mean()) >= 1.01):
        dem_bad.append(c)
rec("W", "the exogenous load sits above the source in every demhi case (mean ratio >= 1.01)", "PASS" if not dem_bad else "FAIL",
    dem_bad[:5] or f"{len(BY_WORLD['demhi'])} demhi cases; mean load ratio "
    f"{np.mean([r['value'] for r in echo_rows if r['metric'] == 'load_mean_rel']):.3f}")
ref_l = load(B_B3, BY_WORLD["demhi"][0], "load_rt").set_index(["r", "t"])["Value"]
dem_same = 0.0
for c in BY_WORLD["demhi"][1:]:
    j = pd.concat([ref_l, load(B_B3, c, "load_rt").set_index(["r", "t"])["Value"]], axis=1, keys=["a", "b"]).fillna(0.0)
    dem_same = max(dem_same, float(((j["a"] - j["b"]).abs() / j["a"].abs().clip(lower=1.0)).max()))
rec("W", "the exogenous load is identical inside the demhi block", "PASS" if dem_same < 1e-6 else "FAIL", f"max rel difference {dem_same:.2e}")
re_bad = []
for w in ("relo", "rehi"):
    for c in BY_WORLD[w]:
        j = pd.concat([re_cost(c), re_cost(c, src=True)], axis=1, keys=["sens", "base"]).dropna()
        rel = j["sens"] / j["base"]
        frac_dir = float((rel <= 1.0 + 1e-9).mean()) if w == "relo" else float((rel >= 1.0 - 1e-9).mean())
        echo_rows.append(dict(case=c, world=w, metric="re_cost_mean_rel", value=float(rel.mean())))
        moved = (float(rel.mean()) <= 0.99) if w == "relo" else (float(rel.mean()) >= 1.01)
        if frac_dir < 0.9 or not moved:
            re_bad.append((c, round(frac_dir, 2), round(float(rel.mean()), 3)))
rec("W", "the renewable capital costs sit below the source in relo and above it in rehi (mean ratio <= 0.99 / >= 1.01)",
    "PASS" if not re_bad else "FAIL",
    re_bad[:5] or f"correct direction in >=90% of (family, year) points, {len(BY_WORLD['relo']) + len(BY_WORLD['rehi'])} RE-world runs")
tr_bad = []
for c in BY_WORLD["translim"]:
    a = float(load(B_B3, c, "invtran_out")["Value"].sum())
    b = float(load_src(c, "invtran_out")["Value"].sum())
    echo_rows.append(dict(case=c, world="translim", metric="new_transmission_rel", value=a / b if b else np.nan))
    if a > b * 0.99:
        tr_bad.append((c, round(a / b, 3)))
rec("W", "the new transmission build sits below the source in translim (national total <= 0.99 x source)",
    "PASS" if not tr_bad else "FAIL",
    tr_bad[:5] or f"national new-transmission totals <= source, {len(BY_WORLD['translim'])} translim cases (national totals only)")

# the CO2-cap world: cap, price, distributed PV
cap_csv = pd.read_csv(INPUTS / "emission_constraints" / "co2_cap.csv").set_index("t")
CAP_PATH = {int(y): float(v) for y, v in cap_csv.loc["start2027_95pct2050"].items()}       # tons CO2
cap_rows, cap_bad, price_bad = [], [], []
for c in BY_WORLD["co2cap"]:
    e = load(B_B3, c, "emit_nat")
    co2 = e[(e["eall"] == "CO2") & (e["etype"] == "process")].groupby("t")["Value"].sum()
    eb = load_src(c, "emit_nat")
    co2b = eb[(eb["eall"] == "CO2") & (eb["etype"] == "process")].groupby("t")["Value"].sum()
    pr = load(B_B3, c, "co2_price").set_index("t")["Value"] if "co2_price" in KS_B3[c] else pd.Series(dtype=float)
    for t in [y for y in YEARS_STD if y >= 2027 and y in CAP_PATH]:
        em, cap_t, p = float(co2.get(t, np.nan)), CAP_PATH[t], float(pr.get(t, 0.0))
        binding = em >= cap_t * (1 - 1e-3)
        cap_rows.append(dict(case=c, block=BLOCK_B3[c], t=t, co2_Gt=em / 1e9, cap_Gt=cap_t / 1e9, source_co2_Gt=float(co2b.get(t, np.nan)) / 1e9,
                             slack_Gt=(cap_t - em) / 1e9, co2_price_2004_t=p, binding=binding))
        if em > cap_t * (1 + 1e-3):
            cap_bad.append((c, t, round(em / 1e9, 3), round(cap_t / 1e9, 3)))
        if (binding and p <= 0) or ((not binding) and p > 1e-6):
            price_bad.append((c, t, round(em / 1e9, 3), round(cap_t / 1e9, 3), round(p, 3)))
capdf = pd.DataFrame(cap_rows)
capdf.to_csv(F_B3 / "co2cap_binding.csv", index=False)
rec("W", "national process CO2 sits at or below the start2027_95pct2050 cap in every capped model year (54 co2cap runs)",
    "PASS" if not cap_bad else "FAIL", cap_bad[:5] or f"{len(capdf)} run-years within the cap (tolerance 0.1%)")
rec("W", "the CO2 price is positive exactly where the cap binds (emissions within 0.1% of the cap) and zero elsewhere",
    "PASS" if not price_bad else "FAIL", price_bad[:5] or "price and binding status agree in every run-year")
bind = capdf.groupby("t")["binding"].mean()
rec("W", "the registered expectation of the cap world (methods v2.6: binding at 2031, slack in 2032-2035, binding at 2038 and after)",
    "INFO", "share of co2cap runs binding by year: " + ", ".join(f"{t} {v:.2f}" for t, v in bind.items())
    + "; median CO2 price (2004 $/t) by year: " + ", ".join(f"{t} {v:.0f}" for t, v in capdf.groupby("t")["co2_price_2004_t"].median().items() if v > 0))
dpv_bad, dpv_rel = [], []
for c in BY_WORLD["co2cap"]:
    a = load(B_B3, c, "cap"); a = a[a["i"].str.startswith("distpv")].groupby("t")["Value"].sum()
    b = load_src(c, "cap"); b = b[b["i"].str.startswith("distpv")].groupby("t")["Value"].sum()
    yrs = [t for t in a.index if t >= 2031 and t in b.index]
    rel = a.reindex(yrs) / b.reindex(yrs)
    dpv_rel.append(float(rel.mean()))
    if not ((rel >= 1.0 - 1e-6).all() and float(rel.mean()) >= 1.01):
        dpv_bad.append((c, round(float(rel.min()), 3)))
rec("W", "the distributed-PV capacity sits above the source in every co2cap case (the Mid_Case_95by2050 distpv path; mean ratio >= 1.01)",
    "PASS" if not dpv_bad else "FAIL", dpv_bad[:5] or f"mean ratio {np.mean(dpv_rel):.3f} (range {min(dpv_rel):.3f}-{max(dpv_rel):.3f}) over 2031-2050")
leak = []
for c in CASES_B3:
    w = WORLD[c]
    if w != "demhi":
        lr, lb = nat_series(c, "load_rt"), nat_series(c, "load_rt", src=True)
        rel = float(((lr - lb).abs() / lb.abs().clip(lower=1.0)).max())
        if rel > 1e-6:
            leak.append((c, "load_rt", round(rel, 6)))
    if w not in ("relo", "rehi"):
        j = pd.concat([re_cost(c), re_cost(c, src=True)], axis=1, keys=["a", "b"]).dropna()
        rel = float(((j["a"] - j["b"]).abs() / j["b"].abs().clip(lower=1e-9)).max())
        if rel > 1e-6:
            leak.append((c, "re_cost", round(rel, 6)))
rec("W", "no market switch leaks into the other worlds (exogenous load and RE capital costs identical to the source outside their own "
    "worlds; the CO2-cap world changes neither)", "PASS" if not leak else "FAIL",
    leak[:8] or "149 runs: load and RE costs identical to the source outside demhi / relo / rehi")
gas_move = {}
for c in CASES_B3:
    if WORLD[c] not in ("gaslo", "gashi"):
        g = load(B_B3, c, "repgasprice_nat").set_index("t")["Value"]
        gb = load_src(c, "repgasprice_nat").set_index("t")["Value"]
        gas_move.setdefault(WORLD[c], []).append(float(((g - gb).abs() / gb.abs().clip(lower=1e-9)).max()))
rec("W", "endogenous gas-price response outside the gas worlds (context, not a leak)", "INFO",
    {w: f"max {max(v):.3f}" for w, v in sorted(gas_move.items())})
pd.DataFrame(echo_rows).to_csv(F_B3 / "switch_echo.csv", index=False)

[PASS] b3 W :: the gas price sits below the source in gaslo and above it in gashi -- correct direction in >=90% of model years, 42 gas-world runs


[PASS] b3 W :: the exogenous load sits above the source in every demhi case (mean ratio >= 1.01) -- 10 demhi cases; mean load ratio 1.156
[PASS] b3 W :: the exogenous load is identical inside the demhi block -- max rel difference 0.00e+00


[PASS] b3 W :: the renewable capital costs sit below the source in relo and above it in rehi (mean ratio <= 0.99 / >= 1.01) -- correct direction in >=90% of (family, year) points, 30 RE-world runs


[PASS] b3 W :: the new transmission build sits below the source in translim (national total <= 0.99 x source) -- national new-transmission totals <= source, 12 translim cases (national totals only)


[PASS] b3 W :: national process CO2 sits at or below the start2027_95pct2050 cap in every capped model year (54 co2cap runs) -- 594 run-years within the cap (tolerance 0.1%)
[PASS] b3 W :: the CO2 price is positive exactly where the cap binds (emissions within 0.1% of the cap) and zero elsewhere -- price and binding status agree in every run-year
[INFO] b3 W :: the registered expectation of the cap world (methods v2.6: binding at 2031, slack in 2032-2035, binding at 2038 and after) -- share of co2cap runs binding by year: 2029 0.00, 2031 0.81, 2032 0.00, 2033 0.00, 2034 0.00, 2035 0.00, 2038 0.50, 2041 0.87, 2044 0.89, 2047 0.91, 2050 1.00; median CO2 price (2004 $/t) by year: 2031 0, 2038 1, 2041 19, 2044 39, 2047 68, 2050 139


[PASS] b3 W :: the distributed-PV capacity sits above the source in every co2cap case (the Mid_Case_95by2050 distpv path; mean ratio >= 1.01) -- mean ratio 1.276 (range 1.276-1.276) over 2031-2050


[PASS] b3 W :: no market switch leaks into the other worlds (exogenous load and RE capital costs identical to the source outside their own worlds; the CO2-cap world changes neither) -- 149 runs: load and RE costs identical to the source outside demhi / relo / rehi


[INFO] b3 W :: endogenous gas-price response outside the gas worlds (context, not a leak) -- {'base': 'max 0.000', 'co2cap': 'max 0.223', 'demhi': 'max 0.185', 'rehi': 'max 0.140', 'relo': 'max 0.130', 'translim': 'max 0.047'}


### 5.1 Shadow prices of the paired samples and the flat-credit outcomes (phase C)

Block A: the duals table (section 1 convention, plus the world, block, schedule, percentile and
source run) with the same three consistency tests and the conversion test. Gate GA-w: every
returned anchor must yield at least one mandated model-year row; an anchor slack in every model
year is reported as slack, not dropped. Block B: the batch-2 classifier with the v2.5 gate rule
for the flat credit (every model year 2031 to 2050 with a positive mandate, plus 2050), joined with
the registered prediction and the base-condition outcome of the source run (from section 4).

In [31]:
SMR_CAP_B3 = {c: smr_cap(B_B3, c) for c in CASES_B3}
DUAL_B3 = {c: dual_series(B_B3, c) for c in A_B3}
duals_b3 = pd.DataFrame([row for c in A_B3 for row in duals_rows(
    c, TRAJ_B3[c], SMR_CAP_B3[c], DUAL_B3[c][0], DUAL_B3[c][1], META_B3[c]["yearset"],
    extra=dict(world=WORLD[c], block=BLOCK_B3[c], sched=sched_of(c), pct=pct_of(c), source=SRC[c]))])
duals_b3.to_csv(F_B3 / "duals_by_year.csv", index=False)
assert duals_b3["case"].nunique() == 100 and len(duals_b3) == 100 * 16, "100 returned anchors x 16 model years"
dual_checks(rec, duals_b3, "100 block-A anchors")
worst_conv = 0.0
for c in A_B3:
    dual, raw = DUAL_B3[c]
    d = dual[dual > 1e-9]
    if not len(d):
        continue
    implied = raw.reindex(d.index) / d / CS_B3[c]
    worst_conv = max(worst_conv, float(((implied - PVF_B3[c].reindex(implied.index)).abs() / PVF_B3[c].reindex(implied.index)).max()))
rec("C", "the dual conversion is correct on each anchor's own pvf_onm (raw = cost_scale x pvf_onm x converted)",
    "PASS" if worst_conv < 1e-6 else "FAIL", f"worst rel error {worst_conv:.1e}")
model_rows = duals_b3[duals_b3["t"].isin(MODEL_YEARS) & (duals_b3["mandate_MW"] > 0)]
mr = model_rows.assign(rate=model_rows["dual_2004_MWyr"] > 1e-9)
n_rate = mr.groupby("case")["rate"].sum()
all_slack = sorted(n_rate[n_rate == 0].index)
by_world = mr.groupby("world")["rate"].agg(["sum", "size"])
_ga_missing = sorted(set(A_B3) - set(model_rows["case"]))
rec("C", "GA-w (methods v2.6): each returned anchor yields a required-rate row set (>= 1 mandated model-year row in duals_by_year); "
    "anchors slack in every model year are reported as slack, not dropped", "PASS" if not _ga_missing else "FAIL",
    (f"anchors without a mandated model-year row: {_ga_missing}; " if _ga_missing else "")
    + f"rate-status model-year cells by world (of mandated cells): "
    + str({w: f"{int(r['sum'])}/{int(r['size'])}" for w, r in by_world.iterrows()}) + f"; anchors slack in every model year: {all_slack or 'none'}")
ovb_b3 = overbuild_rows(duals_b3)
ovb_b3.to_csv(F_B3 / "overbuild_by_case.csv", index=False)

# block B: the flat-50% credit in three market worlds
gap_rows = []
for c in Bb_B3:
    for t in YEARS_STD:
        req = float(TRAJ_B3[c].get(t, 0.0))
        if req <= 0:
            continue
        have = float(SMR_CAP_B3[c].get(t, 0.0))
        gap_rows.append(dict(case=c, block=BLOCK_B3[c], world=WORLD[c], form="flat50", sched=sched_of(c), pct=pct_of(c), t=t,
                             gated=(t in MODEL_YEARS and req > 0) or t == 2050, in_window=t <= W_S,
                             mandate_MW=req, cap_MW=round(have, 1), gap_MW=round(have - req, 1), gap_pct=round(100 * (have - req) / req, 2)))
gaps_b3 = pd.DataFrame(gap_rows)
gaps_b3.to_csv(F_B3 / "trajectory_gaps.csv", index=False)
out_rows = []
for c in Bb_B3:
    sub = gaps_b3[gaps_b3["case"] == c]
    tier = tier_of(sub, sub["gated"])
    cap_max = float(SMR_CAP_B3[c].max()) if len(SMR_CAP_B3[c]) else 0.0
    outcome, misses, gmax, end_req = outcome_of(sub, tier, cap_max, sub["gated"], sub["gated"])
    builds = SMR_CAP_B3[c][SMR_CAP_B3[c] > 5.0]
    p_ = pred.loc[c]
    out_rows.append(dict(case=c, block=BLOCK_B3[c], world=WORLD[c], convention="fbC", sched=sched_of(c), pct=pct_of(c),
                         source=SRC[c], draw_index=META_B3[c]["draw_index"], cap=CAP_S, form="flat50", window=W_S,
                         prediction=p_["prediction"], r_start=p_["r_start"], excess_post=p_["excess_post"], prediction_inputs=p_["inputs"],
                         source_outcome_base=O2.loc[SRC[c], "outcome"], source_delivers_base=bool(O2.loc[SRC[c], "delivers"]),
                         tier=tier, outcome=outcome, delivers=tier in ("exact", "reproduces"), miss_years="|".join(str(t) for t in misses),
                         first_shortfall_year=(min(misses) if misses else 0), in_window_shortfall=bool(any(t <= W_S for t in misses)),
                         max_gated_shortfall_MW=round(gmax, 1), first_build_year=int(builds.index.min()) if len(builds) else 0,
                         smr_2038_MW=round(float(SMR_CAP_B3[c].get(W_S, 0.0)), 1), traj_2038_MW=float(TRAJ_B3[c].get(W_S, 0.0)),
                         smr_2050_MW=round(float(SMR_CAP_B3[c].get(2050, 0.0)), 1), traj_2050_MW=end_req,
                         delivered_frac_2050=round(float(SMR_CAP_B3[c].get(2050, 0.0)) / end_req, 4)))
outc_b3 = pd.DataFrame(out_rows)
outc_b3.to_csv(F_B3 / "delivery_outcomes.csv", index=False)
assert len(outc_b3) == 48 and set(outc_b3["world"]) == {"gashi", "co2cap", "rehi"}
for w in ("gashi", "co2cap", "rehi"):
    rec("C", f"the delivery tiers and outcomes of the flat-50% runs in the {WORLD_LABEL[w]} world (a finding, not a pass condition)",
        "INFO", outc_b3[outc_b3["world"] == w].groupby(["pct", "outcome"])["case"].count().to_dict())
rec("C", "the registered block-B predictions (b3_predictions.csv; 'from block A' cells resolved by the verdict notebook) against the classifier",
    "INFO", outc_b3.groupby(["prediction", "outcome"])["case"].count().to_dict())
LARGE_NEW = {c: load(B_B3, c, "cap_new_ann").query("i == 'nuclear' and t > 2030")["Value"].sum() for c in CASES_B3}
sub_lg = {c: round(float(LARGE_NEW[c]) / 1000.0, 2) for c in CASES_B3 if float(LARGE_NEW[c]) > 0}
rec("C", "large-reactor substitution in the batch-3 runs (a report, not a failure)", "INFO",
    sub_lg or "no new large builds after 2030 in any of the 149 runs")
print(outc_b3.groupby(["world", "pct", "outcome"]).size().to_string())

[PASS] b3 C :: the capacity satisfies the mandate floor in every mandate run (100 block-A anchors) -- no violation larger than 5 MW in any mandated year
[PASS] b3 C :: the capacity equals the floor in each binding year (positive dual) -- every year with dual > $1/MW-yr has |slack| <= 5 MW
[PASS] b3 C :: a positive dual occurs only in the binding years -- every year with slack > 100 MW has a zero dual
[PASS] b3 C :: the dual conversion is correct on each anchor's own pvf_onm (raw = cost_scale x pvf_onm x converted) -- worst rel error 1.3e-07
[PASS] b3 C :: GA-w (methods v2.6): each returned anchor yields a required-rate row set (>= 1 mandated model-year row in duals_by_year); anchors slack in every model year are reported as slack, not dropped -- rate-status model-year cells by world (of mandated cells): {'co2cap': '257/275', 'demhi': '90/90', 'gashi': '107/110', 'gaslo': '110/110', 'rehi': '110/110', 'relo': '110/110', 'translim': '110/110'}; anchors slack in every model year: none


[INFO] b3 C :: the delivery tiers and outcomes of the flat-50% runs in the high gas price (low supply) world (a finding, not a pass condition) -- {('p05', 'full-delivery'): 6, ('p25', 'delayed-start-recover'): 2, ('p25', 'full-delivery'): 3, ('p25', 'under-delivery'): 1, ('p50', 'delayed-start-recover'): 2, ('p50', 'full-delivery'): 1, ('p50', 'under-delivery'): 3}
[INFO] b3 C :: the delivery tiers and outcomes of the flat-50% runs in the 95%-by-2050 national CO2 cap world (a finding, not a pass condition) -- {('p05', 'delayed-start-recover'): 4, ('p05', 'full-delivery'): 2, ('p25', 'delayed-start-recover'): 4, ('p25', 'under-delivery'): 2, ('p50', 'no-build-lock-in'): 2, ('p50', 'under-delivery'): 4, ('p75', 'no-build-lock-in'): 6}
[INFO] b3 C :: the delivery tiers and outcomes of the flat-50% runs in the high RE and storage cost world (a finding, not a pass condition) -- {('p05', 'delayed-start-recover'): 2, ('p05', 'full-delivery'): 4}
[INFO] b3 C :: the registered block-B predictio

[INFO] b3 C :: large-reactor substitution in the batch-3 runs (a report, not a failure) -- no new large builds after 2030 in any of the 149 runs
world   pct  outcome              
co2cap  p05  delayed-start-recover    4
             full-delivery            2
        p25  delayed-start-recover    4
             under-delivery           2
        p50  no-build-lock-in         2
             under-delivery           4
        p75  no-build-lock-in         6
gashi   p05  full-delivery            6
        p25  delayed-start-recover    2
             full-delivery            3
             under-delivery           1
        p50  delayed-start-recover    2
             full-delivery            1
             under-delivery           3
rehi    p05  delayed-start-recover    2
             full-delivery            4


In [32]:
# --- phase D -------------------------------------------------------------------------------------
li_b3 = load_invariance(B_B3, [c for c in CASES_B3 if WORLD[c] != "demhi"], extra=lambda c: dict(world=WORLD[c]))
li_b3.to_csv(F_B3 / "load_invariance.csv", index=False)
rec("D", "the exogenous load is identical to the Step 3 reference load in every run outside the demhi world (131 runs)",
    "PASS" if li_b3.max_rel_diff.max() < 1e-6 else "FAIL", f"max rel difference {li_b3.max_rel_diff.max():.2e} vs {REF_LOAD}")
data_scans(rec, B_B3, CASES_B3, 149, len(UNION_B3))
pre_smr = {c: float(load(B_B3, c, "cap").query("i == 'nuclear-smr' and t < 2031")["Value"].sum()) for c in CASES_B3}
rec("D", "no SMR capacity exists before 2031", "PASS" if max(pre_smr.values()) == 0 else "FAIL",
    f"max pre-2031 SMR capacity {max(pre_smr.values()):.1f} MW")
added_scans(rec, B_B3, CASES_B3, 149)
S_B3 = write_registry(REG_B3, F_B3)

[PASS] b3 D :: the exogenous load is identical to the Step 3 reference load in every run outside the demhi world (131 runs) -- max rel difference 0.00e+00 vs smr100_eia_p50


[PASS] b3 D :: no data value is NaN or infinite in any file -- all 210 keys x 149 files scanned (key inventory)
[PASS] b3 D :: no negative values occur where values must not be negative -- checked ['cap', 'cap_nat', 'load_rt', 'stor_in', 'stor_out', 'curt_ann', 'hours'] (storage charging and upgrades are legitimately negative elsewhere)
[PASS] b3 D :: the nuclear build and generation data is not negative -- cap_new_ann and gen_ann nuclear rows >= 0, all 149 cases


[PASS] b3 D :: the national totals equal the sum of the regional data -- cap, gen_ann, cap_new_ann, ret_ann; all 149 cases within 1e-3


[PASS] b3 D :: no SMR capacity exists before 2031 -- max pre-2031 SMR capacity 0.0 MW


[PASS] b3 X :: no load is dropped after 2025 (the step3_checks E3 rule) -- 149 runs; the historical 2010-2023 artifact 2.94 TWh (case spread 333022.5 MWh)


[INFO] b3 X :: national curtailment in 2050 (context) -- 1.1 to 394.8 TWh (median 33.3) over 149 runs
rate_design2_b3_checks {'PASS': 45, 'INFO': 10}


## 6. Summary

The table below counts the registry statuses per batch. A FAIL anywhere stops notebooks 11 and 12
(their gate R0). The internal registries these replace counted 40 PASS / 13 INFO (ratedesign),
41 / 13 (ratedesign2), 30 / 11 (batch2) and 45 / 10 (batch3), all without a FAIL; the differences
come from the checks this port drops (the archive census, the frozen-file hash guards, the SHA pins
of the ported step3 cells, the t08 port, the five- and eleven-anchor cost-order preflights) and the
checks it adds (phase X). Every check that both versions carry has the same status and the same
numbers.

In [33]:
summary = pd.concat([s.assign(batch=b) for b, s in [("ratedesign", S_RD), ("ratedesign2", S_RD2),
                                                    ("batch2", S_B2REG), ("batch3", S_B3)]])
counts = summary.pivot_table(index="batch", columns="status", values="check", aggfunc="count", fill_value=0)
print(counts.to_string())
fails = summary[summary["status"] == "FAIL"]
assert fails.empty, fails[["batch", "phase", "check", "detail"]].to_string()
print("\nNo FAIL in any batch. Files written:")
for d in sorted(p for p in OUT.iterdir() if p.is_dir()):
    print(f"  {d.name}: {sorted(f.name for f in d.glob('*.csv'))}")

status       INFO  PASS
batch                  
batch2         11    30
batch3         10    45
ratedesign     12    37
ratedesign2    12    39

No FAIL in any batch. Files written:
  rate_design2_b2_checks: ['checks_summary.csv', 'control_outcomes.csv', 'delivery_outcomes.csv', 'fingerprint_errors.csv', 'incentives_echo.csv', 'itc_payments.csv', 'load_invariance.csv', 'pair_identity.csv', 'rate_recovery.csv', 'trajectory_gaps.csv']
  rate_design2_b3_checks: ['case_matrix.csv', 'checks_summary.csv', 'co2cap_binding.csv', 'delivery_outcomes.csv', 'design_echo.csv', 'duals_by_year.csv', 'fingerprint_errors.csv', 'incentives_echo.csv', 'itc_payments.csv', 'load_invariance.csv', 'overbuild_by_case.csv', 'rate_recovery.csv', 'switch_echo.csv', 'trajectory_gaps.csv']
  rate_design2_checks: ['checks_summary.csv', 'delivery_outcomes.csv', 'duals_by_year.csv', 'fingerprint_errors.csv', 'incentives_echo.csv', 'load_invariance.csv', 'overbuild_by_case.csv', 'pair_identity.csv', 'rate_recovery.csv